# Phase 7 - Evaluation

Project: ComplaintIQ

Work through this notebook incrementally. Validate each step before proceeding.

In [5]:
from pathlib import Path
PROJECT_ROOT = Path.home() / 'complaintiq'
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

Project root: /home/sagemaker-user/complaintiq
Exists: True


In [6]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import time
import uuid
import sqlite3
import hashlib
import statistics
import traceback

import numpy as np
import pandas as pd

ROOT = Path("/home/sagemaker-user/complaintiq")
DATA = ROOT / "data"
PROC = DATA / "processed"
INDEX = ROOT / "index"
STATE = ROOT / "state"
SRC = ROOT / "src"
EVAL = ROOT / "eval"

EVAL.mkdir(parents=True, exist_ok=True)

GOLD_DIR = EVAL / "gold"
RESULT_DIR = EVAL / "results"

GOLD_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Evaluation:", EVAL)
print("Golden sets:", GOLD_DIR)
print("Results:", RESULT_DIR)

assert ROOT.exists()
assert (PROC / "cfpb_holdout_june_2026.csv").exists()
assert (INDEX / "reg_vectors.npy").exists()
assert (INDEX / "case_vectors.npy").exists()

print("PASS — Evaluation directories and source artifacts verified.")

Project: /home/sagemaker-user/complaintiq
Evaluation: /home/sagemaker-user/complaintiq/eval
Golden sets: /home/sagemaker-user/complaintiq/eval/gold
Results: /home/sagemaker-user/complaintiq/eval/results
PASS — Evaluation directories and source artifacts verified.


In [7]:
from pathlib import Path
import json
import ast
import re

ROOT = Path("/home/sagemaker-user/complaintiq")
NOTEBOOK_DIR = ROOT / "notebooks"

SOURCE_NOTEBOOKS = {
    "Notebook3": NOTEBOOK_DIR / "03_retrieval.ipynb",
    "Notebook4": NOTEBOOK_DIR / "04_agents.ipynb",
    "Notebook5": NOTEBOOK_DIR / "05_graph.ipynb",
}

NOTEBOOK_CATALOG = {}

for notebook_name, notebook_path in SOURCE_NOTEBOOKS.items():

    assert notebook_path.exists(), (
        f"Notebook not found: {notebook_path}"
    )

    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    catalog = []

    for cell_number, cell in enumerate(notebook["cells"]):

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        if not source.strip():
            continue

        # Skip notebook magic commands for catalog parsing
        clean_source = "\n".join(
            line for line in source.splitlines()
            if not line.lstrip().startswith(("%", "!"))
        )

        try:
            tree = ast.parse(clean_source)
        except SyntaxError:
            continue

        definitions = []

        for node in tree.body:

            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                definitions.append(f"function: {node.name}")

            elif isinstance(node, ast.ClassDef):
                definitions.append(f"class: {node.name}")

            elif isinstance(node, (ast.Import, ast.ImportFrom)):
                definitions.append("import")

            elif isinstance(node, (ast.Assign, ast.AnnAssign)):
                targets = (
                    node.targets if isinstance(node, ast.Assign)
                    else [node.target]
                )

                for target in targets:
                    if isinstance(target, ast.Name):
                        definitions.append(f"variable: {target.id}")

        if definitions:
            catalog.append({
                "cell_number": cell_number,
                "definitions": definitions,
                "source": source
            })

    NOTEBOOK_CATALOG[notebook_name] = catalog

    print(f"\n{'=' * 65}")
    print(notebook_name)
    print(f"{'=' * 65}")

    for item in catalog:
        print(
            f"Cell {item['cell_number']:>3}: "
            + ", ".join(item["definitions"])
        )

print("\nPASS — Saved notebook definitions inventoried.")


Notebook3
Cell   1: import, variable: PROJECT_ROOT
Cell   2: import, import, variable: PROJECT_ROOT, import
Cell   3: import, import, import, import, variable: PROJECT, variable: DATA, variable: PROC, variable: INDEX, variable: REPORTS, variable: EVAL, variable: REGION, variable: BUCKET, variable: S3_INDEX
Cell   5: import, import, variable: PROJECT, variable: PROC, variable: INDEX_DIR, variable: S3_INDEX, variable: required, variable: doc_count, variable: pool_df, variable: holdout_df, variable: pool_count, variable: holdout_count, variable: pool_dates, variable: holdout_dates
Cell   7: import, import
Cell   8: import, variable: AWS_REGION, variable: session, variable: brt
Cell   9: variable: test_body, variable: response, variable: test_vector
Cell  11: variable: CACHE, function: cache_path, function: embed_one, function: embed
Cell  12: function: read_jsonl, variable: reg_chunks, variable: chunk_ids
Cell  14: variable: chunk_lengths
Cell  15: import, import, import, import, import,

In [3]:
import boto3

REGION = "us-east-1"
BUCKET = "sagemaker-complaintiq-456427397380"

MODELS = {
    "worker": "us.anthropic.claude-sonnet-4-6",
    "judge": "us.anthropic.claude-opus-4-8",
    "fast": "us.anthropic.claude-sonnet-4-6",
    "embed": "amazon.titan-embed-text-v2:0"
}

session = boto3.Session(region_name=REGION)

s3 = session.client("s3")
brt = session.client("bedrock-runtime")

# Verify S3 access
s3.head_bucket(Bucket=BUCKET)

# Verify Bedrock client configuration
assert brt.meta.region_name == REGION

print("Region:", REGION)
print("Bucket:", BUCKET)
print("Models:", MODELS)
print("PASS — AWS clients restored.")

Region: us-east-1
Bucket: sagemaker-complaintiq-456427397380
Models: {'worker': 'us.anthropic.claude-sonnet-4-6', 'judge': 'us.anthropic.claude-opus-4-8', 'fast': 'us.anthropic.claude-sonnet-4-6', 'embed': 'amazon.titan-embed-text-v2:0'}
PASS — AWS clients restored.


In [8]:
# ============================================================
# COMPLAINTIQ PHASE 7 — NOTEBOOK 6 BOOTSTRAP
# Restore validated implementations from Notebooks 3, 4 and 5
# No corpus embedding generation, benchmark calls or publishing
# ============================================================

import sys
import os
import json
import ast
import uuid
import re
import hashlib
import sqlite3
import importlib
import inspect

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import boto3
import faiss
import duckdb
import sqlglot

from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from pydantic import BaseModel, Field
from typing import Annotated, Literal, TypedDict
import operator

from botocore.exceptions import ClientError
from langgraph.types import interrupt


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/home/sagemaker-user/complaintiq")

DATA = ROOT / "data"
PROC = DATA / "processed"
INDEX = ROOT / "index"
STATE = ROOT / "state"
SRC = ROOT / "src"
EVAL = ROOT / "eval"

for path in [DATA, PROC, INDEX, STATE, SRC, EVAL]:
    path.mkdir(parents=True, exist_ok=True)

REGION = "us-east-1"
AWS_REGION = REGION
BUCKET = "sagemaker-complaintiq-456427397380"

MODELS = {
    "worker": "us.anthropic.claude-sonnet-4-6",
    "judge": "us.anthropic.claude-opus-4-8",
    "fast": "us.anthropic.claude-sonnet-4-6",
    "embed": "amazon.titan-embed-text-v2:0"
}

session = boto3.Session(region_name=REGION)

s3 = session.client("s3")
brt = session.client("bedrock-runtime")

sys.path.insert(0, str(SRC))

print("PASS — Project and AWS configuration initialized.")


# ------------------------------------------------------------
# 2. RESTORE PHASE 4 LLM MODULE
# ------------------------------------------------------------

import llm
importlib.reload(llm)

assert callable(llm.chat)
assert callable(llm.chat_json)
assert callable(llm.get_usage_report)

print("PASS — Phase 4 LLM wrapper restored.")


# ------------------------------------------------------------
# 3. RESTORE PHASE 3 RETRIEVAL FROM NOTEBOOK 5 CELL 19
# ------------------------------------------------------------

NB5_PATH = ROOT / "notebooks" / "05_graph.ipynb"

assert NB5_PATH.exists(), f"Missing notebook: {NB5_PATH}"

with open(NB5_PATH, "r", encoding="utf-8") as f:
    nb5 = json.load(f)

retrieval_source = "".join(
    nb5["cells"][19]["source"]
)

# Execute the validated original retrieval restoration cell.
# It loads saved vectors and reconstructs the indexes.
# It does not regenerate corpus embeddings.

exec(
    compile(
        retrieval_source,
        "notebook5_cell19_retrieval_restore",
        "exec"
    ),
    globals()
)

assert callable(embed_one)
assert callable(embed)
assert callable(tok)
assert callable(HybridIndex)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("PASS — Phase 3 retrieval layer restored.")


# ------------------------------------------------------------
# 4. RESTORE PHASE 5 AGENTS FROM NOTEBOOK 5 CELL 14
# ------------------------------------------------------------

agent_restore_source = "".join(
    nb5["cells"][14]["source"]
)

# This is the existing selective AST-based restoration
# mechanism from Notebook 5.

exec(
    compile(
        agent_restore_source,
        "notebook5_cell14_agent_restore",
        "exec"
    ),
    globals()
)

print("PASS — Phase 5 agent restoration executed.")


# ------------------------------------------------------------
# 5. RESTORE NOTEBOOK 5 ROUTING HELPERS
# ------------------------------------------------------------

routing_source = "".join(
    nb5["cells"][22]["source"]
)

exec(
    compile(
        routing_source,
        "notebook5_cell22_routing",
        "exec"
    ),
    globals()
)

assert callable(after_analyst)
assert callable(after_critic)
assert callable(after_human)

print("PASS — Graph routing helpers restored.")


# ------------------------------------------------------------
# 6. RESTORE PUBLISHING HELPERS
# ------------------------------------------------------------

# Restore S3 publishing configuration.
# This cell only defines configuration and does not publish.

publishing_config_source = "".join(
    nb5["cells"][25]["source"]
)

exec(
    compile(
        publishing_config_source,
        "notebook5_cell25_publish_config",
        "exec"
    ),
    globals()
)

# Restore usage, audit and publishing functions.
# Definitions are loaded; none are invoked.

publishing_helpers_source = "".join(
    nb5["cells"][26]["source"]
)

exec(
    compile(
        publishing_helpers_source,
        "notebook5_cell26_publish_helpers",
        "exec"
    ),
    globals()
)

publishing_node_source = "".join(
    nb5["cells"][30]["source"]
)

exec(
    compile(
        publishing_node_source,
        "notebook5_cell30_publish_node",
        "exec"
    ),
    globals()
)

assert callable(get_usage_snapshot)
assert callable(append_audit_record)
assert callable(publish_node)

print("PASS — Publishing helper definitions restored.")
print("NOTE — No S3 report or audit write was performed.")


# ------------------------------------------------------------
# 7. VALIDATE REQUIRED AGENT FUNCTIONS
# ------------------------------------------------------------

REQUIRED_FUNCTIONS = [
    "guard_node",
    "router_node",
    "triage_node",
    "analyst_node",
    "cases_node",
    "research_node",
    "drafter_node",
    "critic_node",
    "human_node",
    "refuse_node",
    "after_analyst",
    "after_critic",
    "after_human",
    "publish_node",
]

for name in REQUIRED_FUNCTIONS:
    obj = globals().get(name)

    if not callable(obj):
        raise RuntimeError(
            f"Bootstrap validation failed: {name} is missing."
        )

    print(f"PASS | {name}")


# ------------------------------------------------------------
# 8. VALIDATE SHARED STATE AND RETRIEVAL
# ------------------------------------------------------------

assert "CIQState" in globals()
assert "TriageResult" in globals()

assert len(TAXONOMY) > 0

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

assert len(reg_chunks) == 1784
assert len(case_chunks) == 3000

assert reg_vectors.shape == (1784, 1024)
assert case_vectors.shape == (3000, 1024)


# ------------------------------------------------------------
# 9. FINAL BOOTSTRAP REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("COMPLAINTIQ PHASE 7 BOOTSTRAP VALIDATION")
print("=" * 65)

print("LLM wrapper:          PASS")
print("Embedding functions:  PASS")
print("HybridIndex:           PASS")
print("Regulatory index:     ", REG_INDEX.faiss.ntotal)
print("Complaint index:      ", CASE_INDEX.faiss.ntotal)
print("CIQState:              PASS")
print("Triage taxonomy:      ", len(TAXONOMY))
print("Agent functions:      ", len(REQUIRED_FUNCTIONS))
print("Routing helpers:       PASS")
print("Publishing helpers:    PASS")

print("\nPASS — PHASE 7 RUNTIME RESTORED.")

PASS — Project and AWS configuration initialized.
PASS — Phase 4 LLM wrapper restored.
PASS — All four existing retrieval artifacts found.
Regulatory metadata: 1784
Regulatory vectors: (1784, 1024)
Complaint metadata: 3000
Complaint vectors: (3000, 1024)

PASS — Regulatory index: 1784
PASS — Complaint index: 3000

PASS — Phase 3 retrieval layer restored.
PASS — Phase 3 retrieval layer restored.
Executed implementation cells: [44, 46, 48, 50, 52, 60, 61, 62, 63, 67]
PASS | CIQState
PASS | guard_node
PASS | router_node
PASS | triage_node
PASS | analyst_node
PASS | cases_node
PASS | research_node
PASS | drafter_node
PASS | critic_node
PASS | human_node
PASS | refuse_node
PASS — Phase 5 agent restoration executed.
PASS — All graph-routing functions validated.
PASS — Graph routing helpers restored.
BUCKET: sagemaker-complaintiq-456427397380
REPORT PREFIX: complaintiq/reports
AUDIT KEY: complaintiq/reports/audit.jsonl
PASS — Publishing configuration restored.
PASS — Usage snapshot and audit 

In [9]:
print("CIQState annotations:")
print(CIQState.__annotations__)

CIQState annotations:
{'request': <class 'str'>, 'clean_request': <class 'str'>, 'flags': list[str], 'intent': typing.Literal['TRIAGE', 'ANALYTICS', 'REG_QA', 'BRIEF', 'OUT_OF_SCOPE'], 'triage': <class 'dict'>, 'similar_cases': list[dict], 'sql': <class 'str'>, 'sql_rows': list[dict], 'evidence': list[dict], 'draft': <class 'str'>, 'critique': <class 'dict'>, 'revisions': <class 'int'>, 'human_decision': <class 'dict'>, 'report_uri': <class 'str'>, 'trace': typing.Annotated[list[dict], <built-in function add>]}


In [10]:
def tok(s):
    return re.findall(
        r"[a-z0-9]+(?:\.[0-9]+)?",
        str(s).lower()
    )

In [11]:
print("=== FINAL DEPENDENCY CHECK ===")

checks = [
    ("embed_one", callable(embed_one)),
    ("embed", callable(embed)),
    ("tok", callable(tok)),
    ("HybridIndex", callable(HybridIndex)),
    ("REG_INDEX", REG_INDEX.faiss.ntotal == 1784),
    ("CASE_INDEX", CASE_INDEX.faiss.ntotal == 3000),
    ("CIQState", "CIQState" in globals()),
    ("TriageResult", "TriageResult" in globals()),
    ("llm.chat", callable(llm.chat)),
    ("triage_node", callable(triage_node)),
    ("analyst_node", callable(analyst_node)),
    ("research_node", callable(research_node)),
    ("critic_node", callable(critic_node)),
    ("publish_node", callable(publish_node)),
]

for name, passed in checks:
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

assert all(passed for _, passed in checks)

print("\nALL DEPENDENCY CHECKS PASSED.")

=== FINAL DEPENDENCY CHECK ===
PASS | embed_one
PASS | embed
PASS | tok
PASS | HybridIndex
PASS | REG_INDEX
PASS | CASE_INDEX
PASS | CIQState
PASS | TriageResult
PASS | llm.chat
PASS | triage_node
PASS | analyst_node
PASS | research_node
PASS | critic_node
PASS | publish_node

ALL DEPENDENCY CHECKS PASSED.


In [12]:

import duckdb

# --------------------------------------------------
# Phase 7 — Evaluation database validation
# --------------------------------------------------

DB_PATH = ROOT / "data" / "complaintiq.duckdb"

assert DB_PATH.exists(), f"Database not found: {DB_PATH}"

print("DuckDB:", DB_PATH)

# Close any existing read-only connection from this notebook
if "ro" in globals():
    try:
        ro.close()
    except Exception:
        pass

# Open database in read-only mode
ro = duckdb.connect(str(DB_PATH), read_only=True)

# Verify complaint view
view_check = ro.execute("""
    SELECT COUNT(*)
    FROM v_complaints
""").fetchone()[0]

print("v_complaints rows:", view_check)

assert view_check > 0, "v_complaints is empty"

print("PASS — Read-only evaluation database restored.")


DuckDB: /home/sagemaker-user/complaintiq/data/complaintiq.duckdb
v_complaints rows: 37923
PASS — Read-only evaluation database restored.


In [13]:
sys.path.insert(0, str(SRC))

import importlib
import llm

importlib.reload(llm)

assert callable(llm.chat)
assert callable(llm.get_usage_report)

print(json.dumps(
    llm.get_usage_report(),
    indent=2,
    default=str
))

print("PASS — Phase 4 LLM wrapper restored.")

{
  "ledger": {
    "calls": 0,
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
    "latency": {
      "calls": 0,
      "average_ms": 0,
      "p50_ms": 0,
      "p95_ms": 0
    },
    "models_used": []
  },
  "cache": {
    "cache_path": "/home/sagemaker-user/complaintiq/state/llm_cache.sqlite",
    "cached_responses": 9
  },
  "models_without_temperature": []
}
PASS — Phase 4 LLM wrapper restored.


In [14]:
class HybridIndex:

    def __init__(self, chunks, vectors):
        self.chunks = list(chunks)
        self.vectors = np.asarray(vectors, dtype="float32")

        if len(self.vectors) != len(self.chunks):
            raise ValueError("Vector/metadata count mismatch")

        if not np.isfinite(self.vectors).all():
            raise ValueError("Invalid vectors")

        self.faiss = faiss.IndexFlatIP(
            self.vectors.shape[1]
        )

        self.faiss.add(self.vectors)

        self.bm25 = BM25Okapi([
            tok(c.get("text", ""))
            for c in self.chunks
        ])

    def search(self, query, k=8, pool=40, rrf_k=60):

        if not query or not str(query).strip():
            raise ValueError("Query cannot be blank")

        pool = min(pool, len(self.chunks))

        qv = embed([query], workers=1)

        _, dense_ids = self.faiss.search(qv, pool)

        sparse_scores = self.bm25.get_scores(tok(query))

        sparse_ids = np.argsort(
            sparse_scores
        )[::-1][:pool]

        scores = defaultdict(float)

        for ranking in (dense_ids[0], sparse_ids):
            for rank, idx in enumerate(ranking):
                if int(idx) >= 0:
                    scores[int(idx)] += (
                        1.0 / (rrf_k + rank + 1)
                    )

        citation = re.search(
            r"\b(\d{1,2})\s*CFR\s*(\d{3,4}\.\d+)\b",
            query,
            re.I
        )

        if citation:
            target = (
                f"{citation.group(1)} CFR "
                f"{citation.group(2)}"
            ).lower()

            compact = (
                f"{citation.group(1)}cfr"
                f"{citation.group(2)}"
            ).lower()

            for idx, rec in enumerate(self.chunks):

                haystack = " ".join([
                    str(rec.get("chunk_id", "")),
                    str(rec.get("section", "")),
                    str(rec.get("text", ""))
                ]).lower()

                normalized = re.sub(
                    r"\s+", " ", haystack
                )

                compact_haystack = re.sub(
                    r"[^a-z0-9.]", "", haystack
                )

                if (
                    target in normalized
                    or compact in compact_haystack
                ):
                    scores[idx] += 1.0

        top = sorted(
            scores,
            key=scores.get,
            reverse=True
        )[:k]

        return [
            {
                **self.chunks[i],
                "rrf": round(float(scores[i]), 6)
            }
            for i in top
        ]



In [15]:
assert "HybridIndex" in globals()
assert callable(HybridIndex)

print("PASS — HybridIndex class restored.")

PASS — HybridIndex class restored.


In [16]:
import re

def tok(text):
    """
    Tokenize text for BM25 retrieval.
    Uses the same lowercase alphanumeric tokenization
    expected by the retrieval index.
    """
    return re.findall(r"\b\w+\b", str(text).lower())

print("PASS — tok() function defined.")
print("Test:", tok("12 CFR 1005.11 Electronic Fund Transfers"))

PASS — tok() function defined.
Test: ['12', 'cfr', '1005', '11', 'electronic', 'fund', 'transfers']


In [17]:
assert callable(tok)
assert callable(HybridIndex)

print("PASS — HybridIndex dependencies available.")

PASS — HybridIndex dependencies available.


In [18]:
import re
import faiss
from collections import defaultdict
from rank_bm25 import BM25Okapi

reg_chunks = [
    json.loads(line)
    for line in (INDEX / "reg_chunks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

case_chunks = [
    json.loads(line)
    for line in (INDEX / "case_chunks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

reg_vectors = np.load(INDEX / "reg_vectors.npy")
case_vectors = np.load(INDEX / "case_vectors.npy")

assert reg_vectors.shape == (1784, 1024)
assert case_vectors.shape == (3000, 1024)

# Re-execute the validated HybridIndex class and embed() implementation
# from Notebook 5's saved source cells before running this cell.

REG_INDEX = HybridIndex(reg_chunks, reg_vectors)
CASE_INDEX = HybridIndex(case_chunks, case_vectors)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("PASS — Phase 3 retrieval indexes reconstructed.")

PASS — Phase 3 retrieval indexes reconstructed.


In [19]:
def embed(texts, workers=3, dim=1024, desc="Embedding"):
    texts = list(texts)

    if not texts:
        return np.empty((0, dim), dtype="float32")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        vectors = list(
            tqdm(
                executor.map(lambda t: embed_one(t, dim), texts),
                total=len(texts),
                desc=desc,
                dynamic_ncols=True
            )
        )

    return np.vstack(vectors).astype("float32")


In [20]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import numpy as np

print("PASS — Embedding dependencies imported.")

PASS — Embedding dependencies imported.


In [21]:
print("=== REGULATORY RETRIEVAL TEST ===")

results = REG_INDEX.search(
    "12 CFR 1005.11 error resolution requirements",
    k=5
)

for i, item in enumerate(results, 1):
    print(
        i,
        item.get("chunk_id"),
        str(item.get("text", ""))[:180]
    )

assert len(results) > 0

print("\nPASS — Regulatory retrieval operational.")

=== REGULATORY RETRIEVAL TEST ===


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

1 exam_manual_v2_2012_page_0700_chunk_0001 CFPB Consumer Laws and Regulations EFTA An error includes any of the following: • An unauthorized EFT. • An incorrect EFT to or from the consumer’s account. • The omission from a p
2 exam_manual_v2_2012_page_0735_chunk_0001 Yes No NA CFPB Examination Checklist EFTA 39. If a bill ing error occurred, does the financial institution not impose a charge related to any aspect of the error-resolution process
3 exam_manual_v2_2012_page_0699_chunk_0002 an EFT, for example, when the card is used to draw cash advances directly from a credit line, only the error resolution provisions of Regulation Z will apply. • If the unauthorized
4 12CFR1026_1026.13_chunk_0004 ) Promptly reports that the amount or account is in dispute; (ii) Mails or delivers to the consumer (at the same time the report is made) a written notice of the name and address o
5 exam_manual_v2_2012_page_0734_chunk_0001 CFPB Examination Checklist EFTA CFPB Manual V.2 (October 2012) Checklist 

In [22]:
# ============================================================
# PHASE 7 — G1: JUNE HOLDOUT DATASET INSPECTION
# ============================================================

import pandas as pd
import json
from pathlib import Path

# Paths
GOLD_DIR = EVAL / "gold"
RESULTS_DIR = EVAL / "results"
ANALYSIS_DIR = EVAL / "analysis"
MANIFEST_DIR = EVAL / "manifests"

for directory in [
    GOLD_DIR,
    RESULTS_DIR,
    ANALYSIS_DIR,
    MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

HOLDOUT_FILE = PROC / "cfpb_holdout_june_2026.csv"

assert HOLDOUT_FILE.exists(), (
    f"June holdout file not found: {HOLDOUT_FILE}"
)

# Load holdout dataset
holdout_df = pd.read_csv(HOLDOUT_FILE)

print("=" * 65)
print("G1 — JUNE HOLDOUT DATASET INSPECTION")
print("=" * 65)

print("\nDataset shape:")
print(holdout_df.shape)

print("\nAvailable columns:")
for column in holdout_df.columns:
    print(f"- {column}")

print("\nData types:")
print(holdout_df.dtypes)

print("\nFirst five records:")
display(holdout_df.head())

print("\nMissing values:")
display(
    holdout_df.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\nDuplicate records:")
print(holdout_df.duplicated().sum())

print("\nPASS — June holdout dataset loaded.")

G1 — JUNE HOLDOUT DATASET INSPECTION

Dataset shape:
(6110, 16)

Available columns:
- Date received
- Product
- Sub-product
- Issue
- Sub-issue
- Consumer complaint narrative
- Company public response
- Company
- State
- ZIP code
- Tags
- Submitted via
- Date sent to company
- Company response to consumer
- Timely response?
- Complaint ID

Data types:
Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Complaint ID                     int64
dtype: object

First f

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2026-06-17,Credit card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Didn't receive advertised or promotional terms,I am filing this complaint because Barclays ha...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",MN,554XX,NaN,Web,2026-07-30,Closed with monetary relief,Yes,23300227
1,2026-06-23,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,Despite five separate contacts with Citi betwe...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",MO,65109,NaN,Web,2026-06-23,Closed with monetary relief,Yes,23472102
2,2026-06-25,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,"My name is XXXX XXXX, I am a resident of the c...",Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",FL,32824,NaN,Web,2026-06-25,Closed with explanation,Yes,23576641
3,2026-06-02,Credit card,General-purpose credit card or charge card,Getting a credit card,Problem getting a working replacement card,These r proofs they r keeping my money they wo...,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,AZ,85051,NaN,Web,2026-06-02,Closed with explanation,Yes,22815215
4,2026-06-23,Checking or savings account,Checking account,Managing an account,Problem accessing account,Since my last complaints the company has start...,Company has responded to the consumer and the ...,BMO BANK NATIONAL ASSOCIATION,DE,198XX,NaN,Web,2026-06-23,Closed with monetary relief,Yes,23466116



Missing values:


,missing_count
Tags,4886
Company public response,3420
State,49
ZIP code,42
Sub-issue,7
Issue,0
Date received,0
Product,0
Company,0
Consumer complaint narrative,0



Duplicate records:
0

PASS — June holdout dataset loaded.


In [23]:
required = [
    "ROOT", "BUCKET", "MODELS", "s3", "brt",
    "con", "ro", "llm",
    "REG_INDEX", "CASE_INDEX",
    "CIQState",
    "guard_node", "router_node",
    "triage_node", "analyst_node",
    "cases_node", "research_node",
    "drafter_node", "critic_node",
    "human_node", "refuse_node"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"Missing evaluation dependencies: {missing}")

assert len(reg_chunks) == 1784
assert len(case_chunks) == 3000

print("PASS — Notebook 6 runtime is ready.")

PASS — Notebook 6 runtime is ready.


In [24]:
# ============================================================
# PHASE 7 — G1 CELL 9
# REPRODUCIBLE, PRODUCT-STRATIFIED SAMPLING
# ============================================================

import numpy as np
import pandas as pd

G1_SAMPLE_SIZE = 60
G1_RANDOM_SEED = 20261106

required_columns = [
    "Complaint ID",
    "Product",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative"
]

missing_columns = [
    c for c in required_columns
    if c not in holdout_df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# Only use records with a usable narrative.
g1_source = holdout_df.copy()

g1_source["Consumer complaint narrative"] = (
    g1_source["Consumer complaint narrative"]
    .fillna("")
    .astype(str)
    .str.strip()
)

g1_source = g1_source[
    g1_source["Consumer complaint narrative"].str.len() > 0
].copy()

# Complaint IDs must be unique.
assert g1_source["Complaint ID"].is_unique, (
    "Duplicate Complaint IDs detected in source data."
)

assert len(g1_source) >= G1_SAMPLE_SIZE

# Product-level proportional allocation using largest remainder.
product_counts = (
    g1_source.groupby("Product")
    .size()
    .sort_index()
)

raw_allocations = (
    product_counts / product_counts.sum()
) * G1_SAMPLE_SIZE

allocations = np.floor(raw_allocations).astype(int)

remaining = G1_SAMPLE_SIZE - allocations.sum()

remainders = (
    raw_allocations - allocations
).sort_values(ascending=False)

for product in remainders.index:
    if remaining == 0:
        break

    if allocations[product] < product_counts[product]:
        allocations[product] += 1
        remaining -= 1

assert allocations.sum() == G1_SAMPLE_SIZE

# Sample independently within each product.
sampled_groups = []

for product, sample_count in allocations.items():
    if sample_count == 0:
        continue

    group = g1_source[
        g1_source["Product"] == product
    ]

    sampled_groups.append(
        group.sample(
            n=int(sample_count),
            random_state=G1_RANDOM_SEED
        )
    )

g1_sample = (
    pd.concat(sampled_groups)
    .sample(
        frac=1,
        random_state=G1_RANDOM_SEED
    )
    .reset_index(drop=True)
)

assert len(g1_sample) == G1_SAMPLE_SIZE
assert g1_sample["Complaint ID"].is_unique

print("=" * 65)
print("G1 — STRATIFIED SAMPLE")
print("=" * 65)

print("Sample size:", len(g1_sample))
print("Random seed:", G1_RANDOM_SEED)

print("\nProduct allocation:")
display(
    pd.DataFrame({
        "source_count": product_counts,
        "sample_count": allocations
    }).fillna(0).astype(int)
)

print("\nSample distribution:")
display(
    g1_sample["Product"]
    .value_counts()
    .to_frame("sample_count")
)

print("\nIssue distribution:")
display(
    g1_sample["Issue"]
    .value_counts()
    .to_frame("sample_count")
)

print("\nPASS — 60 reproducible G1 records selected.")

G1 — STRATIFIED SAMPLE
Sample size: 60
Random seed: 20261106

Product allocation:


,source_count,sample_count
Product,,
Checking or savings account,3270,32
Credit card,2840,28



Sample distribution:


,sample_count
Product,
Checking or savings account,32
Credit card,28



Issue distribution:


,sample_count
Issue,
Managing an account,18
Fees or interest,6
Problem with a purchase shown on your statement,6
Problem with a lender or other company charging your account,5
Closing an account,5
"Other features, terms, or problems",4
Closing your account,3
Improper use of your report,3
Opening an account,2



PASS — 60 reproducible G1 records selected.


In [25]:
# ============================================================
# PHASE 7 — G1 CELL 10
# BUILD GOLD EVALUATION RECORDS
# ============================================================

import json
import pandas as pd

def clean_gold_value(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    if value == "":
        return None

    return value


g1_gold_records = []

for _, row in g1_sample.iterrows():

    record = {
        "case_id": str(int(row["Complaint ID"])),

        "request": clean_gold_value(
            row["Consumer complaint narrative"]
        ),

        "expected": {
            "product": clean_gold_value(row["Product"]),
            "issue": clean_gold_value(row["Issue"]),
            "sub_issue": clean_gold_value(row["Sub-issue"])
        },

        # These fields are not supplied as ground truth
        # by the CFPB dataset. They require human annotation.
        "manual_annotation": {
            "severity": None,
            "vulnerable_customer": None,
            "vulnerability_signals": None,
            "regulation_hints": None,
            "summary": None
        },

        "source": {
            "dataset": "cfpb_holdout_june_2026",
            "complaint_id": str(int(row["Complaint ID"])),
            "date_received": clean_gold_value(
                row["Date received"]
            )
        },

        "annotation_status": "pending"
    }

    g1_gold_records.append(record)

assert len(g1_gold_records) == 60

print("=" * 65)
print("G1 — GOLD RECORD GENERATION")
print("=" * 65)

print("Records generated:", len(g1_gold_records))

print("\nExample gold record:")
print(
    json.dumps(
        g1_gold_records[0],
        indent=2,
        ensure_ascii=False
    )
)

print("\nPASS — G1 gold records generated.")

G1 — GOLD RECORD GENERATION
Records generated: 60

Example gold record:
{
  "case_id": "23266147",
  "request": "I am filing this complaint against BMO Bank regarding an unreasonable hold placed on my deposited insurance settlement funds and the overdraft fees that resulted from the banks actions. On XX/XX/year>, I deposited a settlement check from XXXX XXXX at the BMO branch located at XXXX XXXX XXXX, XXXX XXXX, Missouri. BMO placed a hold on the funds but failed to provide adequate notice explaining the reason for the hold, the length of the hold, or my rights under the banks funds availability policy. After the deposit, documentation was provided to verify the legitimacy of the settlement funds. The branch also received verification directly from XXXX XXXX confirming that the check was valid and legitimate. Despite receiving this verification, BMO continued to restrict access to my funds. Because BMO withheld access to verified funds, transactions that would have otherwise been cove

In [26]:
# ============================================================
# PHASE 7 — G1 CELL 11
# HUMAN ANNOTATION TEMPLATE
# ============================================================

import pandas as pd

annotation_rows = []

for record in g1_gold_records:

    annotation_rows.append({
        "case_id": record["case_id"],

        "severity": "",
        "vulnerable_customer": "",
        "vulnerability_signals": "",
        "regulation_hints": "",
        "summary": "",

        "reviewer": "",
        "review_notes": "",
        "annotation_status": "PENDING"
    })

g1_annotation_df = pd.DataFrame(annotation_rows)

G1_ANNOTATION_FILE = (
    GOLD_DIR / "G1_triage_manual_annotations.csv"
)

print("=" * 65)
print("G1 — HUMAN ANNOTATION TEMPLATE")
print("=" * 65)

print("Annotation records:", len(g1_annotation_df))

display(g1_annotation_df.head(10))

print("\nSeverity values to use:")
print(["LOW", "MEDIUM", "HIGH", "CRITICAL"])

print("\nVulnerable customer values to use:")
print(["True", "False"])

print("\nAnnotation file target:")
print(G1_ANNOTATION_FILE)

print("\nPASS — Annotation template prepared.")

G1 — HUMAN ANNOTATION TEMPLATE
Annotation records: 60


,case_id,severity,vulnerable_customer,vulnerability_signals,regulation_hints,summary,reviewer,review_notes,annotation_status
0,23266147,,,,,,,,PENDING
1,23172941,,,,,,,,PENDING
2,23492548,,,,,,,,PENDING
3,23304250,,,,,,,,PENDING
4,22965626,,,,,,,,PENDING
5,23082821,,,,,,,,PENDING
6,23494747,,,,,,,,PENDING
7,23053378,,,,,,,,PENDING
8,22988407,,,,,,,,PENDING
9,22794522,,,,,,,,PENDING



Severity values to use:
['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']

Vulnerable customer values to use:
['True', 'False']

Annotation file target:
/home/sagemaker-user/complaintiq/eval/gold/G1_triage_manual_annotations.csv

PASS — Annotation template prepared.


In [27]:
# ============================================================
# PHASE 7 — G1 CELL 12
# SAVE GOLD DATASET
# ============================================================

import json
import hashlib
from pathlib import Path

G1_GOLD_FILE = GOLD_DIR / "G1_triage_gold.jsonl"

# Write JSONL.
with open(G1_GOLD_FILE, "w", encoding="utf-8") as f:

    for record in g1_gold_records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

# Save human annotation template.
g1_annotation_df.to_csv(
    G1_ANNOTATION_FILE,
    index=False
)

# Calculate SHA256 checksums.
def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


print("=" * 65)
print("G1 — SAVED ARTIFACTS")
print("=" * 65)

print("Gold dataset:")
print(G1_GOLD_FILE)
print("Size:", G1_GOLD_FILE.stat().st_size, "bytes")
print("SHA256:", sha256_file(G1_GOLD_FILE))

print("\nAnnotation template:")
print(G1_ANNOTATION_FILE)
print("Size:", G1_ANNOTATION_FILE.stat().st_size, "bytes")
print("SHA256:", sha256_file(G1_ANNOTATION_FILE))

print("\nPASS — G1 artifacts saved locally.")

G1 — SAVED ARTIFACTS
Gold dataset:
/home/sagemaker-user/complaintiq/eval/gold/G1_triage_gold.jsonl
Size: 116918 bytes
SHA256: f6f405dd741175eb8f5e0aa0eb901eedb34a0c633311d601f9d6299d69420984

Annotation template:
/home/sagemaker-user/complaintiq/eval/gold/G1_triage_manual_annotations.csv
Size: 1564 bytes
SHA256: 11452cfbde9e47de87328aab8a6913cff137174d3822796dc23f561604780987

PASS — G1 artifacts saved locally.


In [28]:
# ============================================================
# PHASE 7 — G1 CELL 13
# GOLD DATASET ACCEPTANCE VALIDATION
# ============================================================

import json
import pandas as pd

# Reload saved artifacts instead of validating only memory objects.
with open(G1_GOLD_FILE, encoding="utf-8") as f:
    g1_reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

g1_annotations = pd.read_csv(
    G1_ANNOTATION_FILE,
    dtype=str
)

# 1. Record count.
assert len(g1_reloaded) == 60

# 2. Unique complaint IDs.
case_ids = [
    record["case_id"]
    for record in g1_reloaded
]

assert len(case_ids) == len(set(case_ids))

# 3. Required input and expected labels.
for record in g1_reloaded:

    assert record["request"].strip()

    assert record["expected"]["product"]
    assert record["expected"]["issue"]

    assert record["source"]["dataset"] == (
        "cfpb_holdout_june_2026"
    )

# 4. Annotation template integrity.
assert len(g1_annotations) == 60
assert g1_annotations["case_id"].is_unique

assert set(g1_annotations["case_id"]) == set(case_ids)

# 5. Verify all IDs belong to the June holdout.
holdout_ids = set(
    holdout_df["Complaint ID"].astype(str)
)

assert set(case_ids).issubset(holdout_ids)

# 6. Check whether complaint identifiers are exposed
# in the existing complaint retrieval metadata.
index_ids = set()

for chunk in CASE_INDEX.chunks:

    possible_ids = [
        chunk.get("complaint_id"),
        chunk.get("Complaint ID"),
        chunk.get("case_id")
    ]

    for value in possible_ids:
        if value is not None:
            index_ids.add(str(value))

overlap = set(case_ids).intersection(index_ids)

print("=" * 65)
print("COMPLAINTIQ — G1 ACCEPTANCE VALIDATION")
print("=" * 65)

print("Gold records:", len(g1_reloaded))
print("Unique IDs:", len(set(case_ids)))
print("Valid narratives:", sum(
    bool(r["request"].strip())
    for r in g1_reloaded
))
print("Valid product labels:", sum(
    bool(r["expected"]["product"])
    for r in g1_reloaded
))
print("Valid issue labels:", sum(
    bool(r["expected"]["issue"])
    for r in g1_reloaded
))
print("Annotation rows:", len(g1_annotations))

print("\nExisting retrieval metadata ID overlap:", len(overlap))

if overlap:
    print("WARNING — Matching IDs found in retrieval metadata.")
    print("Sample matching IDs:", sorted(overlap)[:10])
else:
    print(
        "No matching complaint IDs found in the "
        "explicit metadata ID fields."
    )

print("\nArtifact checksums:")
print("Gold:", sha256_file(G1_GOLD_FILE))
print("Annotations:", sha256_file(G1_ANNOTATION_FILE))

print("\nPASS — G1 GOLD DATASET STRUCTURAL ACCEPTANCE.")

COMPLAINTIQ — G1 ACCEPTANCE VALIDATION
Gold records: 60
Unique IDs: 60
Valid narratives: 60
Valid product labels: 60
Valid issue labels: 60
Annotation rows: 60

Existing retrieval metadata ID overlap: 0
No matching complaint IDs found in the explicit metadata ID fields.

Artifact checksums:
Gold: f6f405dd741175eb8f5e0aa0eb901eedb34a0c633311d601f9d6299d69420984
Annotations: 11452cfbde9e47de87328aab8a6913cff137174d3822796dc23f561604780987

PASS — G1 GOLD DATASET STRUCTURAL ACCEPTANCE.


In [29]:
# ============================================================
# PHASE 7 — RETRIEVAL TOKENIZER CONSISTENCY FIX
# ============================================================

import re
from collections import defaultdict
from rank_bm25 import BM25Okapi

# Restore the original Phase 3 tokenizer.
def tok(s):
    return re.findall(
        r"[a-z0-9]+(?:\.[0-9]+)?",
        str(s).lower()
    )

# Rebuild in-memory indexes using existing artifacts.
REG_INDEX = HybridIndex(reg_chunks, reg_vectors)
CASE_INDEX = HybridIndex(case_chunks, case_vectors)

# Validate the tokenizer.
test_tokens = tok("12 CFR 1005.11 Electronic Fund Transfers")

assert "1005.11" in test_tokens, (
    f"Unexpected tokenization: {test_tokens}"
)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("Tokenizer test:", test_tokens)
print("Regulatory index:", REG_INDEX.faiss.ntotal)
print("Complaint index:", CASE_INDEX.faiss.ntotal)

# Test regulatory retrieval.
results = REG_INDEX.search(
    "12 CFR 1005.11 error resolution requirements",
    k=5
)

assert len(results) == 5

print("\nTop regulatory retrieval results:")
for rank, result in enumerate(results, 1):
    print(
        rank,
        result.get("chunk_id"),
        result.get("rrf")
    )

print("\nPASS — Retrieval tokenizer consistency restored.")

Tokenizer test: ['12', 'cfr', '1005.11', 'electronic', 'fund', 'transfers']
Regulatory index: 1784
Complaint index: 3000


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]


Top regulatory retrieval results:
1 exam_manual_v2_2012_page_0700_chunk_0001 1.032522
2 exam_manual_v2_2012_page_0734_chunk_0001 1.028986
3 exam_manual_v2_2012_page_0699_chunk_0002 1.02885
4 12CFR1026_1026.13_chunk_0004 1.028405
5 exam_manual_v2_2012_page_0735_chunk_0001 1.028283

PASS — Retrieval tokenizer consistency restored.


In [30]:
# ============================================================
# PHASE 7 — G2 CELL 1
# ANALYTICS SCHEMA VALIDATION AND SQL HELPERS
# ============================================================

import json
import math
import pandas as pd
import numpy as np
from datetime import date, datetime
from decimal import Decimal

G2_GOLD_FILE = GOLD_DIR / "G2_analytics_gold.jsonl"
G2_EXPECTED_FILE = GOLD_DIR / "G2_analytics_expected.jsonl"
G2_SUMMARY_FILE = RESULT_DIR / "G2_reference_results_summary.csv"

assert "ro" in globals(), "Read-only DuckDB connection missing."
assert "guard_sql" in globals(), "SQL guard function missing."
assert "analyst_node" in globals(), "Analyst node missing."

# Inspect actual v_complaints schema.
g2_schema_df = ro.execute(
    "DESCRIBE v_complaints"
).fetchdf()

g2_columns = set(g2_schema_df["column_name"].str.lower())

print("=" * 70)
print("G2 — ANALYTICS SCHEMA")
print("=" * 70)

display(g2_schema_df)

# Resolve column aliases safely.
G2_COLUMN_CANDIDATES = {
    "complaint_id": ["complaint_id"],
    "date_received": ["date_received"],
    "product": ["product"],
    "sub_product": ["sub_product", "subproduct"],
    "issue": ["issue"],
    "sub_issue": ["sub_issue", "subissue"],
    "company": ["company"],
    "state": ["state"],
    "tags": ["tags"],
    "submitted_via": ["submitted_via"],
    "company_response": [
        "company_response",
        "company_response_to_consumer"
    ],
    "timely": [
        "timely",
        "timely_response"
    ]
}

G2_COL = {}

for logical_name, candidates in G2_COLUMN_CANDIDATES.items():

    found = next(
        (c for c in candidates if c.lower() in g2_columns),
        None
    )

    if found:
        G2_COL[logical_name] = found


required_g2_columns = [
    "complaint_id",
    "date_received",
    "product",
    "issue",
    "company",
    "state",
    "tags",
    "timely",
    "company_response"
]

missing_g2 = [
    c for c in required_g2_columns
    if c not in G2_COL
]

assert not missing_g2, (
    f"Required G2 columns unavailable: {missing_g2}"
)

# SQL identifier quoting.
def g2_qcol(logical_name):
    actual = G2_COL[logical_name]
    return '"' + actual.replace('"', '""') + '"'


# Common SQL fields.
G2_DATE = g2_qcol("date_received")
G2_PRODUCT = g2_qcol("product")
G2_ISSUE = g2_qcol("issue")
G2_COMPANY = g2_qcol("company")
G2_STATE = g2_qcol("state")
G2_TAGS = g2_qcol("tags")
G2_TIMELY = g2_qcol("timely")
G2_RESPONSE = g2_qcol("company_response")
G2_ID = g2_qcol("complaint_id")

G2_START = "DATE '2026-01-01'"
G2_END = "DATE '2026-07-01'"

G2_WINDOW = f"""
{G2_DATE} >= {G2_START}
AND {G2_DATE} < {G2_END}
"""

# Only two in-scope products.
G2_PRODUCTS = (
    "'Credit card', "
    "'Checking or savings account'"
)

# JSON-safe conversion.
def g2_json_safe(value):

    if value is None:
        return None

    if isinstance(value, dict):
        return {
            str(k): g2_json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            g2_json_safe(v)
            for v in value
        ]

    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.isoformat()

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        return None if np.isnan(value) else float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, Decimal):
        return float(value)

    if isinstance(value, float) and not math.isfinite(value):
        return None

    return value


print("\nResolved columns:")
for logical, actual in G2_COL.items():
    print(f"{logical:20} -> {actual}")

print("\nEvaluation period: January 1 – June 30, 2026")
print("Approved source: v_complaints")
print("Connection: read-only")

print("\nPASS — G2 schema and SQL helpers initialized.")

G2 — ANALYTICS SCHEMA


,column_name,column_type,null,key,default,extra
0,date_received,DATE,YES,None,None,None
1,product,VARCHAR,YES,None,None,None
2,subproduct,VARCHAR,YES,None,None,None
3,issue,VARCHAR,YES,None,None,None
4,subissue,VARCHAR,YES,None,None,None
5,narrative,VARCHAR,YES,None,None,None
6,company_public_response,VARCHAR,YES,None,None,None
7,company,VARCHAR,YES,None,None,None
8,state,VARCHAR,YES,None,None,None
9,zip_code,VARCHAR,YES,None,None,None



Resolved columns:
complaint_id         -> complaint_id
date_received        -> date_received
product              -> product
sub_product          -> subproduct
issue                -> issue
sub_issue            -> subissue
company              -> company
state                -> state
tags                 -> tags
submitted_via        -> submitted_via
company_response     -> company_response_to_consumer
timely               -> timely_response

Evaluation period: January 1 – June 30, 2026
Approved source: v_complaints
Connection: read-only

PASS — G2 schema and SQL helpers initialized.


In [31]:
# ============================================================
# PHASE 7 — G2 CELL 2
# DEFINE 20 ANALYTICS GOLD QUESTIONS
# ============================================================

g2_questions = []

def add_g2_question(
    question_id,
    question,
    category,
    reference_sql=None,
    answerable=True,
    refusal_reason=None
):
    g2_questions.append({
        "question_id": question_id,
        "question": question,
        "category": category,
        "answerable": answerable,
        "reference_sql": reference_sql,
        "refusal_reason": refusal_reason
    })


# ------------------------------------------------------------
# G2-01 — Top five credit card issues in June
# Date filter + GROUP BY + ORDER BY
# ------------------------------------------------------------

add_g2_question(
    "G2-01",
    "What were the five most common issues for credit card complaints received in June 2026, with counts?",
    "date_filter_group_order",
    f"""
    SELECT
        {G2_ISSUE} AS issue,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Credit card'
      AND {G2_DATE} >= DATE '2026-06-01'
      AND {G2_DATE} < DATE '2026-07-01'
    GROUP BY 1
    ORDER BY complaint_count DESC, issue ASC
    LIMIT 5
    """
)


# ------------------------------------------------------------
# G2-02 — Monthly Older American complaints
# ------------------------------------------------------------

add_g2_question(
    "G2-02",
    "Show monthly complaint counts for Checking or savings account complaints tagged Older American from January through June 2026.",
    "date_filter_group_order",
    f"""
    SELECT
        DATE_TRUNC('month', {G2_DATE})::DATE AS complaint_month,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Checking or savings account'
      AND {G2_DATE} >= DATE '2026-01-01'
      AND {G2_DATE} < DATE '2026-07-01'
      AND {G2_TAGS} ILIKE '%Older American%'
    GROUP BY 1
    ORDER BY complaint_month
    """
)


# ------------------------------------------------------------
# G2-03 — Lowest timely response rate
# ------------------------------------------------------------

add_g2_question(
    "G2-03",
    "Among companies with at least 200 complaints from January through June 2026, which five companies had the lowest timely response rate?",
    "group_order",
    f"""
    SELECT
        {G2_COMPANY} AS company,
        COUNT(*) AS complaint_count,
        COUNT(*) FILTER (
            WHERE {G2_TIMELY} = 'Yes'
        ) AS timely_count,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE {G2_TIMELY} = 'Yes'
            ) / COUNT(*),
            2
        ) AS timely_response_pct
    FROM v_complaints
    WHERE {G2_WINDOW}
    GROUP BY 1
    HAVING COUNT(*) >= 200
    ORDER BY timely_response_pct ASC, complaint_count DESC
    LIMIT 5
    """
)


# ------------------------------------------------------------
# G2-04 — Monetary relief share by product
# ------------------------------------------------------------

add_g2_question(
    "G2-04",
    "What percentage of complaints received monetary relief, broken down by product, from January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_PRODUCT} AS product,
        COUNT(*) AS total_complaints,
        COUNT(*) FILTER (
            WHERE {G2_RESPONSE} ILIKE '%monetary relief%'
        ) AS monetary_relief_count,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE {G2_RESPONSE} ILIKE '%monetary relief%'
            ) / COUNT(*),
            2
        ) AS monetary_relief_pct
    FROM v_complaints
    WHERE {G2_WINDOW}
    GROUP BY 1
    ORDER BY monetary_relief_pct DESC
    """
)


# ------------------------------------------------------------
# G2-05 — Fastest growing purchase sub-issue
# ------------------------------------------------------------

assert "sub_issue" in G2_COL, (
    "G2-05 requires the sub_issue column."
)

G2_SUBISSUE = g2_qcol("sub_issue")

add_g2_question(
    "G2-05",
    "Which sub-issue under Problem with a purchase shown on your statement had the largest increase in complaint count from Q1 to Q2 2026?",
    "date_filter_group_order",
    f"""
    WITH quarterly AS (
        SELECT
            {G2_SUBISSUE} AS sub_issue,
            COUNT(*) FILTER (
                WHERE {G2_DATE} < DATE '2026-04-01'
            ) AS q1_count,
            COUNT(*) FILTER (
                WHERE {G2_DATE} >= DATE '2026-04-01'
            ) AS q2_count
        FROM v_complaints
        WHERE {G2_PRODUCT} = 'Credit card'
          AND {G2_ISSUE} =
              'Problem with a purchase shown on your statement'
          AND {G2_DATE} >= DATE '2026-01-01'
          AND {G2_DATE} < DATE '2026-07-01'
          AND {G2_SUBISSUE} IS NOT NULL
        GROUP BY 1
    )
    SELECT
        sub_issue,
        q1_count,
        q2_count,
        q2_count - q1_count AS count_change
    FROM quarterly
    ORDER BY count_change DESC, sub_issue ASC
    LIMIT 10
    """
)


# ------------------------------------------------------------
# G2-06 — Monthly complaints by product
# ------------------------------------------------------------

add_g2_question(
    "G2-06",
    "Show monthly complaint counts by product for January through June 2026.",
    "date_filter_group_order",
    f"""
    SELECT
        DATE_TRUNC('month', {G2_DATE})::DATE AS complaint_month,
        {G2_PRODUCT} AS product,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
    GROUP BY 1, 2
    ORDER BY complaint_month, product
    """
)


# ------------------------------------------------------------
# G2-07 — Credit card monthly trend
# ------------------------------------------------------------

add_g2_question(
    "G2-07",
    "How many credit card complaints were received in each month from January through June 2026?",
    "date_filter_group_order",
    f"""
    SELECT
        DATE_TRUNC('month', {G2_DATE})::DATE AS complaint_month,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Credit card'
      AND {G2_WINDOW}
    GROUP BY 1
    ORDER BY complaint_month
    """
)


# ------------------------------------------------------------
# G2-08 — Top ten companies by complaint volume
# ------------------------------------------------------------

add_g2_question(
    "G2-08",
    "Which ten companies received the most complaints across the two supported products from January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_COMPANY} AS company,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
    GROUP BY 1
    ORDER BY complaint_count DESC, company ASC
    LIMIT 10
    """
)


# ------------------------------------------------------------
# G2-09 — Timely response rate by product
# ------------------------------------------------------------

add_g2_question(
    "G2-09",
    "What was the timely response rate for each supported product from January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_PRODUCT} AS product,
        COUNT(*) AS total_complaints,
        COUNT(*) FILTER (
            WHERE {G2_TIMELY} = 'Yes'
        ) AS timely_count,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE {G2_TIMELY} = 'Yes'
            ) / COUNT(*),
            2
        ) AS timely_response_pct
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
    GROUP BY 1
    ORDER BY product
    """
)


# ------------------------------------------------------------
# G2-10 — Top checking account issues in Q2
# ------------------------------------------------------------

add_g2_question(
    "G2-10",
    "What were the ten most common issues for Checking or savings account complaints in Q2 2026?",
    "date_filter_group_order",
    f"""
    SELECT
        {G2_ISSUE} AS issue,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Checking or savings account'
      AND {G2_DATE} >= DATE '2026-04-01'
      AND {G2_DATE} < DATE '2026-07-01'
    GROUP BY 1
    ORDER BY complaint_count DESC, issue ASC
    LIMIT 10
    """
)


# ------------------------------------------------------------
# G2-11 — Complaints by submission channel
# ------------------------------------------------------------

assert "submitted_via" in G2_COL, (
    "G2-11 requires submitted_via."
)

G2_CHANNEL = g2_qcol("submitted_via")

add_g2_question(
    "G2-11",
    "How many complaints were submitted through each channel from January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_CHANNEL} AS submitted_via,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
    GROUP BY 1
    ORDER BY complaint_count DESC, submitted_via
    """
)


# ------------------------------------------------------------
# G2-12 — Top states for credit card complaints in June
# ------------------------------------------------------------

add_g2_question(
    "G2-12",
    "Which ten states had the highest number of credit card complaints in June 2026?",
    "date_filter_group_order",
    f"""
    SELECT
        {G2_STATE} AS state,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Credit card'
      AND {G2_DATE} >= DATE '2026-06-01'
      AND {G2_DATE} < DATE '2026-07-01'
      AND {G2_STATE} IS NOT NULL
    GROUP BY 1
    ORDER BY complaint_count DESC, state
    LIMIT 10
    """
)


# ------------------------------------------------------------
# G2-13 — Quarterly complaint counts by product
# ------------------------------------------------------------

add_g2_question(
    "G2-13",
    "Compare Q1 and Q2 2026 complaint counts for each supported product.",
    "date_filter_group_order",
    f"""
    SELECT
        CASE
            WHEN {G2_DATE} < DATE '2026-04-01' THEN 'Q1 2026'
            ELSE 'Q2 2026'
        END AS quarter,
        {G2_PRODUCT} AS product,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
    GROUP BY 1, 2
    ORDER BY quarter, product
    """
)


# ------------------------------------------------------------
# G2-14 — Sub-product distribution
# ------------------------------------------------------------

assert "sub_product" in G2_COL, (
    "G2-14 requires sub_product."
)

G2_SUBPRODUCT = g2_qcol("sub_product")

add_g2_question(
    "G2-14",
    "What are the complaint counts by sub-product for each supported product during January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_PRODUCT} AS product,
        {G2_SUBPRODUCT} AS sub_product,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
    GROUP BY 1, 2
    ORDER BY product, complaint_count DESC
    LIMIT 200
    """
)


# ------------------------------------------------------------
# G2-15 — Daily June complaint volume
# ------------------------------------------------------------

add_g2_question(
    "G2-15",
    "Show the daily complaint counts for both supported products during June 2026.",
    "date_filter_group_order",
    f"""
    SELECT
        {G2_DATE} AS complaint_date,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
      AND {G2_DATE} >= DATE '2026-06-01'
      AND {G2_DATE} < DATE '2026-07-01'
    GROUP BY 1
    ORDER BY complaint_date
    """
)


# ------------------------------------------------------------
# G2-16 — Timely response by submission channel
# ------------------------------------------------------------

add_g2_question(
    "G2-16",
    "Compare timely and untimely response counts by submission channel from January through June 2026.",
    "group_order",
    f"""
    SELECT
        {G2_CHANNEL} AS submitted_via,
        {G2_TIMELY} AS timely_response,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
    GROUP BY 1, 2
    ORDER BY submitted_via, timely_response
    """
)


# ------------------------------------------------------------
# G2-17 — Most common credit card issues
# ------------------------------------------------------------

add_g2_question(
    "G2-17",
    "What were the ten most common credit card complaint issues during January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_ISSUE} AS issue,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Credit card'
      AND {G2_WINDOW}
    GROUP BY 1
    ORDER BY complaint_count DESC, issue
    LIMIT 10
    """
)


# ------------------------------------------------------------
# G2-18 — Monthly monetary relief rate
# ------------------------------------------------------------

add_g2_question(
    "G2-18",
    "How did the monthly percentage of complaints receiving monetary relief change from January through June 2026?",
    "date_filter_group_order",
    f"""
    SELECT
        DATE_TRUNC('month', {G2_DATE})::DATE AS complaint_month,
        COUNT(*) AS total_complaints,
        COUNT(*) FILTER (
            WHERE {G2_RESPONSE} ILIKE '%monetary relief%'
        ) AS monetary_relief_count,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE {G2_RESPONSE} ILIKE '%monetary relief%'
            ) / COUNT(*),
            2
        ) AS monetary_relief_pct
    FROM v_complaints
    WHERE {G2_WINDOW}
    GROUP BY 1
    ORDER BY complaint_month
    """
)


# ------------------------------------------------------------
# G2-19 — Unanswerable: per-capita complaints
# ------------------------------------------------------------

add_g2_question(
    "G2-19",
    "Which state has the highest number of complaints per 100,000 residents during January through June 2026?",
    "unanswerable",
    answerable=False,
    refusal_reason=(
        "The v_complaints view contains complaint counts and state "
        "but no state population or demographic denominator. "
        "Per-capita rates cannot be calculated from this dataset."
    )
)


# ------------------------------------------------------------
# G2-20 — Unanswerable: consumer income
# ------------------------------------------------------------

add_g2_question(
    "G2-20",
    "What is the average annual household income of consumers who submitted complaints, broken down by product?",
    "unanswerable",
    answerable=False,
    refusal_reason=(
        "The v_complaints view does not contain consumer household "
        "income. The requested average cannot be calculated."
    )
)


# ------------------------------------------------------------
# Validate question definitions
# ------------------------------------------------------------

assert len(g2_questions) == 20

assert len([
    q for q in g2_questions
    if q["answerable"]
]) == 18

assert len([
    q for q in g2_questions
    if not q["answerable"]
]) == 2

assert len([
    q for q in g2_questions
    if q["category"] == "unanswerable"
]) == 2

assert all(
    q["reference_sql"]
    for q in g2_questions
    if q["answerable"]
)

print("=" * 70)
print("G2 — GOLD QUESTION INVENTORY")
print("=" * 70)

display(pd.DataFrame([
    {
        "question_id": q["question_id"],
        "category": q["category"],
        "answerable": q["answerable"],
        "question": q["question"]
    }
    for q in g2_questions
]))

print("\nTotal questions:", len(g2_questions))
print("Answerable:", sum(q["answerable"] for q in g2_questions))
print("Unanswerable:", sum(not q["answerable"] for q in g2_questions))

print("\nPASS — 20 G2 gold questions defined.")

G2 — GOLD QUESTION INVENTORY


,question_id,category,answerable,question
0,G2-01,date_filter_group_order,True,What were the five most common issues for cred...
1,G2-02,date_filter_group_order,True,Show monthly complaint counts for Checking or ...
2,G2-03,group_order,True,Among companies with at least 200 complaints f...
3,G2-04,group_order,True,What percentage of complaints received monetar...
4,G2-05,date_filter_group_order,True,Which sub-issue under Problem with a purchase ...
5,G2-06,date_filter_group_order,True,Show monthly complaint counts by product for J...
6,G2-07,date_filter_group_order,True,How many credit card complaints were received ...
7,G2-08,group_order,True,Which ten companies received the most complain...
8,G2-09,group_order,True,What was the timely response rate for each sup...
9,G2-10,date_filter_group_order,True,What were the ten most common issues for Check...



Total questions: 20
Answerable: 18
Unanswerable: 2

PASS — 20 G2 gold questions defined.


In [32]:
# ============================================================
# PHASE 7 — G2 CELL 3
# EXECUTE REFERENCE SQL AND SAVE EXPECTED RESULTS
# ============================================================

g2_expected_records = []
g2_execution_summary = []

for item in g2_questions:

    question_id = item["question_id"]

    print("\n" + "=" * 70)
    print(question_id, "|", item["category"])
    print("=" * 70)

    if not item["answerable"]:

        g2_expected_records.append({
            "question_id": question_id,
            "answerable": False,
            "expected_rows": None,
            "expected_row_count": None,
            "refusal_reason": item["refusal_reason"]
        })

        g2_execution_summary.append({
            "question_id": question_id,
            "answerable": False,
            "execution_status": "EXPECTED_REFUSAL",
            "expected_row_count": None
        })

        print("Expected outcome: REFUSAL")
        print("Reason:", item["refusal_reason"])

        continue

    sql = item["reference_sql"]

    # Validate every reference query through the same SQL guard.
    safe_sql = guard_sql(sql)

    # Execute on the read-only database connection.
    result_df = ro.execute(safe_sql).fetchdf()

    rows = [
        {
            str(k): g2_json_safe(v)
            for k, v in row.items()
        }
        for row in result_df.to_dict("records")
    ]

    g2_expected_records.append({
        "question_id": question_id,
        "answerable": True,
        "expected_rows": rows,
        "expected_row_count": len(rows),
        "refusal_reason": None
    })

    g2_execution_summary.append({
        "question_id": question_id,
        "answerable": True,
        "execution_status": "PASS",
        "expected_row_count": len(rows)
    })

    print("Reference SQL: PASS")
    print("Expected rows:", len(rows))

    if rows:
        display(result_df.head(10))


# ------------------------------------------------------------
# Save reference results
# ------------------------------------------------------------

with open(G2_EXPECTED_FILE, "w", encoding="utf-8") as f:

    for record in g2_expected_records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )


# Save question definitions.
with open(G2_GOLD_FILE, "w", encoding="utf-8") as f:

    for item in g2_questions:
        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            ) + "\n"
        )


g2_summary_df = pd.DataFrame(g2_execution_summary)

g2_summary_df.to_csv(
    G2_SUMMARY_FILE,
    index=False
)

print("\n" + "=" * 70)
print("G2 — REFERENCE SQL EXECUTION SUMMARY")
print("=" * 70)

display(g2_summary_df)

answerable_passes = sum(
    x["execution_status"] == "PASS"
    for x in g2_execution_summary
)

refusal_cases = sum(
    x["execution_status"] == "EXPECTED_REFUSAL"
    for x in g2_execution_summary
)

print("Answerable reference queries passed:", answerable_passes)
print("Expected refusal cases:", refusal_cases)

assert answerable_passes == 18
assert refusal_cases == 2
assert len(g2_expected_records) == 20

print("\nPASS — All G2 reference queries executed and saved.")


G2-01 | date_filter_group_order
Reference SQL: PASS
Expected rows: 5


,issue,complaint_count
0,Problem with a purchase shown on your statement,995
1,"Other features, terms, or problems",359
2,Fees or interest,345
3,Getting a credit card,256
4,Closing your account,222



G2-02 | date_filter_group_order
Reference SQL: PASS
Expected rows: 6


,complaint_month,complaint_count
0,2026-01-01,280
1,2026-02-01,169
2,2026-03-01,290
3,2026-04-01,273
4,2026-05-01,254
5,2026-06-01,304



G2-03 | group_order
Reference SQL: PASS
Expected rows: 5


,company,complaint_count,timely_count,timely_response_pct
0,FinCo Services Inc DBA Current,237,189,79.75
1,"Bilt Technologies, Inc.",213,173,81.22
2,SYNCHRONY FINANCIAL,2083,1948,93.52
3,ROBINHOOD MARKETS INC.,240,237,98.75
4,"BANK OF AMERICA, NATIONAL ASSOCIATION",3090,3054,98.83


... (29 outputs left out)

,complaint_month,total_complaints,monetary_relief_count,monetary_relief_pct
0,2026-01-01,7580,1691,22.31
1,2026-02-01,3962,894,22.56
2,2026-03-01,6812,1499,22.01
3,2026-04-01,7380,1625,22.02
4,2026-05-01,6079,1362,22.41
5,2026-06-01,6110,1411,23.09



G2-19 | unanswerable
Expected outcome: REFUSAL
Reason: The v_complaints view contains complaint counts and state but no state population or demographic denominator. Per-capita rates cannot be calculated from this dataset.

G2-20 | unanswerable
Expected outcome: REFUSAL
Reason: The v_complaints view does not contain consumer household income. The requested average cannot be calculated.

G2 — REFERENCE SQL EXECUTION SUMMARY


,question_id,answerable,execution_status,expected_row_count
0,G2-01,True,PASS,5.0
1,G2-02,True,PASS,6.0
2,G2-03,True,PASS,5.0
3,G2-04,True,PASS,2.0
4,G2-05,True,PASS,3.0
5,G2-06,True,PASS,12.0
6,G2-07,True,PASS,6.0
7,G2-08,True,PASS,10.0
8,G2-09,True,PASS,2.0
9,G2-10,True,PASS,10.0


Answerable reference queries passed: 18
Expected refusal cases: 2

PASS — All G2 reference queries executed and saved.


In [33]:
# ============================================================
# PHASE 7 — G2 CELL 4
# GOLD DATASET ACCEPTANCE VALIDATION
# ============================================================

import json
import hashlib
import pandas as pd

# Reload the saved question definitions.
with open(G2_GOLD_FILE, encoding="utf-8") as f:
    g2_gold_reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

# Reload expected result sets.
with open(G2_EXPECTED_FILE, encoding="utf-8") as f:
    g2_expected_reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

# Reload summary.
g2_summary_reloaded = pd.read_csv(
    G2_SUMMARY_FILE
)

# 1. Count validation.
assert len(g2_gold_reloaded) == 20
assert len(g2_expected_reloaded) == 20

# 2. Unique question IDs.
g2_ids = [
    x["question_id"]
    for x in g2_gold_reloaded
]

assert len(g2_ids) == len(set(g2_ids))

# 3. Answerable and unanswerable counts.
answerable_items = [
    x for x in g2_gold_reloaded
    if x["answerable"]
]

refusal_items = [
    x for x in g2_gold_reloaded
    if not x["answerable"]
]

assert len(answerable_items) == 18
assert len(refusal_items) == 2

# 4. Every answerable question has reference SQL.
for item in answerable_items:

    assert item["reference_sql"].strip()

    parsed = sqlglot.parse(
        item["reference_sql"],
        read="duckdb"
    )

    assert len(parsed) == 1
    assert isinstance(parsed[0], sqlglot.exp.Select)

    safe = guard_sql(item["reference_sql"])

    assert "v_complaints" in safe.lower()

# 5. Every refusal question has a reason.
for item in refusal_items:

    assert item["refusal_reason"]
    assert item["reference_sql"] is None

# 6. Expected result records.
expected_by_id = {
    x["question_id"]: x
    for x in g2_expected_reloaded
}

for item in answerable_items:

    expected = expected_by_id[item["question_id"]]

    assert expected["answerable"] is True
    assert expected["expected_rows"] is not None
    assert expected["expected_row_count"] == len(
        expected["expected_rows"]
    )

for item in refusal_items:

    expected = expected_by_id[item["question_id"]]

    assert expected["answerable"] is False
    assert expected["refusal_reason"]

# 7. All answerable reference executions passed.
assert (
    g2_summary_reloaded["execution_status"] == "PASS"
).sum() == 18

assert (
    g2_summary_reloaded["execution_status"] == "EXPECTED_REFUSAL"
).sum() == 2


# 8. SHA256 checksums.
def g2_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


print("=" * 70)
print("COMPLAINTIQ — G2 ACCEPTANCE VALIDATION")
print("=" * 70)

print("Total gold questions:", len(g2_gold_reloaded))
print("Unique question IDs:", len(set(g2_ids)))
print("Answerable questions:", len(answerable_items))
print("Unanswerable questions:", len(refusal_items))

print(
    "Reference SQL execution passes:",
    int((
        g2_summary_reloaded["execution_status"] == "PASS"
    ).sum())
)

print(
    "Expected refusal cases:",
    int((
        g2_summary_reloaded["execution_status"] == "EXPECTED_REFUSAL"
    ).sum())
)

print("\nSaved artifacts:")

for path in [
    G2_GOLD_FILE,
    G2_EXPECTED_FILE,
    G2_SUMMARY_FILE
]:

    assert path.exists()

    print("\n", path)
    print("Size:", path.stat().st_size, "bytes")
    print("SHA256:", g2_sha256(path))


print("\nPASS — G2 GOLD DATASET STRUCTURAL ACCEPTANCE.")

COMPLAINTIQ — G2 ACCEPTANCE VALIDATION
Total gold questions: 20
Unique question IDs: 20
Answerable questions: 18
Unanswerable questions: 2
Reference SQL execution passes: 18
Expected refusal cases: 2

Saved artifacts:

 /home/sagemaker-user/complaintiq/eval/gold/G2_analytics_gold.jsonl
Size: 13141 bytes
SHA256: 21f268c50803d8286db93377dbf5d2d342e3b63fa302216d2e33239ed1bd133b

 /home/sagemaker-user/complaintiq/eval/gold/G2_analytics_expected.jsonl
Size: 12950 bytes
SHA256: 9e57b566bc49d056d1b6e818b78037eb62af0553f3d3252c76b1aa830a87bbac

 /home/sagemaker-user/complaintiq/eval/results/G2_reference_results_summary.csv
Size: 485 bytes
SHA256: bc83b3ca5dff7a962388d1b0364d9b142480d8231f288de0090b5cce891330f9

PASS — G2 GOLD DATASET STRUCTURAL ACCEPTANCE.


In [34]:
# ============================================================
# PHASE 7 — G3 CELL 43
# REGULATORY CORPUS INSPECTION
# ============================================================

import re
import json
import pandas as pd
from collections import Counter

assert "REG_INDEX" in globals()
assert len(REG_INDEX.chunks) == 1784

g3_chunks = REG_INDEX.chunks

print("=" * 70)
print("G3 — REGULATORY CORPUS INSPECTION")
print("=" * 70)

print("Total indexed chunks:", len(g3_chunks))

# Inspect metadata structure.
print("\nFirst chunk metadata:")
print(json.dumps(g3_chunks[0], indent=2, ensure_ascii=False)[:3500])

# Source distribution.
source_counts = Counter(
    str(c.get("source_type", "UNKNOWN"))
    for c in g3_chunks
)

print("\nSource type distribution:")
display(
    pd.DataFrame(
        source_counts.items(),
        columns=["source_type", "chunk_count"]
    )
)

# Source file distribution.
file_counts = Counter(
    str(c.get("source_file", "UNKNOWN"))
    for c in g3_chunks
)

print("\nSource file distribution:")
display(
    pd.DataFrame(
        file_counts.items(),
        columns=["source_file", "chunk_count"]
    ).sort_values("chunk_count", ascending=False)
)

# Extract CFR references from metadata and text.
cfr_pattern = re.compile(
    r"12\s*CFR\s*(1005|1026|1022|1006|1030)\.(\d+[a-z]?)",
    re.I
)

g3_cfr_counts = Counter()

for chunk in g3_chunks:

    searchable = " ".join([
        str(chunk.get("chunk_id", "")),
        str(chunk.get("source_id", "")),
        str(chunk.get("citation", "")),
        str(chunk.get("title", "")),
        str(chunk.get("text", ""))
    ])

    for part, section in cfr_pattern.findall(searchable):
        g3_cfr_counts[f"12 CFR {part}.{section}"] += 1


print("\nCFR sections detected in corpus:")
display(
    pd.DataFrame(
        g3_cfr_counts.items(),
        columns=["cfr_section", "matching_chunks"]
    ).sort_values("cfr_section")
)

assert len(g3_cfr_counts) > 0

print("\nPASS — Regulatory corpus metadata inspected.")

G3 — REGULATORY CORPUS INSPECTION
Total indexed chunks: 1784

First chunk metadata:
{
  "chunk_id": "12CFR1002_1002.1_chunk_0001",
  "source_type": "ecfr",
  "source_id": "12CFR1002_1002.1",
  "source_file": "title_12_part_1002_2026-11-03.xml",
  "citation": "12 CFR Part 1002, Section 1002.1",
  "title": "§ 1002.1 Authority, scope and purpose.",
  "chunk_number": 1,
  "text": "(a) Authority and scope. This part, known as Regulation B, is issued by the Bureau of Consumer Financial Protection (Bureau) pursuant to title VII (Equal Credit Opportunity Act) of the Consumer Credit Protection Act, as amended (15 U.S.C. 1601 et seq. ). Except as otherwise provided herein, this subpart applies to all persons who are creditors, as defined in § 1002.2(l), other than a person excluded from coverage of this part by section 1029 of the Consumer Financial Protection Act of 2010, title X of the Dodd-Frank Wall Street Reform and Consumer Protection Act, Public Law 111-203, 124 Stat. 1376. Information co

,source_type,chunk_count
0,ecfr,646
1,pdf,1138



Source file distribution:


,source_file,chunk_count
6,exam_manual_v2_2012.pdf,1029
4,title_12_part_1026_2026-11-03.xml,358
1,title_12_part_1005_2026-11-03.xml,100
3,title_12_part_1022_2026-11-03.xml,99
0,title_12_part_1002_2026-11-03.xml,55
9,sh_issue22_2020-09.pdf,37
2,title_12_part_1006_2026-11-03.xml,31
10,sh_issue23_2021-01.pdf,30
7,sh_issue17_2018-09.pdf,22
8,sh_issue18_2019-03.pdf,20



CFR sections detected in corpus:


,cfr_section,matching_chunks
68,12 CFR 1005.10,33
15,12 CFR 1005.11,38
14,12 CFR 1005.12,7
69,12 CFR 1005.13,5
74,12 CFR 1005.14,6
...,...,...
85,12 CFR 1030.5,41
82,12 CFR 1030.6,31
84,12 CFR 1030.7,31
80,12 CFR 1030.8,103



PASS — Regulatory corpus metadata inspected.


In [35]:
# ============================================================
# PHASE 7 — G3 CELL 44
# DEFINE REGULATORY GOLD QUESTIONS
# ============================================================

g3_questions = []

def add_g3_question(
    question_id,
    question,
    regulation,
    expected_sections=None,
    expected_source=None,
    answerable=True,
    abstention_reason=None
):

    g3_questions.append({
        "question_id": question_id,
        "question": question,
        "regulation": regulation,
        "answerable": answerable,
        "expected_sections": expected_sections or [],
        "expected_source": expected_source,
        "gold_chunk_ids": [],
        "abstention_reason": abstention_reason
    })


# ------------------------------------------------------------
# REGULATION E — 12 CFR PART 1005
# ------------------------------------------------------------

add_g3_question(
    "G3-01",
    "After a consumer notifies a financial institution of an electronic fund transfer error, what investigation and resolution steps are required, and within what time limits?",
    "Regulation E",
    ["12 CFR 1005.11"]
)

add_g3_question(
    "G3-02",
    "What liability limits apply when an access device is lost or stolen and an unauthorized electronic fund transfer occurs?",
    "Regulation E",
    ["12 CFR 1005.6"]
)

add_g3_question(
    "G3-03",
    "What initial disclosures must a financial institution provide for an electronic fund transfer service?",
    "Regulation E",
    ["12 CFR 1005.7"]
)

add_g3_question(
    "G3-04",
    "What are the requirements for providing periodic statements for accounts subject to Regulation E?",
    "Regulation E",
    ["12 CFR 1005.9"]
)


# ------------------------------------------------------------
# REGULATION Z — 12 CFR PART 1026
# ------------------------------------------------------------

add_g3_question(
    "G3-05",
    "What constitutes a billing error on an open-end credit account, and what must a creditor do after receiving a proper billing-error notice?",
    "Regulation Z",
    ["12 CFR 1026.13"]
)

add_g3_question(
    "G3-06",
    "What limits apply to a consumer's liability for unauthorized use of a credit card?",
    "Regulation Z",
    ["12 CFR 1026.12"]
)

add_g3_question(
    "G3-07",
    "What disclosures must be provided before opening an open-end credit account?",
    "Regulation Z",
    ["12 CFR 1026.6"]
)


# ------------------------------------------------------------
# REGULATION V — 12 CFR PART 1022
# ------------------------------------------------------------

add_g3_question(
    "G3-08",
    "What duties does a furnisher have after receiving a direct dispute from a consumer about information furnished to a consumer reporting agency?",
    "Regulation V",
    ["12 CFR 1022.43"]
)

add_g3_question(
    "G3-09",
    "What policies and procedures must furnishers maintain to ensure the accuracy and integrity of information they report to consumer reporting agencies?",
    "Regulation V",
    ["12 CFR 1022.42"]
)

add_g3_question(
    "G3-10",
    "What types of direct disputes are excluded from the furnisher investigation requirements, and what happens when a dispute is frivolous or irrelevant?",
    "Regulation V",
    ["12 CFR 1022.43"]
)


# ------------------------------------------------------------
# REGULATION F — 12 CFR PART 1006
# ------------------------------------------------------------

add_g3_question(
    "G3-11",
    "What restrictions apply to the frequency and timing of telephone calls by a debt collector to a consumer?",
    "Regulation F",
    ["12 CFR 1006.14"]
)

add_g3_question(
    "G3-12",
    "When and how may a debt collector communicate with a consumer, and what restrictions apply to communication at inconvenient times or places?",
    "Regulation F",
    ["12 CFR 1006.6"]
)

add_g3_question(
    "G3-13",
    "What information must a debt collector provide in a validation notice, and when must it be sent?",
    "Regulation F",
    ["12 CFR 1006.34"]
)


# ------------------------------------------------------------
# REGULATION DD — 12 CFR PART 1030
# ------------------------------------------------------------

add_g3_question(
    "G3-14",
    "What account-opening disclosures must a depository institution provide under Regulation DD?",
    "Regulation DD",
    ["12 CFR 1030.4"]
)

add_g3_question(
    "G3-15",
    "What rules govern the payment of interest on interest-bearing deposit accounts?",
    "Regulation DD",
    ["12 CFR 1030.7"]
)

add_g3_question(
    "G3-16",
    "What requirements apply to the calculation of interest on deposit accounts?",
    "Regulation DD",
    ["12 CFR 1030.7"]
)


# ------------------------------------------------------------
# SUPERVISORY DOCUMENTS
# ------------------------------------------------------------

add_g3_question(
    "G3-17",
    "What consumer-protection or compliance findings concerning deposit accounts are described in the CFPB Supervisory Highlights documents included in the corpus?",
    "CFPB Supervisory Highlights",
    [],
    expected_source="supervisory_highlights"
)


# ------------------------------------------------------------
# THREE QUESTIONS THAT MUST BE REFUSED
# ------------------------------------------------------------

add_g3_question(
    "G3-18",
    "What is Harbourline Bank's internal policy on cryptocurrency custody, and which internal approval committee authorizes it?",
    "Not in corpus",
    answerable=False,
    abstention_reason=(
        "The corpus contains no Harbourline internal cryptocurrency "
        "custody policy or internal committee records."
    )
)

add_g3_question(
    "G3-19",
    "What is the exact contractual refund processing time promised by Harbourline Bank to its customers?",
    "Not in corpus",
    answerable=False,
    abstention_reason=(
        "The corpus contains regulations and supervisory documents, "
        "not Harbourline's internal customer contracts or service-level commitments."
    )
)

add_g3_question(
    "G3-20",
    "Which named Harbourline employee approved a specific customer's refund last week?",
    "Not in corpus",
    answerable=False,
    abstention_reason=(
        "The corpus contains no Harbourline employee records or internal "
        "customer-level approval history."
    )
)


# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

assert len(g3_questions) == 20

assert sum(q["answerable"] for q in g3_questions) == 17

assert sum(not q["answerable"] for q in g3_questions) == 3

assert len({
    q["question_id"] for q in g3_questions
}) == 20

print("=" * 70)
print("G3 — QUESTION INVENTORY")
print("=" * 70)

display(pd.DataFrame([
    {
        "question_id": q["question_id"],
        "regulation": q["regulation"],
        "answerable": q["answerable"],
        "expected_sections": ", ".join(q["expected_sections"]),
        "question": q["question"]
    }
    for q in g3_questions
]))

print("\nTotal questions:", len(g3_questions))
print("Answerable:", sum(q["answerable"] for q in g3_questions))
print("Abstention:", sum(not q["answerable"] for q in g3_questions))

print("\nPASS — 20 G3 questions defined.")

G3 — QUESTION INVENTORY


,question_id,regulation,answerable,expected_sections,question
0,G3-01,Regulation E,True,12 CFR 1005.11,After a consumer notifies a financial institut...
1,G3-02,Regulation E,True,12 CFR 1005.6,What liability limits apply when an access dev...
2,G3-03,Regulation E,True,12 CFR 1005.7,What initial disclosures must a financial inst...
3,G3-04,Regulation E,True,12 CFR 1005.9,What are the requirements for providing period...
4,G3-05,Regulation Z,True,12 CFR 1026.13,What constitutes a billing error on an open-en...
5,G3-06,Regulation Z,True,12 CFR 1026.12,What limits apply to a consumer's liability fo...
6,G3-07,Regulation Z,True,12 CFR 1026.6,What disclosures must be provided before openi...
7,G3-08,Regulation V,True,12 CFR 1022.43,What duties does a furnisher have after receiv...
8,G3-09,Regulation V,True,12 CFR 1022.42,What policies and procedures must furnishers m...
9,G3-10,Regulation V,True,12 CFR 1022.43,What types of direct disputes are excluded fro...



Total questions: 20
Answerable: 17
Abstention: 3

PASS — 20 G3 questions defined.


In [36]:
# ============================================================
# PHASE 7 — G3 CELL 45
# RESOLVE GOLD SECTION IDS AND CHUNK IDS
# ============================================================

import re
import pandas as pd
from collections import defaultdict

def g3_chunk_searchable(chunk):

    return " ".join([
        str(chunk.get("chunk_id", "")),
        str(chunk.get("source_id", "")),
        str(chunk.get("citation", "")),
        str(chunk.get("title", "")),
        str(chunk.get("source_file", "")),
        str(chunk.get("text", ""))
    ])


def g3_normalize_cfr(section):

    return re.sub(
        r"\s+",
        "",
        str(section).lower()
    ).replace("12cfr", "")


def g3_find_section_chunks(section):

    target = g3_normalize_cfr(section)

    matches = []

    for chunk in g3_chunks:

        searchable = g3_chunk_searchable(chunk)

        normalized = re.sub(
            r"[^a-z0-9.]",
            "",
            searchable.lower()
        )

        if target in normalized:
            matches.append(chunk)

    return matches


g3_resolution_rows = []

for item in g3_questions:

    if not item["answerable"]:
        continue

    if item["expected_sections"]:

        matched_chunks = []

        for section in item["expected_sections"]:

            section_chunks = g3_find_section_chunks(section)

            for chunk in section_chunks:
                if chunk not in matched_chunks:
                    matched_chunks.append(chunk)

        item["gold_chunk_ids"] = [
            str(c["chunk_id"])
            for c in matched_chunks
            if c.get("chunk_id") is not None
        ]

        g3_resolution_rows.append({
            "question_id": item["question_id"],
            "expected_sections": ", ".join(
                item["expected_sections"]
            ),
            "matching_chunks": len(matched_chunks),
            "gold_chunk_ids": ", ".join(
                item["gold_chunk_ids"][:8]
            ),
            "status": (
                "PASS" if item["gold_chunk_ids"] else "MISSING"
            )
        })


# ------------------------------------------------------------
# Supervisory Highlights retrieval candidates
# ------------------------------------------------------------

g3_supervisory_question = next(
    q for q in g3_questions
    if q["question_id"] == "G3-17"
)

g3_supervisory_candidates = REG_INDEX.search(
    g3_supervisory_question["question"],
    k=12
)

g3_supervisory_rows = []

for rank, chunk in enumerate(g3_supervisory_candidates, 1):

    g3_supervisory_rows.append({
        "rank": rank,
        "chunk_id": chunk.get("chunk_id"),
        "source_file": chunk.get("source_file"),
        "citation": chunk.get("citation"),
        "title": chunk.get("title"),
        "text_preview": str(chunk.get("text", ""))[:700]
    })


print("=" * 70)
print("G3 — CFR SECTION RESOLUTION")
print("=" * 70)

g3_resolution_df = pd.DataFrame(g3_resolution_rows)
display(g3_resolution_df)

print("\nSupervisory Highlights retrieval candidates:")
display(pd.DataFrame(g3_supervisory_rows))

# Required sections must exist.
missing_sections = g3_resolution_df[
    g3_resolution_df["status"] != "PASS"
]

if not missing_sections.empty:

    print("\nREVIEW REQUIRED — Missing expected CFR sections:")
    display(missing_sections)

else:

    print("\nPASS — All expected CFR sections resolved.")


# Supervisory answer needs manual source review.
print(
    "\nREVIEW REQUIRED — For G3-17, inspect the candidate text "
    "and select only chunks that actually support the answer."
)

print("\nNo LLM-generated labels were used.")

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

G3 — CFR SECTION RESOLUTION


,question_id,expected_sections,matching_chunks,gold_chunk_ids,status
0,G3-01,12 CFR 1005.11,31,"12CFR1005_1005.1_chunk_0001, 12CFR1005_1005.9_...",PASS
1,G3-02,12 CFR 1005.6,23,"12CFR1005_1005.6_chunk_0001, 12CFR1005_1005.6_...",PASS
2,G3-03,12 CFR 1005.7,33,"12CFR1005_1005.4_chunk_0001, 12CFR1005_1005.5_...",PASS
3,G3-04,12 CFR 1005.9,23,"12CFR1005_1005.7_chunk_0001, 12CFR1005_1005.8_...",PASS
4,G3-05,12 CFR 1026.13,14,"12CFR1005_1005.33_chunk_0005, 12CFR1026_1026.6...",PASS
5,G3-06,12 CFR 1026.12,15,"12CFR1005_1005.12_chunk_0001, 12CFR1005_1005.3...",PASS
6,G3-07,12 CFR 1026.6,133,"12CFR1005_1005.10_chunk_0002, 12CFR1005_1005.1...",PASS
7,G3-08,12 CFR 1022.43,10,"12CFR1022_1022.43_chunk_0001, 12CFR1022_1022.4...",PASS
8,G3-09,12 CFR 1022.42,4,"12CFR1022_1022.42_chunk_0001, exam_manual_v2_2...",PASS
9,G3-10,12 CFR 1022.43,10,"12CFR1022_1022.43_chunk_0001, 12CFR1022_1022.4...",PASS



Supervisory Highlights retrieval candidates:


,rank,chunk_id,source_file,citation,title,text_preview
0,1,sh_issue22_2020-09_page_0003_chunk_0001,sh_issue22_2020-09.pdf,"sh_issue22_2020-09.pdf, page 3",sh_issue22_2020-09,"2 SUPERVISORY HIGHLIGHTS, ISSUE 22 - SUMMER 20..."
1,2,sh_issue17_2018-09_page_0003_chunk_0001,sh_issue17_2018-09.pdf,"sh_issue17_2018-09.pdf, page 3",sh_issue17_2018-09,"2 SUPERVISORY HIGHLIG HTS, ISSUE 17 – SUMMER 2..."
2,3,sh_issue18_2019-03_page_0003_chunk_0001,sh_issue18_2019-03.pdf,"sh_issue18_2019-03.pdf, page 3",sh_issue18_2019-03,"2 SUPERVISORY HIGHLIGHTS, ISSUE 18 – WINTER 20..."
3,4,exam_manual_v2_2012_page_0923_chunk_0001,exam_manual_v2_2012.pdf,"exam_manual_v2_2012.pdf, page 923",exam_manual_v2_2012,CFPB Supervisory Letter <Entity Name> Complian...
4,5,sh_issue18_2019-03_page_0021_chunk_0001,sh_issue18_2019-03.pdf,"sh_issue18_2019-03.pdf, page 21",sh_issue18_2019-03,"20 SUPERVISORY HIGHLIGHTS, ISSUE 18 – WINTER 2..."
5,6,sh_issue18_2019-03_page_0018_chunk_0001,sh_issue18_2019-03.pdf,"sh_issue18_2019-03.pdf, page 18",sh_issue18_2019-03,"17 SUPERVISORY HIGHLIGHTS, ISSUE 18 – WINTER 2..."
6,7,exam_manual_v2_2012_page_0911_chunk_0001,exam_manual_v2_2012.pdf,"exam_manual_v2_2012.pdf, page 911",exam_manual_v2_2012,CFPB Examination Report <Entity Name> Complian...
7,8,sh_issue23_2021-01_page_0028_chunk_0001,sh_issue23_2021-01.pdf,"sh_issue23_2021-01.pdf, page 28",sh_issue23_2021-01,4. Conclusion The Bureau is committed to being...
8,9,sh_issue17_2018-09_page_0023_chunk_0001,sh_issue17_2018-09.pdf,"sh_issue17_2018-09.pdf, page 23",sh_issue17_2018-09,"22 SUPERVISORY HIGHLIG HTS, ISSUE 17 – SUMMER ..."
9,10,exam_manual_v2_2012_page_0921_chunk_0001,exam_manual_v2_2012.pdf,"exam_manual_v2_2012.pdf, page 921",exam_manual_v2_2012,CFPB Supervisory Letter <Entity Name> Review a...



PASS — All expected CFR sections resolved.

REVIEW REQUIRED — For G3-17, inspect the candidate text and select only chunks that actually support the answer.

No LLM-generated labels were used.


In [37]:
# ============================================================
# PHASE 7 — G3 CELL 46
# SAVE GOLD DATASET AND ACCEPTANCE VALIDATION
# ============================================================

import json
import hashlib
import pandas as pd

G3_GOLD_FILE = GOLD_DIR / "G3_regulatory_gold.jsonl"
G3_CANDIDATES_FILE = GOLD_DIR / "G3_supervisory_candidates.jsonl"
G3_SUMMARY_FILE = RESULTS_DIR / "G3_gold_resolution_summary.csv"

# ------------------------------------------------------------
# 1. Assign status to every question
# ------------------------------------------------------------

for item in g3_questions:

    if not item["answerable"]:

        item["gold_status"] = "ABSTENTION_GOLD"

    elif item["question_id"] == "G3-17":

        item["gold_status"] = "PENDING_MANUAL_REVIEW"

    elif item["gold_chunk_ids"]:

        item["gold_status"] = "SECTION_RESOLVED"

    else:

        item["gold_status"] = "MISSING_GOLD_SECTION"


# ------------------------------------------------------------
# 2. Save the 20 question definitions
# ------------------------------------------------------------

with open(G3_GOLD_FILE, "w", encoding="utf-8") as f:

    for item in g3_questions:

        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            ) + "\n"
        )


# ------------------------------------------------------------
# 3. Save supervisory retrieval candidates separately
# ------------------------------------------------------------

with open(G3_CANDIDATES_FILE, "w", encoding="utf-8") as f:

    for candidate in g3_supervisory_rows:

        f.write(
            json.dumps(
                candidate,
                ensure_ascii=False
            ) + "\n"
        )


# ------------------------------------------------------------
# 4. Create resolution summary
# ------------------------------------------------------------

g3_summary_rows = []

for item in g3_questions:

    g3_summary_rows.append({
        "question_id": item["question_id"],
        "regulation": item["regulation"],
        "answerable": item["answerable"],
        "expected_sections": ", ".join(
            item["expected_sections"]
        ),
        "gold_chunk_count": len(item["gold_chunk_ids"]),
        "gold_status": item["gold_status"]
    })


g3_summary_df = pd.DataFrame(g3_summary_rows)

g3_summary_df.to_csv(
    G3_SUMMARY_FILE,
    index=False
)


# ------------------------------------------------------------
# 5. Reload and validate saved files
# ------------------------------------------------------------

with open(G3_GOLD_FILE, encoding="utf-8") as f:

    g3_reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(g3_reloaded) == 20

g3_ids = [
    item["question_id"]
    for item in g3_reloaded
]

assert len(g3_ids) == len(set(g3_ids))

assert sum(
    item["answerable"]
    for item in g3_reloaded
) == 17

assert sum(
    not item["answerable"]
    for item in g3_reloaded
) == 3


# ------------------------------------------------------------
# 6. Validate abstention questions
# ------------------------------------------------------------

for item in g3_reloaded:

    if not item["answerable"]:

        assert item["abstention_reason"]
        assert item["gold_status"] == "ABSTENTION_GOLD"


# ------------------------------------------------------------
# 7. Validate resolved CFR questions
# ------------------------------------------------------------

cfr_questions = [
    item for item in g3_reloaded
    if item["expected_sections"]
]

assert len(cfr_questions) == 16

for item in cfr_questions:

    assert item["gold_chunk_ids"], (
        f"Missing gold chunks for {item['question_id']}"
    )

    assert item["gold_status"] == "SECTION_RESOLVED"

    assert all(
        chunk_id in {
            str(c.get("chunk_id"))
            for c in g3_chunks
        }
        for chunk_id in item["gold_chunk_ids"]
    )


# ------------------------------------------------------------
# 8. SHA256 helper
# ------------------------------------------------------------

def g3_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print("=" * 70)
print("COMPLAINTIQ — G3 STRUCTURAL ACCEPTANCE")
print("=" * 70)

print("Total questions:", len(g3_reloaded))
print("Answerable questions:", sum(
    x["answerable"] for x in g3_reloaded
))
print("Abstention questions:", sum(
    not x["answerable"] for x in g3_reloaded
))
print("CFR questions with resolved chunks:", len(cfr_questions))

print("\nQuestion status:")
display(
    g3_summary_df[
        [
            "question_id",
            "regulation",
            "gold_chunk_count",
            "gold_status"
        ]
    ]
)

print("\nSaved artifacts:")

for path in [
    G3_GOLD_FILE,
    G3_CANDIDATES_FILE,
    G3_SUMMARY_FILE
]:

    assert path.exists()

    print("\n", path)
    print("Size:", path.stat().st_size, "bytes")
    print("SHA256:", g3_sha256(path))


pending = [
    item["question_id"]
    for item in g3_reloaded
    if item["gold_status"] == "PENDING_MANUAL_REVIEW"
]

missing = [
    item["question_id"]
    for item in g3_reloaded
    if item["gold_status"] == "MISSING_GOLD_SECTION"
]

print("\nPending manual review:", pending)
print("Missing gold sections:", missing)

assert not missing, (
    "Some CFR questions have no matching corpus chunks."
)

if pending:

    print(
        "\nREVIEW REQUIRED — G3-17 must be manually checked "
        "against the Supervisory Highlights evidence."
    )

    print(
        "The G3 dataset is saved, but final gold acceptance "
        "is pending that review."
    )

else:

    print("\nPASS — G3 GOLD DATASET STRUCTURAL ACCEPTANCE.")

COMPLAINTIQ — G3 STRUCTURAL ACCEPTANCE
Total questions: 20
Answerable questions: 17
Abstention questions: 3
CFR questions with resolved chunks: 16

Question status:


,question_id,regulation,gold_chunk_count,gold_status
0,G3-01,Regulation E,31,SECTION_RESOLVED
1,G3-02,Regulation E,23,SECTION_RESOLVED
2,G3-03,Regulation E,33,SECTION_RESOLVED
3,G3-04,Regulation E,23,SECTION_RESOLVED
4,G3-05,Regulation Z,14,SECTION_RESOLVED
5,G3-06,Regulation Z,15,SECTION_RESOLVED
6,G3-07,Regulation Z,133,SECTION_RESOLVED
7,G3-08,Regulation V,10,SECTION_RESOLVED
8,G3-09,Regulation V,4,SECTION_RESOLVED
9,G3-10,Regulation V,10,SECTION_RESOLVED



Saved artifacts:

 /home/sagemaker-user/complaintiq/eval/gold/G3_regulatory_gold.jsonl
Size: 20861 bytes
SHA256: 3403b438cec9f11f6c9798091e8906c0a9fce475eb01623b5d9eb0f9548fa8a7

 /home/sagemaker-user/complaintiq/eval/gold/G3_supervisory_candidates.jsonl
Size: 10578 bytes
SHA256: 2a4cd3f820b6992071fee2fe3bdb95f6e21e7e8b8764d8977a6f43b5c68b2a7c

 /home/sagemaker-user/complaintiq/eval/results/G3_gold_resolution_summary.csv
Size: 1216 bytes
SHA256: 687f73b5ddd0fb9cd857e35b5f0676ecf01ebe039308d8d28bb85f24235f0b73

Pending manual review: ['G3-17']
Missing gold sections: []

REVIEW REQUIRED — G3-17 must be manually checked against the Supervisory Highlights evidence.
The G3 dataset is saved, but final gold acceptance is pending that review.


In [41]:
import json

G3_GOLD_PATH = GOLD_DIR / "G3_regulatory_gold.jsonl"

with open(G3_GOLD_PATH, encoding="utf-8") as f:
    g3_gold = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(g3_gold) == 20, f"Expected 20 records, found {len(g3_gold)}"

abstention_patterns = {
    "cryptocurrency custody": "G3-18",
    "contractual refund processing time": "G3-19",
    "approved a specific customer's refund": "G3-20",
}

for item in g3_gold:

    q = str(item.get("question", "")).lower()

    # Default classification and status.
    item["is_abstention"] = False
    item["status"] = (
        "SECTION_RESOLVED"
        if item.get("gold_chunk_ids")
        else "MISSING_GOLD_SECTION"
    )

    # Classify the three intentionally unanswerable questions.
    for phrase, case_id in abstention_patterns.items():
        if phrase in q:
            item["id"] = case_id
            item["is_abstention"] = True
            item["status"] = "ABSTENTION_GOLD"
            item["gold_chunk_ids"] = []
            item["abstention_reason"] = (
                "The requested Harbourline-specific internal "
                "information is not available in the regulatory corpus."
            )
            break

    # G3-17 requires manual review of actual supporting text.
    if "supervisory highlights" in q:
        item["id"] = "G3-17"
        item["is_abstention"] = False
        item["status"] = "PENDING_MANUAL_REVIEW"

# Validate classifications.
answerable = [x for x in g3_gold if not x["is_abstention"]]
abstentions = [x for x in g3_gold if x["is_abstention"]]

assert len(answerable) == 17
assert len(abstentions) == 3

# Validate unique IDs.
ids = [x["id"] for x in g3_gold]
assert len(ids) == len(set(ids)), "Duplicate G3 IDs detected"

# Save.
with open(G3_GOLD_PATH, "w", encoding="utf-8") as f:
    for item in g3_gold:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("G3 CLASSIFICATION VALIDATION")
print("----------------------------")

for item in g3_gold:
    print(
        item["id"],
        "|",
        "ABSTENTION" if item["is_abstention"] else "ANSWERABLE",
        "|",
        item["status"],
        "| gold chunks:",
        len(item.get("gold_chunk_ids", []))
    )

print("\nAnswerable:", len(answerable))
print("Abstention:", len(abstentions))
print("\nPASS — All 20 records have explicit classification and status.")

G3 CLASSIFICATION VALIDATION
----------------------------
G3-01 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 31
G3-02 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 23
G3-03 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 33
G3-04 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 23
G3-05 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 14
G3-06 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 15
G3-07 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 133
G3-08 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 10
G3-09 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 4
G3-10 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 10
G3-11 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 3
G3-12 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 10
G3-13 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 10
G3-14 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 25
G3-15 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 8
G3-16 | ANSWERABLE | SECTION_RESOLVED | gold chunks: 8
G3-17 | ANSWERABLE | PENDING_MANUAL_REVIEW | gold

In [42]:
# G3-17: Supervisory Highlights candidate inspection

supervisory_item = next(
    x for x in g3_gold
    if x.get("id") == "G3-17"
)

supervisory_query = supervisory_item.get(
    "question",
    supervisory_item.get("q", "")
)

assert supervisory_query, "G3-17 question not found"

g3_supervisory_candidates = REG_INDEX.search(
    supervisory_query,
    k=15
)

print("G3-17 QUESTION")
print("----------------")
print(supervisory_query)

print("\nTOP 15 RETRIEVAL CANDIDATES")
print("--------------------------")

for rank, result in enumerate(g3_supervisory_candidates, start=1):

    print(f"\nRANK {rank}")
    print("Chunk ID:", result.get("chunk_id"))
    print("Source file:", result.get("source_file"))
    print("Citation:", result.get("citation"))
    print("Title:", result.get("title"))
    print("Text:")
    print(str(result.get("text", ""))[:1800])
    print("-" * 80)

assert len(g3_supervisory_candidates) > 0

print("\nPASS — Supervisory Highlights candidates retrieved.")

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

G3-17 QUESTION
----------------
What consumer-protection or compliance findings concerning deposit accounts are described in the CFPB Supervisory Highlights documents included in the corpus?

TOP 15 RETRIEVAL CANDIDATES
--------------------------

RANK 1
Chunk ID: sh_issue22_2020-09_page_0003_chunk_0001
Source file: sh_issue22_2020-09.pdf
Citation: sh_issue22_2020-09.pdf, page 3
Title: sh_issue22_2020-09
Text:
2 SUPERVISORY HIGHLIGHTS, ISSUE 22 - SUMMER 2020 1. Introduction The Consumer Financial Protection Bureau (CFPB or Bureau) is committed to a consumer financial marketplace that is free, innovative, competitive, and transparent, where the rights of all parties are protected by the rule of law, and where consumers are free to choose the products and services that best fit their individual needs. To effectively accomplish this, the Bureau remains committed to sharing with the public key findings from its supervisory work to help industry limit risks to consumers and comply with Fede

In [45]:
# CELL 47C — Manually verified supervisory gold chunk selection

# G3-17: CFPB Supervisory Highlights — Deposit account findings
#
# Verified candidate:
# Candidate 15
# Source: sh_issue22_2020-09.pdf
# Page: 11
# Finding: Failure to fulfill advertised deposit account bonus offers.
# Regulation: Regulation DD, 12 CFR 1030.8(a)(1)

SUPERVISORY_GOLD_CHUNK_IDS = [
    "sh_issue22_2020-09_page_0011_chunk_0001"
]

# ---------------------------------------------------------
# Validate selected chunks against the actual corpus
# ---------------------------------------------------------

g3_supervisory_item = next(
    (x for x in g3_gold if x.get("id") == "G3-17"),
    None
)

assert g3_supervisory_item is not None, "G3-17 not found in gold dataset."

assert SUPERVISORY_GOLD_CHUNK_IDS, (
    "At least one manually verified supervisory chunk is required."
)

# Build a lookup of actual regulatory chunks
g3_chunk_lookup = {
    str(chunk.get("chunk_id")): chunk
    for chunk in REG_INDEX.chunks
}

verified_supervisory_chunks = []

for chunk_id in SUPERVISORY_GOLD_CHUNK_IDS:

    assert chunk_id in g3_chunk_lookup, (
        f"Selected chunk does not exist in REG_INDEX: {chunk_id}"
    )

    chunk = g3_chunk_lookup[chunk_id]

    searchable_text = " ".join([
        str(chunk.get("source_file", "")),
        str(chunk.get("citation", "")),
        str(chunk.get("title", "")),
        str(chunk.get("text", ""))
    ]).lower()

    assert "supervisory highlights" in searchable_text, (
        f"Chunk is not identifiable as Supervisory Highlights: {chunk_id}"
    )

    assert (
        "bonus" in searchable_text
        and "deposit" in searchable_text
        and "regulation dd" in searchable_text
    ), (
        f"Chunk does not contain the expected deposit bonus finding: {chunk_id}"
    )

    verified_supervisory_chunks.append(chunk)

# ---------------------------------------------------------
# Assign the verified gold chunks to G3-17
# ---------------------------------------------------------

g3_supervisory_item["gold_chunk_ids"] = (
    SUPERVISORY_GOLD_CHUNK_IDS.copy()
)

g3_supervisory_item["status"] = "MANUALLY_VERIFIED"

g3_supervisory_item["is_abstention"] = False

g3_supervisory_item["gold_notes"] = (
    "CFPB Supervisory Highlights Issue 22, page 10 "
    "(PDF page 11): financial institutions failed to "
    "provide advertised deposit account bonuses to eligible "
    "consumers. The CFPB identified misleading and inaccurate "
    "advertising under Regulation DD, 12 CFR 1030.8(a)(1)."
)

# ---------------------------------------------------------
# Display verification
# ---------------------------------------------------------

print("G3-17 MANUAL GOLD VERIFICATION")
print("=" * 70)

for chunk in verified_supervisory_chunks:
    print("CHUNK ID :", chunk.get("chunk_id"))
    print("SOURCE   :", chunk.get("source_file"))
    print("CITATION :", chunk.get("citation"))
    print("TEXT     :", str(chunk.get("text", ""))[:1200])
    print("-" * 70)

print("G3-17 status:", g3_supervisory_item["status"])
print("Verified gold chunks:", len(g3_supervisory_item["gold_chunk_ids"]))

assert len(g3_supervisory_item["gold_chunk_ids"]) == 1

print("\nPASS — G3-17 supervisory gold chunk manually mapped.")

G3-17 MANUAL GOLD VERIFICATION
CHUNK ID : sh_issue22_2020-09_page_0011_chunk_0001
SOURCE   : sh_issue22_2020-09.pdf
CITATION : sh_issue22_2020-09.pdf, page 11
TEXT     : 10 SUPERVISORY HIGHLIGHTS, ISSUE 22 - SUMMER 2020 documents that the financial institutions relied on in making its determination, as required by Regulation E.31 2.3.4 Failure to fulfill advertised bonus offer Regulation DD requires that advertisements of deposit accounts not mislead, be inaccurate, or misrepresent the financial institution’s deposit contract.32 Examiners found that one or more financial institutions advertised bonuses for consumers who opened an account at the financial institutions and met certain requirements that the advertisement specified. These financial institutions failed to provide the promised bonuses in instances where consumers met the requirements. The financial institutions did not have appropriate quality control and monitoring procedures to ensure all eligible consumers received the bo

In [46]:
# CELL 47D — G3 Gold Set Acceptance Validation

import json
from pathlib import Path

G3_GOLD_PATH = ROOT / "eval" / "gold" / "G3_regulatory_gold.jsonl"

# Save the updated gold dataset
G3_GOLD_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(G3_GOLD_PATH, "w", encoding="utf-8") as f:
    for item in g3_gold:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Reload from disk
with open(G3_GOLD_PATH, "r", encoding="utf-8") as f:
    g3_gold_reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

# Build corpus chunk lookup
g3_valid_chunk_ids = {
    str(chunk.get("chunk_id"))
    for chunk in REG_INDEX.chunks
}

# Basic counts
assert len(g3_gold_reloaded) == 20, (
    f"Expected 20 questions, found {len(g3_gold_reloaded)}"
)

ids = [item["id"] for item in g3_gold_reloaded]

assert len(ids) == len(set(ids)), "Duplicate question IDs found."

abstention_items = [
    x for x in g3_gold_reloaded
    if x.get("is_abstention") is True
]

answerable_items = [
    x for x in g3_gold_reloaded
    if x.get("is_abstention") is not True
]

assert len(abstention_items) == 3, (
    f"Expected 3 abstention questions, found {len(abstention_items)}"
)

assert len(answerable_items) == 17, (
    f"Expected 17 answerable questions, found {len(answerable_items)}"
)

# Validate every answerable question has at least one gold chunk
missing_gold = [
    x["id"] for x in answerable_items
    if not x.get("gold_chunk_ids")
]

assert not missing_gold, (
    f"Answerable questions missing gold chunks: {missing_gold}"
)

# Validate all gold chunk references exist
invalid_chunks = []

for item in answerable_items:
    for chunk_id in item.get("gold_chunk_ids", []):
        if chunk_id not in g3_valid_chunk_ids:
            invalid_chunks.append((item["id"], chunk_id))

assert not invalid_chunks, (
    f"Invalid gold chunk references: {invalid_chunks}"
)

# Validate abstention records
for item in abstention_items:
    assert item.get("status") == "ABSTENTION_GOLD", (
        f"Incorrect abstention status: {item['id']}"
    )
    assert item.get("gold_chunk_ids") == [], (
        f"Abstention question must not have gold chunks: {item['id']}"
    )
    assert item.get("abstention_reason"), (
        f"Missing abstention reason: {item['id']}"
    )

# Validate G3-17 manual mapping
supervisory_item = next(
    x for x in g3_gold_reloaded
    if x["id"] == "G3-17"
)

assert supervisory_item["status"] == "MANUALLY_VERIFIED"
assert supervisory_item["gold_chunk_ids"] == [
    "sh_issue22_2020-09_page_0011_chunk_0001"
]

# Display results
print("G3 GOLD SET ACCEPTANCE")
print("=" * 65)
print("Total questions       :", len(g3_gold_reloaded))
print("Answerable questions  :", len(answerable_items))
print("Abstention questions  :", len(abstention_items))
print("Missing gold mappings:", len(missing_gold))
print("Invalid chunk IDs     :", len(invalid_chunks))
print("G3-17 status          :", supervisory_item["status"])
print("G3-17 gold chunks     :", len(supervisory_item["gold_chunk_ids"]))
print("Saved file            :", G3_GOLD_PATH)

print("\nPASS — G3 GOLD SET ACCEPTED.")

G3 GOLD SET ACCEPTANCE
Total questions       : 20
Answerable questions  : 17
Abstention questions  : 3
Missing gold mappings: 0
Invalid chunk IDs     : 0
G3-17 status          : MANUALLY_VERIFIED
G3-17 gold chunks     : 1
Saved file            : /home/sagemaker-user/complaintiq/eval/gold/G3_regulatory_gold.jsonl

PASS — G3 GOLD SET ACCEPTED.


In [47]:
# CELL 48 — G3 Regulatory Retrieval Evaluation

import pandas as pd
import json
from pathlib import Path

# Use the accepted gold dataset
g3_answerable = [
    x for x in g3_gold_reloaded
    if x.get("is_abstention") is not True
]

G3_RETRIEVAL_K = 10

g3_retrieval_results = []

for item in g3_answerable:

    question_id = item["id"]
    question = item.get("question", item.get("q", ""))
    gold_ids = set(item.get("gold_chunk_ids", []))

    assert question.strip(), f"Missing question text: {question_id}"
    assert gold_ids, f"Missing gold chunks: {question_id}"

    retrieved = REG_INDEX.search(question, k=G3_RETRIEVAL_K)

    retrieved_ids = [
        str(r.get("chunk_id"))
        for r in retrieved
        if r.get("chunk_id") is not None
    ]

    # Recall at 5 and 10
    top5_ids = set(retrieved_ids[:5])
    top10_ids = set(retrieved_ids[:10])

    recall5 = len(gold_ids.intersection(top5_ids)) / len(gold_ids)
    recall10 = len(gold_ids.intersection(top10_ids)) / len(gold_ids)

    # Reciprocal rank of first relevant chunk
    reciprocal_rank = 0.0

    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id in gold_ids:
            reciprocal_rank = 1.0 / rank
            break

    g3_retrieval_results.append({
        "question_id": question_id,
        "question": question,
        "gold_chunk_count": len(gold_ids),
        "retrieved_chunk_count": len(retrieved_ids),
        "recall_at_5": recall5,
        "recall_at_10": recall10,
        "reciprocal_rank": reciprocal_rank,
        "hit_at_5": int(recall5 > 0),
        "hit_at_10": int(recall10 > 0),
        "retrieved_ids": retrieved_ids,
        "gold_ids": sorted(gold_ids)
    })

# Summary
g3_retrieval_df = pd.DataFrame(g3_retrieval_results)

g3_retrieval_summary = {
    "answerable_questions": len(g3_retrieval_df),
    "mean_recall_at_5": g3_retrieval_df["recall_at_5"].mean(),
    "mean_recall_at_10": g3_retrieval_df["recall_at_10"].mean(),
    "hit_rate_at_5": g3_retrieval_df["hit_at_5"].mean(),
    "hit_rate_at_10": g3_retrieval_df["hit_at_10"].mean(),
    "mean_reciprocal_rank": g3_retrieval_df["reciprocal_rank"].mean()
}

print("G3 RETRIEVAL EVALUATION")
print("=" * 70)

for metric, value in g3_retrieval_summary.items():
    if isinstance(value, float):
        print(f"{metric:30s}: {value:.4f}")
    else:
        print(f"{metric:30s}: {value}")

# Failed questions
g3_retrieval_failures = g3_retrieval_df[
    g3_retrieval_df["hit_at_5"] == 0
]

print("\nQUESTIONS WITH NO GOLD CHUNK IN TOP 5")
print("-" * 70)

if len(g3_retrieval_failures):
    for _, row in g3_retrieval_failures.iterrows():
        print(row["question_id"], "-", row["question"])
else:
    print("None")

# Save detailed results
G3_RETRIEVAL_PATH = ROOT / "eval" / "results" / "G3_retrieval_results.csv"
G3_RETRIEVAL_PATH.parent.mkdir(parents=True, exist_ok=True)

g3_retrieval_df.drop(
    columns=["retrieved_ids", "gold_ids"]
).to_csv(G3_RETRIEVAL_PATH, index=False)

# Save summary
G3_RETRIEVAL_SUMMARY_PATH = (
    ROOT / "eval" / "results" / "G3_retrieval_summary.json"
)

with open(G3_RETRIEVAL_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(g3_retrieval_summary, f, indent=2)

print("\nSaved detailed results:", G3_RETRIEVAL_PATH)
print("Saved summary:", G3_RETRIEVAL_SUMMARY_PATH)
print("\nPASS — G3 retrieval evaluation executed.")

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

... (8 outputs left out)

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

G3 RETRIEVAL EVALUATION
answerable_questions          : 17
mean_recall_at_5              : 0.2564
mean_recall_at_10             : 0.3930
hit_rate_at_5                 : 0.9412
hit_rate_at_10                : 0.9412
mean_reciprocal_rank          : 0.7990

QUESTIONS WITH NO GOLD CHUNK IN TOP 5
----------------------------------------------------------------------
G3-17 - What consumer-protection or compliance findings concerning deposit accounts are described in the CFPB Supervisory Highlights documents included in the corpus?

Saved detailed results: /home/sagemaker-user/complaintiq/eval/results/G3_retrieval_results.csv
Saved summary: /home/sagemaker-user/complaintiq/eval/results/G3_retrieval_summary.json

PASS — G3 retrieval evaluation executed.


In [49]:
# CELL 49 — Runtime inspection of regulatory agent nodes
# Safe: no LLM calls, no graph invocation, no S3 writes.

import inspect
import dis

AGENT_NODES = {
    "research_node": research_node,
    "drafter_node": drafter_node,
    "critic_node": critic_node
}

for node_name, node_func in AGENT_NODES.items():

    print("\n" + "=" * 80)
    print(node_name.upper())
    print("=" * 80)

    # Function signature
    try:
        print("SIGNATURE:", inspect.signature(node_func))
    except Exception as e:
        print("Signature unavailable:", repr(e))

    # Function metadata
    print("MODULE:", getattr(node_func, "__module__", None))
    print("QUALNAME:", getattr(node_func, "__qualname__", None))
    print("CODE NAME:", getattr(node_func, "__code__", None).co_name
          if hasattr(node_func, "__code__") else "Unavailable")

    # Try retrieving source without stopping the cell
    try:
        print("\nSOURCE:")
        print(inspect.getsource(node_func))

    except (OSError, TypeError) as e:
        print("\nSOURCE UNAVAILABLE:", str(e))

        # Disassemble runtime bytecode as fallback
        if hasattr(node_func, "__code__"):
            print("\nBYTECODE DISASSEMBLY:")
            dis.dis(node_func)

    # Display referenced global names
    if hasattr(node_func, "__code__"):
        print("\nREFERENCED GLOBAL NAMES:")
        print(node_func.__code__.co_names)

print("\nPASS — Runtime agent inspection completed.")


RESEARCH_NODE
SIGNATURE: (state: __main__.CIQState) -> dict
MODULE: __main__
QUALNAME: research_node
CODE NAME: research_node

SOURCE:

SOURCE UNAVAILABLE: source code not available

BYTECODE DISASSEMBLY:
  1           0 RESUME                   0

  2           2 LOAD_FAST                0 (state)
              4 LOAD_CONST               1 ('clean_request')
              6 BINARY_SUBSCR
             10 STORE_FAST               1 (query)

  4          12 LOAD_FAST                0 (state)
             14 LOAD_ATTR                1 (NULL|self + get)
             34 LOAD_CONST               2 ('triage')
             36 CALL                     1
             44 POP_JUMP_IF_FALSE       52 (to 150)

  5          46 LOAD_FAST                0 (state)
             48 LOAD_CONST               2 ('triage')
             50 BINARY_SUBSCR
             54 STORE_FAST               2 (t)

  7          56 LOAD_FAST                2 (t)
             58 LOAD_CONST               3 ('issue')
           

In [50]:
# CELL 49B — Agent dependency inspection

for name, func in AGENT_NODES.items():

    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)

    if not hasattr(func, "__code__"):
        print("No Python code object available.")
        continue

    print("Arguments:", func.__code__.co_varnames[:func.__code__.co_argcount])
    print("Referenced names:", func.__code__.co_names)

    globals_used = func.__globals__

    for key in func.__code__.co_names:
        if key in globals_used:
            value = globals_used[key]
            print(f"{key}: {type(value).__name__}")


RESEARCH_NODE
Arguments: ('state',)
Referenced names: ('get', 'join', 'REG_INDEX', 'search', 'append')
REG_INDEX: HybridIndex

DRAFTER_NODE
Arguments: ('state',)
Referenced names: ('DRAFT_SYS', 'ValueError', 'get', 'join', 'llm', 'chat', 'MODELS', 'COMMON_DRAFT_RULES')
DRAFT_SYS: dict
llm: module
MODELS: dict
COMMON_DRAFT_RULES: str

CRITIC_NODE
Arguments: ('state',)
Referenced names: ('get', 'str', 'set', 're', 'findall', 'sorted', 'PII', 'search', 'append', 'join', 'llm', 'chat_json', 'MODELS', 'CRITIC_SYS', 'CriticOut', 'verdict', 'model_dump', 'len')
re: module
PII: list
llm: module
MODELS: dict
CRITIC_SYS: str
CriticOut: ModelMetaclass


In [51]:
# CELL 50 — G3 Regulatory Answer Generation

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

g3_answer_results = []

g3_answerable = [
    x for x in g3_gold_reloaded
    if x.get("is_abstention") is not True
]

print("Generating answers for", len(g3_answerable), "answerable questions")

for idx, item in enumerate(g3_answerable, start=1):

    question_id = item["id"]
    question = item.get("question", item.get("q", ""))

    print(f"\n[{idx}/{len(g3_answerable)}] {question_id}")
    print(question)

    try:
        # Research agent
        research_output = research_node({
            "request": question,
            "clean_request": question
        })

        if not isinstance(research_output, dict):
            raise TypeError(
                f"Research output must be dict, got {type(research_output)}"
            )

        # Extract evidence from research output
        evidence = research_output.get("evidence", [])

        if not evidence:
            evidence = research_output.get("regulatory_evidence", [])

        if not evidence:
            evidence = research_output.get("results", [])

        if not isinstance(evidence, list):
            raise TypeError(
                f"Evidence must be a list, got {type(evidence)}"
            )

        # Drafter agent
        draft_output = drafter_node({
            "request": question,
            "clean_request": question,
            "intent": "REG_QA",
            "evidence": evidence,
            "sql_rows": [],
            "similar_cases": [],
            "draft": "",
            "revisions": 0,
            "trace": []
        })

        if not isinstance(draft_output, dict):
            raise TypeError(
                f"Drafter output must be dict, got {type(draft_output)}"
            )

        draft = draft_output.get("draft", "")

        if not draft:
            draft = draft_output.get("answer", "")

        if not draft:
            raise ValueError("Drafter returned no answer.")

        g3_answer_results.append({
            "question_id": question_id,
            "question": question,
            "answer": str(draft),
            "evidence": evidence,
            "evidence_count": len(evidence),
            "research_output": research_output,
            "draft_output": draft_output,
            "generation_status": "SUCCESS",
            "error": None
        })

        print("Status: SUCCESS")
        print("Evidence count:", len(evidence))
        print("Answer preview:", str(draft)[:500])

    except Exception as e:

        g3_answer_results.append({
            "question_id": question_id,
            "question": question,
            "answer": "",
            "evidence": [],
            "evidence_count": 0,
            "research_output": None,
            "draft_output": None,
            "generation_status": "FAILED",
            "error": repr(e)
        })

        print("Status: FAILED")
        print("Error:", repr(e))


# Summary
g3_generation_df = pd.DataFrame([
    {
        "question_id": x["question_id"],
        "generation_status": x["generation_status"],
        "evidence_count": x["evidence_count"],
        "answer_length": len(x["answer"]),
        "error": x["error"]
    }
    for x in g3_answer_results
])

print("\nG3 GENERATION SUMMARY")
print("=" * 70)
print(g3_generation_df.to_string(index=False))

print("\nSuccessful:", sum(
    x["generation_status"] == "SUCCESS"
    for x in g3_answer_results
))

print("Failed:", sum(
    x["generation_status"] == "FAILED"
    for x in g3_answer_results
))

# Persist detailed results
G3_ANSWER_PATH = ROOT / "eval" / "results" / "G3_generated_answers.jsonl"
G3_ANSWER_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(G3_ANSWER_PATH, "w", encoding="utf-8") as f:
    for item in g3_answer_results:
        f.write(json.dumps(item, ensure_ascii=False, default=str) + "\n")

print("\nSaved:", G3_ANSWER_PATH)

Generating answers for 17 answerable questions

[1/17] G3-01
After a consumer notifies a financial institution of an electronic fund transfer error, what investigation and resolution steps are required, and within what time limits?


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Status: SUCCESS
Evidence count: 8
Answer preview: ## Electronic Fund Transfer Error Investigation and Resolution Requirements

### 1. Qualifying Notice of Error

A financial institution must comply with error resolution procedures when it receives oral or written notice from a consumer that:
- Is received **no later than 60 days** after the institution sends the periodic statement reflecting the alleged error [C:12CFR1005_1005.11_chunk_0001]

---

### 2. Standard 10-Business-Day Investigation Period

The default timeline requires the financial 

[2/17] G3-02
What liability limits apply when an access device is lost or stolen and an unauthorized electronic fund transfer occurs?


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Status: SUCCESS
Evidence count: 8
Answer preview: ## Liability Limits for Lost or Stolen Access Devices

When an access device is lost or stolen and unauthorized electronic fund transfers occur, liability is determined based on **when the consumer notifies the financial institution**. The framework operates in tiers:

---

### Threshold Conditions

Before any liability can be imposed, the financial institution must have:
- Provided the disclosures required by § 1005.7(b)(1), (2), and (3)
- Issued an **accepted** access device
- Provided a means

[3/17] G3-03
What initial disclosures must a financial institution provide for an electronic fund transfer service?


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

... (25 outputs left out)

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Status: SUCCESS
Evidence count: 8
Answer preview: ## Requirements for Calculating Interest on Deposit Accounts

### Permissible Balance Calculation Methods

Under 12 CFR 1030.7(a)(1), an institution **must** calculate interest on the full amount of principal in an account for each day using one of two permitted methods [C:exam_manual_v2_2012_page_0764_chunk_0001]:

1. **Daily Balance Method** – The daily periodic rate is applied to the full amount of principal in the account each day.
2. **Average Daily Balance Method** – A periodic rate is app

[17/17] G3-17
What consumer-protection or compliance findings concerning deposit accounts are described in the CFPB Supervisory Highlights documents included in the corpus?


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Status: SUCCESS
Evidence count: 8
Answer preview: ## Deposit Account Compliance Findings in CFPB Supervisory Highlights

### Coverage of Deposit Accounts in the Corpus

The evidence confirms that deposit accounts were examined and findings were included in at least two issues of Supervisory Highlights:

- **Issue 18 (Winter 2019)** covered examinations "in the areas of automobile loan servicing, **deposits**, mortgage servicing, and remittances that were generally completed between June and November 2018" [C:sh_issue18_2019-03_page_0003_chunk_0

G3 GENERATION SUMMARY
question_id generation_status  evidence_count  answer_length error
      G3-01           SUCCESS               8           3322  None
      G3-02           SUCCESS               8           2501  None
      G3-03           SUCCESS               8           3111  None
      G3-04           SUCCESS               8           4098  None
      G3-05           SUCCESS               8           3500  None
      G3-06           SU

In [52]:
# CELL 51 — G3 Citation Validation

import re
import pandas as pd

g3_citation_results = []

for result in g3_answer_results:

    if result["generation_status"] != "SUCCESS":
        continue

    answer = result["answer"]
    evidence = result["evidence"]

    # Extract citations such as [C:chunk_id]
    cited_ids = re.findall(
        r"\[C:([^\]]+)\]",
        answer
    )

    available_ids = {
        str(x.get("chunk_id"))
        for x in evidence
        if isinstance(x, dict) and x.get("chunk_id")
    }

    valid_ids = [
        cid for cid in cited_ids
        if cid in available_ids
    ]

    invalid_ids = [
        cid for cid in cited_ids
        if cid not in available_ids
    ]

    g3_citation_results.append({
        "question_id": result["question_id"],
        "citation_count": len(cited_ids),
        "valid_citations": len(valid_ids),
        "invalid_citations": len(invalid_ids),
        "citation_validity": (
            len(valid_ids) / len(cited_ids)
            if cited_ids else 0.0
        ),
        "has_citation": bool(cited_ids),
        "invalid_chunk_ids": invalid_ids
    })

g3_citation_df = pd.DataFrame(g3_citation_results)

print("G3 CITATION VALIDATION")
print("=" * 70)

if len(g3_citation_df):
    print(g3_citation_df.to_string(index=False))

    print("\nCitation-bearing answer rate:",
          g3_citation_df["has_citation"].mean())

    print("Citation validity:",
          (
              g3_citation_df["valid_citations"].sum()
              / max(g3_citation_df["citation_count"].sum(), 1)
          ))
else:
    print("No successfully generated answers to evaluate.")

G3_CITATION_PATH = ROOT / "eval" / "results" / "G3_citation_results.csv"

g3_citation_df.to_csv(G3_CITATION_PATH, index=False)

print("\nSaved:", G3_CITATION_PATH)

G3 CITATION VALIDATION
question_id  citation_count  valid_citations  invalid_citations  citation_validity  has_citation                                                                                                                                                                                                                                              invalid_chunk_ids
      G3-01               9                9                  0           1.000000          True                                                                                                                                                                                                                                                             []
      G3-02               8                8                  0           1.000000          True                                                                                                                                                                                 

In [53]:
# CELL 52 — G3 Critic / Answer Grounding Evaluation

import json
import pandas as pd
from pathlib import Path

g3_critic_results = []

successful_answers = [
    x for x in g3_answer_results
    if x.get("generation_status") == "SUCCESS"
]

print("Evaluating", len(successful_answers), "generated answers")

for idx, result in enumerate(successful_answers, start=1):

    question_id = result["question_id"]

    print(f"\n[{idx}/{len(successful_answers)}] {question_id}")

    # Construct a state compatible with CIQState and critic_node
    critic_state = {
        "request": result["question"],
        "clean_request": result["question"],
        "intent": "REG_QA",
        "draft": result["answer"],
        "evidence": result["evidence"],
        "sql_rows": [],
        "similar_cases": [],
        "revisions": 0,
        "trace": [],
        "critique": None
    }

    try:
        critic_output = critic_node(critic_state)

        if not isinstance(critic_output, dict):
            raise TypeError(
                f"Expected dict from critic_node, got {type(critic_output)}"
            )

        # Preserve the complete output for later analysis
        critique_value = critic_output.get(
            "critique",
            critic_output.get("critic", critic_output)
        )

        # Extract pass/fail where available
        passed = None

        if isinstance(critique_value, dict):
            passed = critique_value.get(
                "pass",
                critique_value.get("passed")
            )

        if passed is None:
            passed = critic_output.get(
                "pass",
                critic_output.get("passed")
            )

        verdict = critic_output.get(
            "verdict",
            critique_value.get("verdict")
            if isinstance(critique_value, dict)
            else None
        )

        g3_critic_results.append({
            "question_id": question_id,
            "critic_status": "SUCCESS",
            "passed": passed,
            "verdict": verdict,
            "critique": critique_value,
            "critic_output": critic_output,
            "error": None
        })

        print("Critic execution: SUCCESS")
        print("Pass status:", passed)
        print("Verdict:", verdict)
        print("Critique:", str(critique_value)[:700])

    except Exception as e:

        g3_critic_results.append({
            "question_id": question_id,
            "critic_status": "FAILED",
            "passed": None,
            "verdict": None,
            "critique": None,
            "critic_output": None,
            "error": repr(e)
        })

        print("Critic execution: FAILED")
        print("Error:", repr(e))


# Summary
g3_critic_df = pd.DataFrame([
    {
        "question_id": x["question_id"],
        "critic_status": x["critic_status"],
        "passed": x["passed"],
        "verdict": x["verdict"],
        "error": x["error"]
    }
    for x in g3_critic_results
])

print("\nG3 CRITIC SUMMARY")
print("=" * 70)
print(g3_critic_df.to_string(index=False))

critic_success_count = sum(
    x["critic_status"] == "SUCCESS"
    for x in g3_critic_results
)

critic_failed_count = sum(
    x["critic_status"] == "FAILED"
    for x in g3_critic_results
)

critic_with_verdict = [
    x for x in g3_critic_results
    if isinstance(x.get("passed"), bool)
]

print("\nCritic executions successful:", critic_success_count)
print("Critic executions failed:", critic_failed_count)
print(
    "Explicit pass/fail decisions:",
    len(critic_with_verdict)
)

# Save detailed critic results
G3_CRITIC_PATH = ROOT / "eval" / "results" / "G3_critic_results.jsonl"
G3_CRITIC_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(G3_CRITIC_PATH, "w", encoding="utf-8") as f:
    for item in g3_critic_results:
        f.write(
            json.dumps(item, ensure_ascii=False, default=str) + "\n"
        )

# Save summary CSV
G3_CRITIC_CSV = ROOT / "eval" / "results" / "G3_critic_summary.csv"
g3_critic_df.to_csv(G3_CRITIC_CSV, index=False)

print("\nSaved detailed results:", G3_CRITIC_PATH)
print("Saved summary:", G3_CRITIC_CSV)
print("\nPASS — G3 critic evaluation execution completed.")

Evaluating 17 generated answers

[1/17] G3-01
Critic execution: FAILED
Error: ValueError('No JSON object found')

[2/17] G3-02
Critic execution: SUCCESS
Pass status: True
Verdict: PASS
Critique: {'pass': True, 'problems': [], 'verdict': 'PASS', 'unsupported_claims': [], 'tone_issues': [], 'feedback': 'The draft accurately reflects the cited evidence. Tier 1 and Tier 2 liability limits ($50/$500 framework) are supported by 1005.6_chunk_0001 and exam_manual page 0698. The threshold conditions (disclosures under 1005.7(b)(1),(2),(3), accepted access device, means to identify consumer) are directly supported by 1005.6_chunk_0001. Tier 3 unlimited liability after 60 days is consistent with the evidence structure in page 0698. The limitations on imposing liability (no negligence/deposit-agreement basis, may impose less under state law, promptness is sole determinant) are supported by page

[3/17] G3-03
Critic execution: SUCCESS
Pass status: False
Verdict: REVISE
Critique: {'pass': False, 'pr

In [54]:
# CELL 53 — G3 Unanswerable Question / Abstention Evaluation

import re
import json
import pandas as pd

g3_abstention_gold = [
    x for x in g3_gold_reloaded
    if x.get("is_abstention") is True
]

# Conservative refusal indicators.
# These are screening signals, not a substitute for manual review.
REFUSAL_PATTERNS = [
    r"\bnot available in (the )?(provided )?(documents|corpus|sources|evidence)\b",
    r"\bno information (is )?(available|provided)\b",
    r"\bcannot (verify|confirm|determine|establish)\b",
    r"\bunable to (verify|confirm|determine|establish)\b",
    r"\bdo not have (sufficient )?(information|evidence)\b",
    r"\bnot supported by (the )?(available )?(evidence|documents|sources)\b",
    r"\bi (cannot|can't) answer\b",
    r"\bnot enough evidence\b",
    r"\bno supporting evidence\b"
]

def g3_detect_refusal(answer):
    text = str(answer or "").lower()

    return any(
        re.search(pattern, text)
        for pattern in REFUSAL_PATTERNS
    )


g3_abstention_results = []

print("Testing", len(g3_abstention_gold), "unanswerable questions")

for idx, item in enumerate(g3_abstention_gold, start=1):

    question_id = item["id"]
    question = item.get("question", item.get("q", ""))

    print(f"\n[{idx}/{len(g3_abstention_gold)}] {question_id}")
    print(question)

    try:
        research_output = research_node({
            "request": question,
            "clean_request": question
        })

        if not isinstance(research_output, dict):
            raise TypeError("Research output was not a dictionary.")

        evidence = (
            research_output.get("evidence", [])
            or research_output.get("regulatory_evidence", [])
            or research_output.get("results", [])
        )

        if not isinstance(evidence, list):
            raise TypeError("Research evidence was not a list.")

        draft_output = drafter_node({
            "request": question,
            "clean_request": question,
            "intent": "REG_QA",
            "evidence": evidence,
            "sql_rows": [],
            "similar_cases": [],
            "draft": "",
            "revisions": 0,
            "trace": []
        })

        if not isinstance(draft_output, dict):
            raise TypeError("Drafter output was not a dictionary.")

        answer = (
            draft_output.get("draft")
            or draft_output.get("answer")
            or ""
        )

        if not answer:
            raise ValueError("No answer returned by drafter.")

        refusal_detected = g3_detect_refusal(answer)

        g3_abstention_results.append({
            "question_id": question_id,
            "question": question,
            "answer": str(answer),
            "evidence_count": len(evidence),
            "refusal_detected": refusal_detected,
            "abstention_result": (
                "SCREENING_PASS"
                if refusal_detected
                else "MANUAL_REVIEW_REQUIRED"
            ),
            "execution_status": "SUCCESS",
            "error": None
        })

        print("Evidence count:", len(evidence))
        print("Refusal indicator:", refusal_detected)
        print("Answer:", str(answer)[:1000])

    except Exception as e:

        g3_abstention_results.append({
            "question_id": question_id,
            "question": question,
            "answer": "",
            "evidence_count": 0,
            "refusal_detected": False,
            "abstention_result": "EXECUTION_FAILED",
            "execution_status": "FAILED",
            "error": repr(e)
        })

        print("Execution failed:", repr(e))


# Results
g3_abstention_df = pd.DataFrame([
    {
        "question_id": x["question_id"],
        "evidence_count": x["evidence_count"],
        "refusal_detected": x["refusal_detected"],
        "abstention_result": x["abstention_result"],
        "execution_status": x["execution_status"]
    }
    for x in g3_abstention_results
])

print("\nG3 ABSTENTION SUMMARY")
print("=" * 70)
print(g3_abstention_df.to_string(index=False))

print("\nScreening passes:", sum(
    x["abstention_result"] == "SCREENING_PASS"
    for x in g3_abstention_results
))

print("Manual reviews required:", sum(
    x["abstention_result"] == "MANUAL_REVIEW_REQUIRED"
    for x in g3_abstention_results
))

print("Execution failures:", sum(
    x["execution_status"] == "FAILED"
    for x in g3_abstention_results
))

# Save results
G3_ABSTENTION_PATH = ROOT / "eval" / "results" / "G3_abstention_results.jsonl"

with open(G3_ABSTENTION_PATH, "w", encoding="utf-8") as f:
    for item in g3_abstention_results:
        f.write(json.dumps(item, ensure_ascii=False, default=str) + "\n")

G3_ABSTENTION_CSV = ROOT / "eval" / "results" / "G3_abstention_summary.csv"
g3_abstention_df.to_csv(G3_ABSTENTION_CSV, index=False)

print("\nSaved:", G3_ABSTENTION_PATH)
print("Saved:", G3_ABSTENTION_CSV)

Testing 3 unanswerable questions

[1/3] G3-18
What is Harbourline Bank's internal policy on cryptocurrency custody, and which internal approval committee authorizes it?


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Evidence count: 8
Refusal indicator: False
Answer: The evidence provided contains no information about "Harbourline Bank," its internal policies on cryptocurrency custody, or any internal approval committee related to such activities.

The available evidence consists exclusively of CFPB examination procedure manuals covering topics such as HMDA, fair lending, the Homeowners Protection Act, GLBA Privacy, and TILA. None of these documents reference cryptocurrency custody policies or any institution named Harbourline Bank.

**The evidence is insufficient to answer this request.**

[2/3] G3-19
What is the exact contractual refund processing time promised by Harbourline Bank to its customers?


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Evidence count: 8
Refusal indicator: False
Answer: The evidence provided does not contain any information about "Harbourline Bank" or any contractual refund processing time promised by such an institution. "Harbourline Bank" does not appear in any of the supplied evidence chunks.

The evidence does reference some regulatory refund timeframes applicable to servicers generally, such as:

- A **15-day** window for servicers to cancel force-placed insurance and refund charges upon receiving evidence of overlapping coverage [C:sh_issue22_2020-09_page_0016_chunk_0001]
- A **45-day** window for servicers to return unearned PMI premiums to borrowers after cancellation [C:exam_manual_v2_2012_page_0538_chunk_0001]
- A **30-day** limit after which a servicer may not require further PMI payments following cancellation or termination [C:exam_manual_v2_2012_page_0537_chunk_0001]

However, these are **regulatory requirements** applicable broadly to mortgage servicers under federal law — they are not 

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Evidence count: 8
Refusal indicator: False
Answer: The evidence provided contains no information about any company called "Harbourline," any named employee of such a company, or any specific customer refund approval from last week.

The available evidence consists solely of regulatory materials (CFPB Supervisory Highlights, Regulation Z, Regulation X, Regulation E, and examination procedures) and does not contain any operational records, employee names, or transaction-level data that would answer this question.

**The evidence is insufficient to answer this request.**

G3 ABSTENTION SUMMARY
question_id  evidence_count  refusal_detected      abstention_result execution_status
      G3-18               8             False MANUAL_REVIEW_REQUIRED          SUCCESS
      G3-19               8             False MANUAL_REVIEW_REQUIRED          SUCCESS
      G3-20               8             False MANUAL_REVIEW_REQUIRED          SUCCESS

Screening passes: 0
Manual reviews required: 3
Execution 

In [55]:
# CELL 54 — G3 Manual Answer Quality Review Template

import pandas as pd
import json

g3_gold_lookup = {
    x["id"]: x for x in g3_gold_reloaded
}

g3_critic_lookup = {
    x["question_id"]: x for x in g3_critic_results
}

g3_review_rows = []

for result in g3_answer_results:

    question_id = result["question_id"]
    gold = g3_gold_lookup.get(question_id, {})
    critic = g3_critic_lookup.get(question_id, {})

    evidence = result.get("evidence", [])

    evidence_text = []

    for ev in evidence:
        if isinstance(ev, dict):
            evidence_text.append(
                "CHUNK: " + str(ev.get("chunk_id", "")) +
                "\nCITATION: " + str(ev.get("citation", "")) +
                "\nTEXT: " + str(ev.get("text", ""))[:2500]
            )
        else:
            evidence_text.append(str(ev)[:2500])

    g3_review_rows.append({
        "question_id": question_id,
        "question": result["question"],
        "gold_sections": json.dumps(
            gold.get("expected_sections", []),
            ensure_ascii=False
        ),
        "gold_chunk_ids": json.dumps(
            gold.get("gold_chunk_ids", []),
            ensure_ascii=False
        ),
        "gold_notes": gold.get("gold_notes", ""),
        "generated_answer": result.get("answer", ""),
        "retrieved_evidence": "\n\n---\n\n".join(evidence_text),
        "critic_pass": critic.get("passed"),
        "critic_verdict": critic.get("verdict"),

        # Complete these manually
        "factual_correctness_1_to_5": "",
        "regulatory_accuracy_1_to_5": "",
        "evidence_grounding_1_to_5": "",
        "citation_quality_1_to_5": "",
        "completeness_1_to_5": "",
        "reviewer_comments": "",
        "manual_review_status": "PENDING"
    })

g3_manual_review_df = pd.DataFrame(g3_review_rows)

G3_MANUAL_REVIEW_PATH = (
    ROOT / "eval" / "results" / "G3_manual_answer_review.csv"
)

g3_manual_review_df.to_csv(
    G3_MANUAL_REVIEW_PATH,
    index=False
)

print("G3 MANUAL REVIEW TEMPLATE")
print("=" * 70)
print("Questions:", len(g3_manual_review_df))
print("Pending manual reviews:", len(g3_manual_review_df))
print("Output:", G3_MANUAL_REVIEW_PATH)

print("\nRating scale:")
print("1 = Incorrect / unsupported")
print("2 = Major deficiencies")
print("3 = Partially correct")
print("4 = Mostly correct")
print("5 = Accurate and well supported")

print("\nPASS — G3 manual review template generated.")

G3 MANUAL REVIEW TEMPLATE
Questions: 17
Pending manual reviews: 17
Output: /home/sagemaker-user/complaintiq/eval/results/G3_manual_answer_review.csv

Rating scale:
1 = Incorrect / unsupported
2 = Major deficiencies
3 = Partially correct
4 = Mostly correct
5 = Accurate and well supported

PASS — G3 manual review template generated.


In [56]:
# CELL 55 — G3 Consolidated Evaluation Summary

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# ---------------------------------------------------------
# 1. Retrieval metrics
# ---------------------------------------------------------

retrieval_metrics = {}

if "g3_retrieval_summary" in globals():
    retrieval_metrics = g3_retrieval_summary.copy()

# ---------------------------------------------------------
# 2. Generation metrics
# ---------------------------------------------------------

generation_total = len(g3_answer_results)

generation_success = sum(
    x["generation_status"] == "SUCCESS"
    for x in g3_answer_results
)

generation_failed = generation_total - generation_success

generation_rate = (
    generation_success / generation_total
    if generation_total else 0
)

# ---------------------------------------------------------
# 3. Citation metrics
# ---------------------------------------------------------

citation_metrics = {}

if len(g3_citation_df):

    citation_total = int(
        g3_citation_df["citation_count"].sum()
    )

    citation_valid = int(
        g3_citation_df["valid_citations"].sum()
    )

    citation_metrics = {
        "answers_evaluated": len(g3_citation_df),
        "answers_with_citations": int(
            g3_citation_df["has_citation"].sum()
        ),
        "total_citations": citation_total,
        "valid_citations": citation_valid,
        "invalid_citations": int(
            g3_citation_df["invalid_citations"].sum()
        ),
        "citation_validity": (
            citation_valid / citation_total
            if citation_total else None
        )
    }

# ---------------------------------------------------------
# 4. Critic metrics
# ---------------------------------------------------------

critic_success = sum(
    x["critic_status"] == "SUCCESS"
    for x in g3_critic_results
)

critic_failed = sum(
    x["critic_status"] == "FAILED"
    for x in g3_critic_results
)

critic_explicit = [
    x for x in g3_critic_results
    if isinstance(x.get("passed"), bool)
]

critic_metrics = {
    "executions_successful": critic_success,
    "executions_failed": critic_failed,
    "explicit_decisions": len(critic_explicit),
    "explicit_passes": sum(
        x["passed"] is True for x in critic_explicit
    ),
    "explicit_failures": sum(
        x["passed"] is False for x in critic_explicit
    ),
    "pass_rate_among_explicit_decisions": (
        sum(x["passed"] is True for x in critic_explicit)
        / len(critic_explicit)
        if critic_explicit else None
    )
}

# ---------------------------------------------------------
# 5. Abstention metrics
# ---------------------------------------------------------

abstention_success = sum(
    x["execution_status"] == "SUCCESS"
    for x in g3_abstention_results
)

abstention_failed = sum(
    x["execution_status"] == "FAILED"
    for x in g3_abstention_results
)

abstention_screening_pass = sum(
    x["abstention_result"] == "SCREENING_PASS"
    for x in g3_abstention_results
)

abstention_metrics = {
    "unanswerable_questions": len(g3_abstention_results),
    "executions_successful": abstention_success,
    "executions_failed": abstention_failed,
    "refusal_screening_passes": abstention_screening_pass,
    "manual_reviews_required": sum(
        x["abstention_result"] == "MANUAL_REVIEW_REQUIRED"
        for x in g3_abstention_results
    )
}

# ---------------------------------------------------------
# 6. Manual review status
# ---------------------------------------------------------

manual_review_metrics = {
    "answers_requiring_review": len(g3_manual_review_df),
    "reviewed_answers": int(
        (
            g3_manual_review_df["manual_review_status"]
            == "REVIEWED"
        ).sum()
    ),
    "pending_answers": int(
        (
            g3_manual_review_df["manual_review_status"]
            != "REVIEWED"
        ).sum()
    )
}

# ---------------------------------------------------------
# 7. Overall report
# ---------------------------------------------------------

g3_final_report = {
    "evaluation": "ComplaintIQ Phase 7 - G3 Regulatory Q&A",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "gold_dataset": {
        "total_questions": len(g3_gold_reloaded),
        "answerable_questions": len(g3_answerable),
        "unanswerable_questions": len(g3_abstention_gold),
        "g3_17_status": g3_supervisory_item["status"]
    },
    "retrieval": retrieval_metrics,
    "generation": {
        "total": generation_total,
        "successful": generation_success,
        "failed": generation_failed,
        "success_rate": generation_rate
    },
    "citations": citation_metrics,
    "critic": critic_metrics,
    "abstention": abstention_metrics,
    "manual_review": manual_review_metrics,
    "overall_status": (
        "PENDING_MANUAL_REVIEW"
        if manual_review_metrics["pending_answers"] > 0
        else "MANUAL_REVIEW_COMPLETED"
    )
}

print("COMPLAINTIQ — G3 EVALUATION REPORT")
print("=" * 75)
print(json.dumps(g3_final_report, indent=2, ensure_ascii=False))

# Save JSON report
G3_FINAL_REPORT_PATH = (
    ROOT / "eval" / "results" / "G3_final_evaluation_report.json"
)

G3_FINAL_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(G3_FINAL_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        g3_final_report,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

# Save concise CSV summary
summary_rows = []

for category, metrics in g3_final_report.items():

    if isinstance(metrics, dict):
        for metric, value in metrics.items():
            summary_rows.append({
                "category": category,
                "metric": metric,
                "value": value
            })

G3_FINAL_CSV_PATH = (
    ROOT / "eval" / "results" / "G3_final_evaluation_summary.csv"
)

pd.DataFrame(summary_rows).to_csv(
    G3_FINAL_CSV_PATH,
    index=False
)

print("\nSaved JSON report:", G3_FINAL_REPORT_PATH)
print("Saved CSV summary:", G3_FINAL_CSV_PATH)

print("\nPASS — G3 automated evaluation summary generated.")
print("OVERALL STATUS:", g3_final_report["overall_status"])

COMPLAINTIQ — G3 EVALUATION REPORT
{
  "evaluation": "ComplaintIQ Phase 7 - G3 Regulatory Q&A",
  "generated_at_utc": "2026-11-06T13:42:13.130525+00:00",
  "gold_dataset": {
    "total_questions": 20,
    "answerable_questions": 17,
    "unanswerable_questions": 3,
    "g3_17_status": "MANUALLY_VERIFIED"
  },
  "retrieval": {
    "answerable_questions": 17,
    "mean_recall_at_5": 0.25644972668600646,
    "mean_recall_at_10": 0.3929931395040015,
    "hit_rate_at_5": 0.9411764705882353,
    "hit_rate_at_10": 0.9411764705882353,
    "mean_reciprocal_rank": 0.7990196078431372
  },
  "generation": {
    "total": 17,
    "successful": 17,
    "failed": 0,
    "success_rate": 1.0
  },
  "citations": {
    "answers_evaluated": 17,
    "answers_with_citations": 17,
    "total_citations": 167,
    "valid_citations": 164,
    "invalid_citations": 3,
    "citation_validity": 0.9820359281437125
  },
  "critic": {
    "executions_successful": 13,
    "executions_failed": 4,
    "explicit_decisions"

In [57]:
# CELL 56 — G3 Artifact Validation

from pathlib import Path
import json

G3_REQUIRED_FILES = [
    ROOT / "eval/gold/G3_regulatory_gold.jsonl",
    ROOT / "eval/results/G3_retrieval_results.csv",
    ROOT / "eval/results/G3_retrieval_summary.json",
    ROOT / "eval/results/G3_generated_answers.jsonl",
    ROOT / "eval/results/G3_citation_results.csv",
    ROOT / "eval/results/G3_critic_results.jsonl",
    ROOT / "eval/results/G3_critic_summary.csv",
    ROOT / "eval/results/G3_abstention_results.jsonl",
    ROOT / "eval/results/G3_abstention_summary.csv",
    ROOT / "eval/results/G3_manual_answer_review.csv",
    ROOT / "eval/results/G3_final_evaluation_report.json",
    ROOT / "eval/results/G3_final_evaluation_summary.csv"
]

print("G3 ARTIFACT VALIDATION")
print("=" * 75)

missing_files = []

for path in G3_REQUIRED_FILES:

    if path.exists() and path.stat().st_size > 0:
        print("PASS |", path.relative_to(ROOT),
              "|", path.stat().st_size, "bytes")
    else:
        print("FAIL |", path.relative_to(ROOT))
        missing_files.append(str(path))

assert not missing_files, (
    "Missing or empty G3 artifacts: " + str(missing_files)
)

# Validate report JSON
with open(
    ROOT / "eval/results/G3_final_evaluation_report.json",
    encoding="utf-8"
) as f:
    report = json.load(f)

assert report["gold_dataset"]["total_questions"] == 20
assert report["gold_dataset"]["answerable_questions"] == 17
assert report["gold_dataset"]["unanswerable_questions"] == 3

print("\nPASS — All required G3 artifacts exist.")
print("PASS — Gold dataset counts validated.")
print("G3 STATUS:", report["overall_status"])

G3 ARTIFACT VALIDATION
PASS | eval/gold/G3_regulatory_gold.jsonl | 22514 bytes
PASS | eval/results/G3_retrieval_results.csv | 2833 bytes
PASS | eval/results/G3_retrieval_summary.json | 243 bytes
PASS | eval/results/G3_generated_answers.jsonl | 811980 bytes
PASS | eval/results/G3_citation_results.csv | 807 bytes
PASS | eval/results/G3_critic_results.jsonl | 48754 bytes
PASS | eval/results/G3_critic_summary.csv | 603 bytes
PASS | eval/results/G3_abstention_results.jsonl | 3192 bytes
PASS | eval/results/G3_abstention_summary.csv | 214 bytes
PASS | eval/results/G3_manual_answer_review.csv | 387388 bytes
PASS | eval/results/G3_final_evaluation_report.json | 1437 bytes
PASS | eval/results/G3_final_evaluation_summary.csv | 1223 bytes

PASS — All required G3 artifacts exist.
PASS — Gold dataset counts validated.
G3 STATUS: PENDING_MANUAL_REVIEW


In [58]:

from pathlib import Path
import json
import pandas as pd
from collections import Counter

print("=" * 90)
print("COMPLAINTIQ PHASE 7 — G1 TO G3 CONSOLIDATED ACCEPTANCE VALIDATION")
print("=" * 90)

EVAL = ROOT / "eval"
GOLD = EVAL / "gold"
RESULTS = EVAL / "results"

checks = []
warnings = []
failures = []

def record_check(name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    checks.append({
        "check": name,
        "status": status,
        "detail": str(detail)
    })
    print(f"{status:6} | {name} | {detail}")
    if not passed:
        failures.append(name)

def record_warning(name, detail):
    warnings.append({"check": name, "detail": str(detail)})
    print(f"REVIEW | {name} | {detail}")

def read_jsonl(path):
    if not path.exists():
        return None, f"Missing file: {path}"
    try:
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                if line.strip():
                    try:
                        rows.append(json.loads(line))
                    except Exception as exc:
                        return None, f"Invalid JSON at line {line_no}: {exc}"
        return rows, None
    except Exception as exc:
        return None, str(exc)

def load_csv(path):
    if not path.exists():
        return None, f"Missing file: {path}"
    try:
        return pd.read_csv(path), None
    except Exception as exc:
        return None, str(exc)

def check_jsonl(name, filename, expected_count=None):
    rows, error = read_jsonl(filename)
    if error:
        record_check(name, False, error)
        return None

    count_ok = expected_count is None or len(rows) == expected_count
    record_check(
        name,
        count_ok,
        f"{len(rows)} records" +
        (f"; expected {expected_count}" if expected_count is not None else "")
    )
    return rows

def check_csv(name, filename, minimum_rows=None):
    df, error = load_csv(filename)
    if error:
        record_check(name, False, error)
        return None

    count_ok = minimum_rows is None or len(df) >= minimum_rows
    record_check(
        name,
        count_ok,
        f"{len(df)} rows; {len(df.columns)} columns"
    )
    return df

# -------------------------------------------------------------------
# G1 — TRIAGE
# -------------------------------------------------------------------

print("\n" + "=" * 90)
print("G1 — TRIAGE EVALUATION")
print("=" * 90)

g1_gold = check_jsonl(
    "G1 gold dataset",
    GOLD / "G1_triage_gold.jsonl",
    expected_count=60
)

g1_annotations = check_csv(
    "G1 manual annotation template",
    GOLD / "G1_triage_manual_annotations.csv",
    minimum_rows=60
)

if g1_gold is not None:
    g1_ids = [str(x.get("complaint_id", "")) for x in g1_gold]

    record_check(
        "G1 unique complaint IDs",
        len(g1_ids) == len(set(g1_ids)) and all(g1_ids),
        f"{len(set(g1_ids))} unique IDs"
    )

    record_check(
        "G1 product and issue labels",
        all(x.get("product") and x.get("issue") for x in g1_gold),
        "Every record has product and issue"
    )

    record_check(
        "G1 complaint narrative",
        all(
            x.get("narrative") or x.get("consumer_complaint_narrative")
            for x in g1_gold
        ),
        "Narrative availability"
    )

if g1_annotations is not None:
    required_g1_cols = [
        "severity",
        "vulnerability",
        "regulation_hints"
    ]

    missing_cols = [
        c for c in required_g1_cols
        if c not in g1_annotations.columns
    ]

    record_check(
        "G1 manual annotation columns",
        not missing_cols,
        f"Missing columns: {missing_cols}" if missing_cols else "All present"
    )

    annotation_columns = [
        c for c in required_g1_cols
        if c in g1_annotations.columns
    ]

    if annotation_columns:
        completed = (
            g1_annotations[annotation_columns]
            .notna()
            .any(axis=1)
            .sum()
        )
        record_warning(
            "G1 severity/vulnerability annotation",
            f"{completed}/{len(g1_annotations)} rows have at least one manual annotation. "
            "Complete manual labels before reporting severity/vulnerability metrics."
        )

# -------------------------------------------------------------------
# G2 — ANALYTICS
# -------------------------------------------------------------------

print("\n" + "=" * 90)
print("G2 — ANALYTICS EVALUATION")
print("=" * 90)

g2_gold = check_jsonl(
    "G2 analytics gold questions",
    GOLD / "G2_analytics_gold.jsonl",
    expected_count=20
)

g2_expected = check_jsonl(
    "G2 expected reference results",
    GOLD / "G2_analytics_expected.jsonl",
    expected_count=20
)

g2_summary = check_csv(
    "G2 reference result summary",
    RESULTS / "G2_reference_results_summary.csv",
    minimum_rows=1
)

if g2_gold is not None:
    g2_ids = [str(x.get("question_id", "")) for x in g2_gold]

    record_check(
        "G2 unique question IDs",
        len(g2_ids) == len(set(g2_ids)) and all(g2_ids),
        f"{len(set(g2_ids))} unique IDs"
    )

    # Identify the answerability field used by the gold dataset.
    answerability_counts = Counter(
        str(x.get("answerable", x.get("is_answerable", "UNKNOWN")))
        for x in g2_gold
    )

    print("G2 answerability distribution:", dict(answerability_counts))

if g2_expected is not None:
    expected_ids = [str(x.get("question_id", "")) for x in g2_expected]

    record_check(
        "G2 expected result IDs",
        len(expected_ids) == len(set(expected_ids)) and all(expected_ids),
        f"{len(set(expected_ids))} unique IDs"
    )

    record_check(
        "G2 expected result payloads",
        all(
            any(k in x for k in ["expected_rows", "rows", "result"])
            for x in g2_expected
        ),
        "Expected result payload field present"
    )

# -------------------------------------------------------------------
# G3 — REGULATORY Q&A
# -------------------------------------------------------------------

print("\n" + "=" * 90)
print("G3 — REGULATORY Q&A EVALUATION")
print("=" * 90)

g3_gold = check_jsonl(
    "G3 regulatory gold dataset",
    GOLD / "G3_regulatory_gold.jsonl",
    expected_count=20
)

g3_generated = check_jsonl(
    "G3 generated answers",
    RESULTS / "G3_generated_answers.jsonl",
    expected_count=17
)

g3_critic = check_jsonl(
    "G3 critic results",
    RESULTS / "G3_critic_results.jsonl",
    expected_count=17
)

g3_abstention = check_jsonl(
    "G3 abstention results",
    RESULTS / "G3_abstention_results.jsonl",
    expected_count=3
)

g3_retrieval = check_csv(
    "G3 retrieval results",
    RESULTS / "G3_retrieval_results.csv",
    minimum_rows=17
)

g3_citations = check_csv(
    "G3 citation results",
    RESULTS / "G3_citation_results.csv",
    minimum_rows=1
)

g3_manual = check_csv(
    "G3 manual answer review",
    RESULTS / "G3_manual_answer_review.csv",
    minimum_rows=17
)

g3_retrieval_summary = None
retrieval_summary_path = RESULTS / "G3_retrieval_summary.json"

if retrieval_summary_path.exists():
    try:
        with retrieval_summary_path.open("r", encoding="utf-8") as f:
            g3_retrieval_summary = json.load(f)
        record_check(
            "G3 retrieval summary JSON",
            isinstance(g3_retrieval_summary, dict),
            "Valid JSON object"
        )
    except Exception as exc:
        record_check("G3 retrieval summary JSON", False, str(exc))
else:
    record_check(
        "G3 retrieval summary JSON",
        False,
        "Missing G3_retrieval_summary.json"
    )

# Gold dataset validation
if g3_gold is not None:
    g3_ids = [str(x.get("question_id", "")) for x in g3_gold]

    record_check(
        "G3 unique question IDs",
        len(g3_ids) == len(set(g3_ids)) and all(g3_ids),
        f"{len(set(g3_ids))} unique IDs"
    )

    abstention_items = [
        x for x in g3_gold
        if x.get("is_abstention") is True
    ]

    answerable_items = [
        x for x in g3_gold
        if x.get("is_abstention") is False
    ]

    record_check(
        "G3 answerable/abstention split",
        len(answerable_items) == 17 and len(abstention_items) == 3,
        f"Answerable={len(answerable_items)}, Abstention={len(abstention_items)}"
    )

    record_check(
        "G3 abstention gold reasons",
        all(x.get("abstention_reason") for x in abstention_items),
        "All abstention cases have reasons"
    )

    unresolved = [
        x.get("question_id")
        for x in g3_gold
        if x.get("status") not in ["SECTION_RESOLVED", "MANUALLY_VERIFIED", "ABSTENTION_GOLD"]
    ]

    record_check(
        "G3 gold mapping statuses",
        len(unresolved) == 0,
        f"Unresolved: {unresolved}"
    )

    g3_chunk_ids = {
        str(c.get("chunk_id"))
        for c in REG_INDEX.chunks
        if c.get("chunk_id")
    }

    missing_gold_chunks = [
        (x.get("question_id"), cid)
        for x in g3_gold
        for cid in x.get("gold_chunk_ids", [])
        if str(cid) not in g3_chunk_ids
    ]

    record_check(
        "G3 gold chunk references",
        len(missing_gold_chunks) == 0,
        f"Missing chunk references: {len(missing_gold_chunks)}"
    )

# Generated answers validation
if g3_generated is not None:
    successful_answers = [
        x for x in g3_generated
        if x.get("generation_status") == "SUCCESS"
        and x.get("answer")
    ]

    failed_answers = [
        x for x in g3_generated
        if x.get("generation_status") != "SUCCESS"
    ]

    record_check(
        "G3 answer generation coverage",
        len(successful_answers) == 17,
        f"Successful={len(successful_answers)}, Failed={len(failed_answers)}"
    )

    if failed_answers:
        record_warning(
            "G3 generation errors",
            [
                {
                    "question_id": x.get("question_id"),
                    "error": x.get("error")
                }
                for x in failed_answers
            ]
        )

# Critic validation
if g3_critic is not None:
    critic_passed = [
        x for x in g3_critic
        if x.get("passed") is True
    ]

    critic_failed = [
        x for x in g3_critic
        if x.get("passed") is False
    ]

    critic_unknown = [
        x for x in g3_critic
        if x.get("passed") is None
    ]

    record_check(
        "G3 critic result coverage",
        len(g3_critic) == 17,
        f"Critic records={len(g3_critic)}"
    )

    record_warning(
        "G3 critic outcomes",
        f"Passed={len(critic_passed)}, Failed={len(critic_failed)}, "
        f"Unknown={len(critic_unknown)}. Review critic verdicts and exceptions."
    )

# Abstention validation
if g3_abstention is not None:
    abstention_statuses = Counter(
        str(x.get("status", "UNKNOWN"))
        for x in g3_abstention
    )

    record_check(
        "G3 abstention test coverage",
        len(g3_abstention) == 3,
        f"Three abstention test records; statuses={dict(abstention_statuses)}"
    )

    record_warning(
        "G3 abstention manual review",
        "Node-level abstention screening is not equivalent to full graph-level "
        "end-to-end abstention validation. Review all three answers manually."
    )

# Retrieval and citation metrics
if g3_retrieval is not None:
    record_warning(
        "G3 retrieval metrics",
        f"Inspect retrieval columns and actual Recall@5/Recall@10/MRR values. "
        f"Rows={len(g3_retrieval)}"
    )

if g3_citations is not None:
    print("\nG3 citation result columns:", list(g3_citations.columns))

    invalid_columns = [
        c for c in g3_citations.columns
        if "invalid" in c.lower()
    ]

    if invalid_columns:
        for col in invalid_columns:
            print(
                f"Citation metric — {col}: "
                f"{g3_citations[col].sum()}"
            )

# Manual review validation
if g3_manual is not None:
    if "manual_review_status" in g3_manual.columns:
        pending_count = (
            g3_manual["manual_review_status"]
            .astype(str)
            .str.upper()
            .eq("PENDING")
            .sum()
        )

        record_check(
            "G3 manual review file coverage",
            len(g3_manual) == 17,
            f"{len(g3_manual)} review rows"
        )

        if pending_count:
            record_warning(
                "G3 answer-quality manual review",
                f"{pending_count} of {len(g3_manual)} answers remain pending. "
                "Do not declare final G3 acceptance until reviewed."
            )
        else:
            record_check(
                "G3 answer-quality manual review",
                True,
                "No PENDING review rows"
            )
    else:
        record_check(
            "G3 manual review status column",
            False,
            "manual_review_status column missing"
        )

# -------------------------------------------------------------------
# FINAL ACCEPTANCE SUMMARY
# -------------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL G1–G3 ACCEPTANCE SUMMARY")
print("=" * 90)

checks_df = pd.DataFrame(checks)
warnings_df = pd.DataFrame(warnings)

print("\nCheck results:")
if not checks_df.empty:
    print(checks_df.to_string(index=False))

print("\nWarnings / manual review:")
if not warnings_df.empty:
    print(warnings_df.to_string(index=False))
else:
    print("No warnings recorded.")

pass_count = sum(x["status"] == "PASS" for x in checks)
fail_count = sum(x["status"] == "FAIL" for x in checks)

print("\nPASS:", pass_count)
print("FAIL:", fail_count)
print("WARNINGS:", len(warnings))

if fail_count == 0 and len(warnings) == 0:
    overall_status = "PASS — G1–G3 VALIDATION"
elif fail_count == 0:
    overall_status = "STRUCTURAL PASS — MANUAL REVIEW / WARNINGS PENDING"
else:
    overall_status = "FAIL — CORRECTIONS REQUIRED"

print("\n" + "=" * 90)
print(overall_status)
print("=" * 90)

# Save local validation report only
validation_report = {
    "phase": "Phase 7",
    "scope": ["G1", "G2", "G3"],
    "status": overall_status,
    "pass_count": pass_count,
    "fail_count": fail_count,
    "warning_count": len(warnings),
    "checks": checks,
    "warnings": warnings
}

validation_path = RESULTS / "G1_G3_consolidated_validation.json"
RESULTS.mkdir(parents=True, exist_ok=True)

with validation_path.open("w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, default=str)

print("\nValidation report saved locally:")
print(validation_path)


COMPLAINTIQ PHASE 7 — G1 TO G3 CONSOLIDATED ACCEPTANCE VALIDATION

G1 — TRIAGE EVALUATION
PASS   | G1 gold dataset | 60 records; expected 60
PASS   | G1 manual annotation template | 60 rows; 9 columns
FAIL   | G1 unique complaint IDs | 1 unique IDs
FAIL   | G1 product and issue labels | Every record has product and issue
FAIL   | G1 complaint narrative | Narrative availability
FAIL   | G1 manual annotation columns | Missing columns: ['vulnerability']
REVIEW | G1 severity/vulnerability annotation | 0/60 rows have at least one manual annotation. Complete manual labels before reporting severity/vulnerability metrics.

G2 — ANALYTICS EVALUATION
PASS   | G2 analytics gold questions | 20 records; expected 20
PASS   | G2 expected reference results | 20 records; expected 20
PASS   | G2 reference result summary | 20 rows; 4 columns
PASS   | G2 unique question IDs | 20 unique IDs
G2 answerability distribution: {'True': 18, 'False': 2}
PASS   | G2 expected result IDs | 20 unique IDs
PASS   | G2 e

In [59]:

import json
from pathlib import Path

g1_path = ROOT / "eval/gold/G1_triage_gold.jsonl"

with g1_path.open("r", encoding="utf-8") as f:
    g1_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("TOTAL RECORDS:", len(g1_records))

print("\nACTUAL FIELD NAMES:")
print(sorted(set(
    key
    for record in g1_records
    for key in record.keys()
)))

print("\nFIRST THREE RECORDS:")
for i, record in enumerate(g1_records[:3], start=1):
    print(f"\nRECORD {i}")
    print(json.dumps(record, indent=2, default=str)[:4000])

print("\nUNIQUE COMPLAINT IDS:")
for key in ["complaint_id", "Complaint ID", "complaintId"]:
    values = [
        str(x.get(key))
        for x in g1_records
        if x.get(key) is not None
    ]
    if values:
        print(key, "count:", len(values),
              "unique:", len(set(values)))


TOTAL RECORDS: 60

ACTUAL FIELD NAMES:
['annotation_status', 'case_id', 'expected', 'manual_annotation', 'request', 'source']

FIRST THREE RECORDS:

RECORD 1
{
  "case_id": "23266147",
  "request": "I am filing this complaint against BMO Bank regarding an unreasonable hold placed on my deposited insurance settlement funds and the overdraft fees that resulted from the banks actions. On XX/XX/year>, I deposited a settlement check from XXXX XXXX at the BMO branch located at XXXX XXXX XXXX, XXXX XXXX, Missouri. BMO placed a hold on the funds but failed to provide adequate notice explaining the reason for the hold, the length of the hold, or my rights under the banks funds availability policy. After the deposit, documentation was provided to verify the legitimacy of the settlement funds. The branch also received verification directly from XXXX XXXX confirming that the check was valid and legitimate. Despite receiving this verification, BMO continued to restrict access to my funds. Because B

In [60]:

import json

critic_path = ROOT / "eval/results/G3_critic_results.jsonl"

with critic_path.open("r", encoding="utf-8") as f:
    critic_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

for item in critic_records:
    if item.get("passed") is not True:
        print("\n" + "=" * 80)
        print("QUESTION:", item.get("question_id"))
        print("PASSED:", item.get("passed"))
        print("VERDICT:", item.get("verdict"))
        print("CRITIQUE:")
        print(json.dumps(
            item.get("critique"),
            indent=2,
            default=str
        )[:5000])
        print("ERROR:", item.get("error"))



QUESTION: G3-01
PASSED: None
VERDICT: None
CRITIQUE:
null
ERROR: ValueError('No JSON object found')

QUESTION: G3-03
PASSED: False
VERDICT: REVISE
CRITIQUE:
{
  "pass": false,
  "problems": [],
  "verdict": "REVISE",
  "unsupported_claims": [
    "The claim that 'Disclosures provided earlier than required (e.g., when a consumer opens a checking account) need not be repeated when the consumer later authorizes an electronic check conversion or agrees to preauthorized transfers' cited to exam_manual_v2_2012_page_0685_chunk_0001 is not supported. That chunk addresses timing and content of initial disclosures and does not state earlier disclosures need not be repeated. The cited evidence (page_0686) actually indicates new disclosures ARE required for a new EFT service with different terms, contradicting this statement.",
    "Item 6 'Documentation ... notices regarding preauthorized transfers (per \u00a7 1005.10(a) and (d))' is cited to 12CFR1005_1005.7_chunk_0001, but the supplied text of

In [61]:

citation_path = ROOT / "eval/results/G3_citation_results.csv"

citation_df = pd.read_csv(citation_path)

failed_citations = citation_df[
    citation_df["invalid_citations"] > 0
]

print("INVALID CITATION RECORDS:", len(failed_citations))

for _, row in failed_citations.iterrows():
    print("\nQUESTION:", row["question_id"])
    print("Citation count:", row["citation_count"])
    print("Valid:", row["valid_citations"])
    print("Invalid:", row["invalid_citations"])
    print("Invalid IDs:", row["invalid_chunk_ids"])


INVALID CITATION RECORDS: 1

QUESTION: G3-17
Citation count: 7
Valid: 4
Invalid: 3
Invalid IDs: ['sh_issue18_2019-03_page_0003_chunk_0001; C:sh_issue22_2020-09_page_0003_chunk_0001', 'sh_issue18_2019-03_page_0021_chunk_0001; C:sh_issue23_2021-01_page_0028_chunk_0001', 'exam_manual_v2_2012_page_0911_chunk_0001; C:exam_manual_v2_2012_page_0923_chunk_0001']


In [62]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

G1_JSONL = ROOT / "eval/gold/G1_triage_gold.jsonl"
G1_CSV = ROOT / "eval/gold/G1_triage_manual_annotations.csv"

assert G1_JSONL.exists(), f"Missing: {G1_JSONL}"
assert G1_CSV.exists(), f"Missing: {G1_CSV}"

# Load JSONL
g1_records = []
with G1_JSONL.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        if line.strip():
            try:
                g1_records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_no}: {e}")

# Validate actual nested schema
case_ids = [r.get("case_id") for r in g1_records]
products = [r.get("expected", {}).get("product") for r in g1_records]
issues = [r.get("expected", {}).get("issue") for r in g1_records]
requests = [r.get("request") for r in g1_records]

required_annotation_fields = [
    "severity",
    "vulnerable_customer",
    "vulnerability_signals",
    "regulation_hints",
    "summary",
]

missing_fields = []
for i, r in enumerate(g1_records, 1):
    if not isinstance(r.get("expected"), dict):
        missing_fields.append(f"Record {i}: expected is not an object")
    if not isinstance(r.get("manual_annotation"), dict):
        missing_fields.append(f"Record {i}: manual_annotation is not an object")
    else:
        for field in required_annotation_fields:
            if field not in r["manual_annotation"]:
                missing_fields.append(f"Record {i}: missing manual_annotation.{field}")

# CSV inspection
g1_df = pd.read_csv(G1_CSV)
print("G1 JSONL records:", len(g1_records))
print("Unique case IDs:", len(set(case_ids)))
print("Missing case IDs:", sum(not x for x in case_ids))
print("Missing expected products:", sum(not x for x in products))
print("Missing expected issues:", sum(not x for x in issues))
print("Missing request narratives:", sum(not isinstance(x, str) or not x.strip() for x in requests))
print("Duplicate case IDs:", len(case_ids) - len(set(case_ids)))
print("Schema errors:", len(missing_fields))
print("CSV rows:", len(g1_df))
print("CSV columns:", list(g1_df.columns))
print("\nCSV non-empty counts:")
print(g1_df.notna().sum().to_string())

# Show annotation progress using whichever annotation columns exist
for field in required_annotation_fields:
    if field in g1_df.columns:
        print(f"{field}: {g1_df[field].notna().sum()} / {len(g1_df)} completed")
    else:
        print(f"{field}: CSV column not found — inspect the column mapping")

if missing_fields:
    print("\nSCHEMA ERRORS:")
    print("\n".join(missing_fields[:30]))

assert len(g1_records) == 60, f"Expected 60 records, found {len(g1_records)}"
assert len(set(case_ids)) == len(case_ids), "Duplicate case IDs found"
assert all(case_ids), "Missing case IDs found"
assert all(products), "Missing expected products found"
assert all(issues), "Missing expected issues found"
assert all(isinstance(x, str) and x.strip() for x in requests), "Missing narratives found"
assert not missing_fields, "Nested schema validation failed"

print("\nPASS — G1 JSONL STRUCTURE VALIDATED")

G1 JSONL records: 60
Unique case IDs: 60
Missing case IDs: 0
Missing expected products: 0
Missing expected issues: 0
Missing request narratives: 0
Duplicate case IDs: 0
Schema errors: 0
CSV rows: 60
CSV columns: ['case_id', 'severity', 'vulnerable_customer', 'vulnerability_signals', 'regulation_hints', 'summary', 'reviewer', 'review_notes', 'annotation_status']

CSV non-empty counts:
case_id                  60
severity                  0
vulnerable_customer       0
vulnerability_signals     0
regulation_hints          0
summary                   0
reviewer                  0
review_notes              0
annotation_status        60
severity: 0 / 60 completed
vulnerable_customer: 0 / 60 completed
vulnerability_signals: 0 / 60 completed
regulation_hints: 0 / 60 completed
summary: 0 / 60 completed

PASS — G1 JSONL STRUCTURE VALIDATED


In [64]:
import ast
import json
from pathlib import Path

NOTEBOOK_PATH = ROOT / "notebooks/06_eval.ipynb"

with NOTEBOOK_PATH.open("r", encoding="utf-8") as f:
    nb_data = json.load(f)

terms = [
    "extract_citation",
    "citation",
    "C:",
    "invalid_chunk_ids",
    "valid_citations",
    "Unknown citation ID",
]

print("=== NOTEBOOK CELLS CONTAINING CITATION LOGIC ===")

for cell_no, cell in enumerate(nb_data.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(term.lower() in source.lower() for term in terms):
        print(f"\n{'='*25} CELL {cell_no} {'='*25}")
        print(source[:14000])

=== NOTEBOOK CELLS CONTAINING CITATION LOGIC ===

========================= CELL 14 =========================
class HybridIndex:

    def __init__(self, chunks, vectors):
        self.chunks = list(chunks)
        self.vectors = np.asarray(vectors, dtype="float32")

        if len(self.vectors) != len(self.chunks):
            raise ValueError("Vector/metadata count mismatch")

        if not np.isfinite(self.vectors).all():
            raise ValueError("Invalid vectors")

        self.faiss = faiss.IndexFlatIP(
            self.vectors.shape[1]
        )

        self.faiss.add(self.vectors)

        self.bm25 = BM25Okapi([
            tok(c.get("text", ""))
            for c in self.chunks
        ])

    def search(self, query, k=8, pool=40, rrf_k=60):

        if not query or not str(query).strip():
            raise ValueError("Query cannot be blank")

        pool = min(pool, len(self.chunks))

        qv = embed([query], workers=1)

        _, dense_ids = self.faiss.search(qv, p

In [65]:
import json
from pathlib import Path

def read_jsonl_records(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

G3_GOLD = ROOT / "eval/gold/G3_regulatory_gold.jsonl"
G3_ANSWERS = ROOT / "eval/results/G3_generated_answers.jsonl"
G3_CRITIC = ROOT / "eval/results/G3_critic_results.jsonl"

gold_records = read_jsonl_records(G3_GOLD)
answer_records = read_jsonl_records(G3_ANSWERS)
critic_records = read_jsonl_records(G3_CRITIC)

for label, records in [
    ("GOLD", gold_records),
    ("GENERATED ANSWER", answer_records),
    ("CRITIC", critic_records),
]:
    print(f"\n{'='*20} {label}: G3-17 {'='*20}")
    matches = [
        r for r in records
        if str(r.get("question_id", r.get("id", ""))) == "G3-17"
    ]
    if not matches:
        print("No G3-17 record found using question_id/id.")
    else:
        for r in matches:
            print(json.dumps(r, indent=2, ensure_ascii=False)[:16000])


==================== GOLD: G3-17 ====================
{
  "question_id": "G3-17",
  "question": "What consumer-protection or compliance findings concerning deposit accounts are described in the CFPB Supervisory Highlights documents included in the corpus?",
  "regulation": "CFPB Supervisory Highlights",
  "answerable": true,
  "expected_sections": [],
  "expected_source": "supervisory_highlights",
  "gold_chunk_ids": [
    "sh_issue22_2020-09_page_0011_chunk_0001"
  ],
  "abstention_reason": null,
  "gold_status": "PENDING_MANUAL_REVIEW",
  "id": "G3-17",
  "is_abstention": false,
  "status": "MANUALLY_VERIFIED",
  "gold_notes": "CFPB Supervisory Highlights Issue 22, page 10 (PDF page 11): financial institutions failed to provide advertised deposit account bonuses to eligible consumers. The CFPB identified misleading and inaccurate advertising under Regulation DD, 12 CFR 1030.8(a)(1)."
}

==================== GENERATED ANSWER: G3-17 ====================
{
  "question_id": "G3-17",
  "

In [66]:
import json
from pathlib import Path

NOTEBOOK_PATH = ROOT / "notebooks/06_eval.ipynb"

with NOTEBOOK_PATH.open("r", encoding="utf-8") as f:
    nb_data = json.load(f)

terms = [
    "def critic",
    "critic_node",
    "chat_json",
    "No JSON object found",
    "G3_critic_results",
    "verdict",
    "unsupported_claims",
]

for cell_no, cell in enumerate(nb_data.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(term.lower() in source.lower() for term in terms):
        print(f"\n{'='*25} CELL {cell_no} {'='*25}")
        print(source[:18000])


========================= CELL 6 =========================
# ============================================================
# COMPLAINTIQ PHASE 7 — NOTEBOOK 6 BOOTSTRAP
# Restore validated implementations from Notebooks 3, 4 and 5
# No corpus embedding generation, benchmark calls or publishing
# ============================================================

import sys
import os
import json
import ast
import uuid
import re
import hashlib
import sqlite3
import importlib
import inspect

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import boto3
import faiss
import duckdb
import sqlglot

from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from pydantic import BaseModel, Field
from typing import Annotated, Literal, TypedDict
import operator

from botocore.exceptions import ClientError
from langgraph.types import interrupt


# -------------------------------------

In [67]:
import json
from pathlib import Path
from collections import Counter

critic_path = ROOT / "eval/results/G3_critic_results.jsonl"

critic_records = []
with critic_path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            critic_records.append(json.loads(line))

print("Total critic records:", len(critic_records))
print("\nCritic record keys:")
if critic_records:
    print(list(critic_records[0].keys()))

print("\nOutcome counts:")
print(Counter(str(r.get("verdict")) for r in critic_records))

print("\nRecords with parsing errors or unknown verdicts:")
for r in critic_records:
    verdict = r.get("verdict")
    error = r.get("error")
    if verdict is None or error or r.get("passed") is None:
        print(json.dumps(r, indent=2, ensure_ascii=False)[:10000])

Total critic records: 17

Critic record keys:
['question_id', 'critic_status', 'passed', 'verdict', 'critique', 'critic_output', 'error']

Outcome counts:
Counter({'REVISE': 10, 'None': 4, 'PASS': 3})

Records with parsing errors or unknown verdicts:
{
  "question_id": "G3-01",
  "critic_status": "FAILED",
  "passed": null,
  "verdict": null,
  "critique": null,
  "critic_output": null,
  "error": "ValueError('No JSON object found')"
}
{
  "question_id": "G3-09",
  "critic_status": "FAILED",
  "passed": null,
  "verdict": null,
  "critique": null,
  "critic_output": null,
  "error": "ValueError('No JSON object found')"
}
{
  "question_id": "G3-10",
  "critic_status": "FAILED",
  "passed": null,
  "verdict": null,
  "critique": null,
  "critic_output": null,
  "error": "ValueError('No JSON object found')"
}
{
  "question_id": "G3-16",
  "critic_status": "FAILED",
  "passed": null,
  "verdict": null,
  "critique": null,
  "critic_output": null,
  "error": "ValueError('No JSON object foun

In [68]:
import json
import pandas as pd
from pathlib import Path

retrieval_csv = ROOT / "eval/results/G3_retrieval_results.csv"
retrieval_summary = ROOT / "eval/results/G3_retrieval_summary.json"

df_retrieval = pd.read_csv(retrieval_csv)

print("Retrieval result columns:")
print(list(df_retrieval.columns))

print("\nRetrieval results:")
print(df_retrieval.to_string(index=False))

print("\nRetrieval summary:")
with retrieval_summary.open("r", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2))

Retrieval result columns:
['question_id', 'question', 'gold_chunk_count', 'retrieved_chunk_count', 'recall_at_5', 'recall_at_10', 'reciprocal_rank', 'hit_at_5', 'hit_at_10']

Retrieval results:
question_id                                                                                                                                                                   question  gold_chunk_count  retrieved_chunk_count  recall_at_5  recall_at_10  reciprocal_rank  hit_at_5  hit_at_10
      G3-01 After a consumer notifies a financial institution of an electronic fund transfer error, what investigation and resolution steps are required, and within what time limits?                31                     10     0.096774      0.258065         1.000000         1          1
      G3-02                                                   What liability limits apply when an access device is lost or stolen and an unauthorized electronic fund transfer occurs?                23                     10     

In [69]:
import json
from pathlib import Path

def load_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

answers = load_jsonl(ROOT / "eval/results/G3_generated_answers.jsonl")
critics = load_jsonl(ROOT / "eval/results/G3_critic_results.jsonl")

def record_id(record):
    for key in ("question_id", "id", "case_id"):
        if record.get(key) is not None:
            return str(record[key])
    return None

failed_ids = {
    record_id(r)
    for r in critics
    if r.get("passed") is not True
}

print("Questions needing inspection:", sorted(x for x in failed_ids if x))

for answer in answers:
    qid = record_id(answer)
    if qid in failed_ids:
        print(f"\n{'='*25} {qid} {'='*25}")
        print(json.dumps(answer, indent=2, ensure_ascii=False)[:14000])

Questions needing inspection: ['G3-01', 'G3-03', 'G3-04', 'G3-05', 'G3-07', 'G3-08', 'G3-09', 'G3-10', 'G3-11', 'G3-12', 'G3-13', 'G3-14', 'G3-15', 'G3-16', 'G3-17']

========================= G3-01 =========================
{
  "question_id": "G3-01",
  "question": "After a consumer notifies a financial institution of an electronic fund transfer error, what investigation and resolution steps are required, and within what time limits?",
  "answer": "## Electronic Fund Transfer Error Investigation and Resolution Requirements\n\n### 1. Qualifying Notice of Error\n\nA financial institution must comply with error resolution procedures when it receives oral or written notice from a consumer that:\n- Is received **no later than 60 days** after the institution sends the periodic statement reflecting the alleged error [C:12CFR1005_1005.11_chunk_0001]\n\n---\n\n### 2. Standard 10-Business-Day Investigation Period\n\nThe default timeline requires the financial institution to:\n\n| Obligation | D

In [70]:

# ============================================================
# COMPLAINTIQ PHASE 7 — CONSOLIDATED CORRECTION CELL
# G1 schema validation + G3 citation correction + critic rerun
#
# Notebook: 06_eval.ipynb
# Safe:
#   - Does not modify gold datasets
#   - Does not modify graph/checkpoint database
#   - Does not publish reports to S3
#   - Backs up existing citation/critic result files locally
#
# IMPORTANT:
#   - Critic rerun invokes Bedrock.
#   - This cell does not force any critic result to PASS.
# ============================================================

import json
import re
import shutil
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter


# ------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------

G1_JSONL = ROOT / "eval/gold/G1_triage_gold.jsonl"
G1_CSV = ROOT / "eval/gold/G1_triage_manual_annotations.csv"

G3_ANSWER_PATH = ROOT / "eval/results/G3_generated_answers.jsonl"
G3_CITATION_PATH = ROOT / "eval/results/G3_citation_results.csv"
G3_CRITIC_PATH = ROOT / "eval/results/G3_critic_results.jsonl"
G3_CRITIC_CSV = ROOT / "eval/results/G3_critic_summary.csv"

RUN_CRITIC = True

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

BACKUP_DIR = ROOT / "eval" / "results" / "correction_backups" / RUN_ID
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 75)
print("COMPLAINTIQ PHASE 7 CORRECTION")
print("=" * 75)
print("Run ID:", RUN_ID)
print("Critic rerun enabled:", RUN_CRITIC)


# ------------------------------------------------------------
# 1. BACK UP EXISTING RESULTS
# ------------------------------------------------------------

for path in [
    G3_CITATION_PATH,
    G3_CRITIC_PATH,
    G3_CRITIC_CSV
]:
    if path.exists():
        backup_path = BACKUP_DIR / path.name
        shutil.copy2(path, backup_path)
        print("BACKUP:", backup_path)

print("PASS — Existing result files backed up.")


# ------------------------------------------------------------
# 2. G1 — VALIDATE ACTUAL NESTED JSON SCHEMA
# ------------------------------------------------------------

assert G1_JSONL.exists(), f"Missing G1 file: {G1_JSONL}"
assert G1_CSV.exists(), f"Missing G1 CSV: {G1_CSV}"

g1_records = []

with G1_JSONL.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        if line.strip():
            try:
                g1_records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"G1 invalid JSON at line {line_no}: {exc}"
                )

g1_ids = [x.get("case_id") for x in g1_records]

g1_schema_errors = []

required_annotation_fields = [
    "severity",
    "vulnerable_customer",
    "vulnerability_signals",
    "regulation_hints",
    "summary"
]

for idx, record in enumerate(g1_records, 1):

    if not record.get("case_id"):
        g1_schema_errors.append(
            f"Record {idx}: missing case_id"
        )

    if not isinstance(record.get("request"), str) or not record["request"].strip():
        g1_schema_errors.append(
            f"Record {idx}: missing request"
        )

    expected = record.get("expected")

    if not isinstance(expected, dict):
        g1_schema_errors.append(
            f"Record {idx}: expected must be an object"
        )
    else:
        for key in ["product", "issue"]:
            if not expected.get(key):
                g1_schema_errors.append(
                    f"Record {idx}: missing expected.{key}"
                )

    annotation = record.get("manual_annotation")

    if not isinstance(annotation, dict):
        g1_schema_errors.append(
            f"Record {idx}: manual_annotation must be an object"
        )
    else:
        for key in required_annotation_fields:
            if key not in annotation:
                g1_schema_errors.append(
                    f"Record {idx}: missing manual_annotation.{key}"
                )

g1_df = pd.read_csv(G1_CSV)

print("\nG1 VALIDATION")
print("-" * 75)
print("JSONL records:", len(g1_records))
print("Unique case IDs:", len(set(g1_ids)))
print("CSV rows:", len(g1_df))
print("CSV columns:", list(g1_df.columns))
print("Schema errors:", len(g1_schema_errors))

if g1_schema_errors:
    for error in g1_schema_errors[:30]:
        print("ERROR:", error)

assert len(g1_records) == 60, (
    f"Expected 60 G1 records; found {len(g1_records)}"
)

assert len(set(g1_ids)) == len(g1_ids), (
    "Duplicate G1 case IDs found"
)

assert not g1_schema_errors, (
    "G1 nested schema validation failed"
)

print("PASS — G1 JSONL STRUCTURE VALIDATED")

print("\nG1 MANUAL ANNOTATION PROGRESS")

for field in required_annotation_fields:

    if field in g1_df.columns:
        count = int(g1_df[field].notna().sum())
        print(f"{field}: {count}/{len(g1_df)}")

    else:
        print(
            f"{field}: CSV column not found; "
            "manual annotation mapping needs inspection"
        )


# ------------------------------------------------------------
# 3. G3 — ROBUST CITATION EXTRACTION
# ------------------------------------------------------------

def extract_citation_ids(answer):

    """
    Extract individual IDs from [C:...] markers.

    Supports:
        [C:chunk_A]
        [C:chunk_A] [C:chunk_B]
        [C:chunk_A; C:chunk_B]

    Does not treat the whole semicolon-separated value
    as one citation.
    """

    if not isinstance(answer, str):
        return []

    markers = re.findall(
        r"\[C:([^\]]+)\]",
        answer,
        flags=re.IGNORECASE
    )

    citation_ids = []

    for marker in markers:

        parts = re.split(r"\s*;\s*", marker)

        for part in parts:

            cleaned = part.strip()

            cleaned = re.sub(
                r"^\s*C\s*:\s*",
                "",
                cleaned,
                flags=re.IGNORECASE
            )

            cleaned = cleaned.strip().strip("`'\" ")

            if cleaned:
                citation_ids.append(cleaned)

    return citation_ids


# ------------------------------------------------------------
# 4. G3 — REVALIDATE CITATIONS AGAINST ACTUAL EVIDENCE
# ------------------------------------------------------------

assert "g3_answer_results" in globals(), (
    "g3_answer_results is unavailable. "
    "Run the G3 answer-generation cell first."
)

g3_citation_results = []

for result in g3_answer_results:

    if result.get("generation_status") != "SUCCESS":
        continue

    answer = result.get("answer", "")
    evidence = result.get("evidence", [])

    cited_ids = extract_citation_ids(answer)

    available_ids = {
        str(item.get("chunk_id")).strip()
        for item in evidence
        if isinstance(item, dict) and item.get("chunk_id")
    }

    valid_ids = [
        cid for cid in cited_ids
        if cid in available_ids
    ]

    invalid_ids = [
        cid for cid in cited_ids
        if cid not in available_ids
    ]

    g3_citation_results.append({
        "question_id": result["question_id"],
        "citation_count": len(cited_ids),
        "valid_citations": len(valid_ids),
        "invalid_citations": len(invalid_ids),
        "citation_validity": (
            len(valid_ids) / len(cited_ids)
            if cited_ids else 0.0
        ),
        "has_citation": bool(cited_ids),
        "invalid_chunk_ids": "; ".join(invalid_ids)
    })

g3_citation_df = pd.DataFrame(g3_citation_results)

if len(g3_citation_df):

    assert (
        g3_citation_df["valid_citations"]
        + g3_citation_df["invalid_citations"]
        == g3_citation_df["citation_count"]
    ).all()

    print("\nG3 CITATION VALIDATION")
    print("-" * 75)
    print(g3_citation_df.to_string(index=False))

    total_citations = int(
        g3_citation_df["citation_count"].sum()
    )

    total_valid = int(
        g3_citation_df["valid_citations"].sum()
    )

    total_invalid = int(
        g3_citation_df["invalid_citations"].sum()
    )

    print("\nTotal citations:", total_citations)
    print("Valid citations:", total_valid)
    print("Invalid citations:", total_invalid)

else:
    total_citations = 0
    total_valid = 0
    total_invalid = 0
    print("No successful generated answers available.")

G3_CITATION_PATH.parent.mkdir(parents=True, exist_ok=True)

g3_citation_df.to_csv(
    G3_CITATION_PATH,
    index=False
)

print("Saved:", G3_CITATION_PATH)


# ------------------------------------------------------------
# 5. G3 — RERUN CRITIC WITHOUT HIDING FAILURES
# ------------------------------------------------------------

g3_critic_results = []

if RUN_CRITIC:

    assert callable(critic_node), (
        "critic_node is unavailable"
    )

    successful_answers = [
        x for x in g3_answer_results
        if x.get("generation_status") == "SUCCESS"
    ]

    print("\nG3 CRITIC RERUN")
    print("-" * 75)
    print("Answers to evaluate:", len(successful_answers))

    for idx, result in enumerate(successful_answers, 1):

        question_id = result["question_id"]

        print(
            f"\n[{idx}/{len(successful_answers)}] "
            f"{question_id}"
        )

        critic_state = {
            "request": result["question"],
            "clean_request": result["question"],
            "intent": "REG_QA",
            "draft": result["answer"],
            "evidence": result["evidence"],
            "sql_rows": [],
            "similar_cases": [],
            "revisions": 0,
            "trace": [],
            "critique": None
        }

        try:

            critic_output = critic_node(critic_state)

            if not isinstance(critic_output, dict):
                raise TypeError(
                    "critic_node did not return a dictionary: "
                    f"{type(critic_output)}"
                )

            critique_value = critic_output.get(
                "critique",
                critic_output.get(
                    "critic",
                    critic_output
                )
            )

            passed = None

            if isinstance(critique_value, dict):
                passed = critique_value.get(
                    "passed",
                    critique_value.get("pass")
                )

            if passed is None:
                passed = critic_output.get(
                    "passed",
                    critic_output.get("pass")
                )

            verdict = critic_output.get("verdict")

            if verdict is None and isinstance(critique_value, dict):
                verdict = critique_value.get("verdict")

            # Do not infer PASS from a missing verdict.
            # Do not overwrite a deterministic failure.
            if passed is True and verdict not in ("PASS", None):
                passed = False

            g3_critic_results.append({
                "question_id": question_id,
                "critic_status": "SUCCESS",
                "passed": passed,
                "verdict": verdict,
                "critique": critique_value,
                "critic_output": critic_output,
                "error": None
            })

            print("Execution: SUCCESS")
            print("Passed:", passed)
            print("Verdict:", verdict)

        except Exception as exc:

            g3_critic_results.append({
                "question_id": question_id,
                "critic_status": "FAILED",
                "passed": None,
                "verdict": None,
                "critique": None,
                "critic_output": None,
                "error": repr(exc)
            })

            print("Execution: FAILED")
            print("Error:", repr(exc))


    # --------------------------------------------------------
    # 6. SAVE CRITIC RESULTS
    # --------------------------------------------------------

    g3_critic_df = pd.DataFrame([
        {
            "question_id": x["question_id"],
            "critic_status": x["critic_status"],
            "passed": x["passed"],
            "verdict": x["verdict"],
            "error": x["error"]
        }
        for x in g3_critic_results
    ])

    G3_CRITIC_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with G3_CRITIC_PATH.open("w", encoding="utf-8") as f:

        for item in g3_critic_results:
            f.write(
                json.dumps(
                    item,
                    ensure_ascii=False,
                    default=str
                ) + "\n"
            )

    g3_critic_df.to_csv(
        G3_CRITIC_CSV,
        index=False
    )

    print("\nG3 CRITIC SUMMARY")
    print("-" * 75)
    print(g3_critic_df.to_string(index=False))

else:

    print("\nCritic rerun skipped by configuration.")


# ------------------------------------------------------------
# 7. FINAL CORRECTION SUMMARY
# ------------------------------------------------------------

critic_passes = sum(
    x.get("passed") is True
    for x in g3_critic_results
)

critic_failures = sum(
    x.get("passed") is False
    for x in g3_critic_results
)

critic_unknown = sum(
    x.get("passed") is None
    for x in g3_critic_results
)

citation_failed_questions = (
    int((g3_citation_df["invalid_citations"] > 0).sum())
    if len(g3_citation_df) else 0
)

correction_summary = {
    "run_id": RUN_ID,
    "g1_records": len(g1_records),
    "g1_schema_errors": len(g1_schema_errors),
    "g1_manual_annotation_rows": len(g1_df),
    "g3_answers_evaluated": len(g3_citation_df),
    "g3_total_citations": total_citations,
    "g3_valid_citations": total_valid,
    "g3_invalid_citations": total_invalid,
    "g3_questions_with_invalid_citations": citation_failed_questions,
    "g3_critic_executions": len(g3_critic_results),
    "g3_critic_passes": critic_passes,
    "g3_critic_failures": critic_failures,
    "g3_critic_unknown": critic_unknown,
    "backup_directory": str(BACKUP_DIR)
}

SUMMARY_PATH = (
    ROOT / "eval/results/phase7_correction_summary.json"
)

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        correction_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 75)
print("PHASE 7 CORRECTION SUMMARY")
print("=" * 75)
print(json.dumps(correction_summary, indent=2))

print("\nSaved:", SUMMARY_PATH)

print("\nIMPORTANT:")
print("- G1 structural validation is separate from manual annotation.")
print("- Citation validity checks membership in supplied evidence only.")
print("- Critic PASS does not independently prove regulatory correctness.")
print("- Critic JSON parsing failures remain visible as UNKNOWN/FAILED.")
print("- Retrieval and evidence-grounding defects still require review.")
print("- No S3 publishing or audit writes were performed.")

print("\nPASS — CONSOLIDATED CORRECTION CELL EXECUTED.")


COMPLAINTIQ PHASE 7 CORRECTION
Run ID: 20261106T141625Z
Critic rerun enabled: True
BACKUP: /home/sagemaker-user/complaintiq/eval/results/correction_backups/20261106T141625Z/G3_citation_results.csv
BACKUP: /home/sagemaker-user/complaintiq/eval/results/correction_backups/20261106T141625Z/G3_critic_results.jsonl
BACKUP: /home/sagemaker-user/complaintiq/eval/results/correction_backups/20261106T141625Z/G3_critic_summary.csv
PASS — Existing result files backed up.

G1 VALIDATION
---------------------------------------------------------------------------
JSONL records: 60
Unique case IDs: 60
CSV rows: 60
CSV columns: ['case_id', 'severity', 'vulnerable_customer', 'vulnerability_signals', 'regulation_hints', 'summary', 'reviewer', 'review_notes', 'annotation_status']
Schema errors: 0
PASS — G1 JSONL STRUCTURE VALIDATED

G1 MANUAL ANNOTATION PROGRESS
severity: 0/60
vulnerable_customer: 0/60
vulnerability_signals: 0/60
regulation_hints: 0/60
summary: 0/60

G3 CITATION VALIDATION
----------------

In [71]:

# ============================================================
# CELL 125 — CRITIC / LLM JSON PARSING DIAGNOSTICS
# ============================================================

import inspect
import dis
import json
import types
from pathlib import Path

print("=" * 80)
print("COMPLAINTIQ — CRITIC AND JSON PARSING DIAGNOSTICS")
print("=" * 80)


# ------------------------------------------------------------
# 1. INSPECT CRITIC NODE
# ------------------------------------------------------------

print("\n1. CRITIC NODE")
print("-" * 80)

print("Object:", critic_node)
print("Type:", type(critic_node))

try:
    print("Signature:", inspect.signature(critic_node))
except Exception as e:
    print("Signature unavailable:", repr(e))

critic_code = getattr(critic_node, "__code__", None)

if critic_code:
    print("Function name:", critic_code.co_name)
    print("Referenced global names:")
    print(sorted(set(critic_code.co_names)))

    print("\nFunction constants:")
    for value in critic_code.co_consts:
        if isinstance(value, str):
            print(repr(value[:500]))

    print("\nDisassembled critic_node:")
    dis.dis(critic_node)

else:
    print("No Python code object available.")


# ------------------------------------------------------------
# 2. INSPECT CRITIC GLOBAL DEPENDENCIES
# ------------------------------------------------------------

print("\n\n2. CRITIC GLOBAL DEPENDENCIES")
print("-" * 80)

critic_globals = getattr(critic_node, "__globals__", {})

dependency_names = (
    critic_code.co_names if critic_code else []
)

for name in dependency_names:

    if name not in critic_globals:
        continue

    obj = critic_globals[name]

    if inspect.isfunction(obj):

        print(f"\nFUNCTION: {name}")

        try:
            print("Signature:", inspect.signature(obj))
        except Exception:
            pass

        code = getattr(obj, "__code__", None)

        if code:
            print("Referenced names:", sorted(set(code.co_names)))

        try:
            print(inspect.getsource(obj)[:8000])
        except Exception:
            print("Source unavailable; runtime function is dynamically restored.")

    elif isinstance(obj, type):

        print(f"\nCLASS: {name}")
        print("Class:", obj)

    else:

        print(f"\nGLOBAL: {name}")
        print("Type:", type(obj))


# ------------------------------------------------------------
# 3. INSPECT LLM WRAPPER
# ------------------------------------------------------------

print("\n\n3. LLM WRAPPER")
print("-" * 80)

print("LLM object:", llm)
print("LLM type:", type(llm))

for method_name in ["chat_json", "chat"]:

    method = getattr(llm, method_name, None)

    print(f"\nMETHOD: llm.{method_name}")

    if method is None:
        print("Not found")
        continue

    try:
        print("Signature:", inspect.signature(method))
    except Exception as e:
        print("Signature unavailable:", repr(e))

    function_obj = getattr(method, "__func__", method)
    code = getattr(function_obj, "__code__", None)

    if code:

        print("Function name:", code.co_name)
        print("Referenced names:", sorted(set(code.co_names)))

        print("String constants:")
        for value in code.co_consts:
            if isinstance(value, str):
                print(repr(value[:1000]))

        print("\nDisassembly:")
        dis.dis(function_obj)

    try:
        print("\nPython source:")
        print(inspect.getsource(function_obj)[:12000])
    except Exception:
        print("Source unavailable.")


# ------------------------------------------------------------
# 4. INSPECT LLM CLASS JSON-RELATED METHODS
# ------------------------------------------------------------

print("\n\n4. LLM CLASS METHODS")
print("-" * 80)

llm_class = type(llm)

for method_name in dir(llm_class):

    if any(
        term in method_name.lower()
        for term in ["json", "parse", "repair", "response"]
    ):

        print("\nMethod:", method_name)

        try:
            method = getattr(llm_class, method_name)
            print("Signature:", inspect.signature(method))
        except Exception:
            pass


# ------------------------------------------------------------
# 5. INSPECT FAILED CRITIC RECORDS
# ------------------------------------------------------------

print("\n\n5. FAILED CRITIC RECORDS")
print("-" * 80)

critic_path = ROOT / "eval/results/G3_critic_results.jsonl"

critic_records = []

with critic_path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            critic_records.append(json.loads(line))

for record in critic_records:

    if record.get("critic_status") == "FAILED":

        print("\nQUESTION:", record.get("question_id"))
        print("ERROR:", record.get("error"))

        print("Available fields:", list(record.keys()))

        print(
            "Raw response field present:",
            any(
                key in record
                for key in [
                    "raw_response",
                    "response",
                    "model_response"
                ]
            )
        )


# ------------------------------------------------------------
# 6. INSPECT G3-17 CONTRADICTION
# ------------------------------------------------------------

print("\n\n6. G3-17 CRITIC CONTRADICTION")
print("-" * 80)

for record in critic_records:

    if record.get("question_id") == "G3-17":

        print(json.dumps(
            record,
            indent=2,
            ensure_ascii=False,
            default=str
        )[:18000])


# ------------------------------------------------------------
# 7. CHECK WHETHER THE ORIGINAL CRITIC OUTPUT IS AVAILABLE
# ------------------------------------------------------------

print("\n\n7. ORIGINAL CRITIC RESULT BACKUP")
print("-" * 80)

backup_path = (
    ROOT
    / "eval/results/correction_backups/20261106T141625Z/"
    / "G3_critic_results.jsonl"
)

if backup_path.exists():

    with backup_path.open("r", encoding="utf-8") as f:
        original_records = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    print("Original records:", len(original_records))

    for record in original_records:

        if record.get("question_id") in [
            "G3-01", "G3-08", "G3-09", "G3-10", "G3-17"
        ]:

            print("\nQUESTION:", record.get("question_id"))
            print("Verdict:", record.get("verdict"))
            print("Passed:", record.get("passed"))
            print("Error:", record.get("error"))

            print("Fields:", list(record.keys()))

else:
    print("Original backup not found.")


print("\n" + "=" * 80)
print("DIAGNOSTICS COMPLETE — NO BEDROCK CALLS MADE")
print("=" * 80)


COMPLAINTIQ — CRITIC AND JSON PARSING DIAGNOSTICS

1. CRITIC NODE
--------------------------------------------------------------------------------
Object: <function critic_node at 0x7fcdb09071a0>
Type: <class 'function'>
Signature: (state: __main__.CIQState) -> dict
Function name: critic_node
Referenced global names:
['CRITIC_SYS', 'CriticOut', 'MODELS', 'PII', 'append', 'chat_json', 'findall', 'get', 'join', 'len', 'llm', 'model_dump', 're', 'search', 'set', 'sorted', 'str', 'verdict']

Function constants:
'draft'
''
'evidence'
'chunk_id'
'\\[C:([^\\]]+)\\]'
'Unknown citation ID: '
'Possible unredacted PII: '
'\n\n'
'judge'
'EVIDENCE:\n'
'\n\nDATA:\n'
'sql_rows'
'\n\nDRAFT:\n'
'PASS'
'revisions'
'critic'

Disassembled critic_node:
 23           0 RESUME                   0

 24           2 LOAD_FAST                0 (state)
              4 LOAD_ATTR                1 (NULL|self + get)
             24 LOAD_CONST               1 ('draft')
             26 LOAD_CONST               2 ('')
 

In [72]:

import inspect
import json
import re

print("=" * 90)
print("1. CRITIC OUTPUT SCHEMA")
print("=" * 90)

print(CriticOut.model_json_schema())

print("\n" + "=" * 90)
print("2. COMPLETE CRITIC SYSTEM PROMPT")
print("=" * 90)

print(CRITIC_SYS)

print("\n" + "=" * 90)
print("3. G3-17 DRAFT AND EVIDENCE")
print("=" * 90)

g3_17 = next(
    (r for r in g3_answer_results if r.get("question_id") == "G3-17"),
    None
)

if g3_17 is None:
    print("G3-17 not found in g3_answer_results")
else:
    print("\nDRAFT:")
    print(g3_17.get("draft", ""))

    print("\nCITATION MARKERS:")
    print(re.findall(r"\[C:([^\]]+)\]", g3_17.get("draft", "")))

    print("\nEVIDENCE:")
    for e in g3_17.get("evidence", []):
        print({
            "chunk_id": e.get("chunk_id"),
            "text": str(e.get("text", ""))[:1500]
        })

print("\n" + "=" * 90)
print("4. ANSWER RECORDS FOR JSON PARSING FAILURES")
print("=" * 90)

for qid in ["G3-01", "G3-08", "G3-09", "G3-10"]:
    record = next(
        (r for r in g3_answer_results if r.get("question_id") == qid),
        None
    )

    print(f"\n{'-' * 35} {qid} {'-' * 35}")

    if record is None:
        print("Answer record not found")
        continue

    print("\nDRAFT:")
    print(record.get("draft", ""))

    print("\nEVIDENCE COUNT:")
    evidence = record.get("evidence", [])
    print(len(evidence))

    print("\nEVIDENCE IDS:")
    print([e.get("chunk_id") for e in evidence])

print("\n" + "=" * 90)
print("DIAGNOSTICS COMPLETE — NO BEDROCK CALLS OR FILE WRITES")
print("=" * 90)


1. CRITIC OUTPUT SCHEMA
{'properties': {'verdict': {'enum': ['PASS', 'REVISE'], 'title': 'Verdict', 'type': 'string'}, 'unsupported_claims': {'items': {'type': 'string'}, 'title': 'Unsupported Claims', 'type': 'array'}, 'tone_issues': {'items': {'type': 'string'}, 'title': 'Tone Issues', 'type': 'array'}, 'feedback': {'title': 'Feedback', 'type': 'string'}}, 'required': ['verdict', 'feedback'], 'title': 'CriticOut', 'type': 'object'}

2. COMPLETE CRITIC SYSTEM PROMPT

You are the compliance critic for a bank complaints-intelligence copilot.

Review the draft against the supplied evidence and data.
Return REVISE if:
- A regulatory statement lacks a citation or is not supported by its cited evidence.
- A numeric claim is not supported by the supplied data.
- The draft admits liability on behalf of a bank.
- It presents an allegation as a proven fact.
- It gives an unsupported promise or conclusion.

Return PASS only when the draft is adequately supported.
Unknown citations are invalid.
T

In [73]:

# =============================================================================
# COMPLAINTIQ — PHASE 7 G3 VALIDATION AND RESOLUTION
# =============================================================================
# Scope:
#   - Back up existing critic outputs
#   - Repair JSON extraction
#   - Repair critic citation parsing
#   - Enforce consistent PASS/REVISE decisions
#   - Recover or regenerate missing drafts where possible
#   - Rerun G3 critic evaluation
#
# Does NOT:
#   - Write to S3
#   - Publish reports
#   - Modify graph checkpoints
#   - Recompile the LangGraph workflow
# =============================================================================

import os
import re
import json
import shutil
import inspect
import hashlib
from pathlib import Path
from datetime import datetime, timezone

from pydantic import ValidationError

# -----------------------------------------------------------------------------
# 0. PRE-FLIGHT
# -----------------------------------------------------------------------------

print("=" * 85)
print("COMPLAINTIQ — G3 VALIDATION AND RESOLUTION")
print("=" * 85)

required_objects = [
    "llm", "critic_node", "CriticOut", "CRITIC_SYS",
    "MODELS", "g3_answer_results"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required runtime objects are missing: "
        + ", ".join(missing_objects)
        + ". Run the Notebook 6 bootstrap cells first."
    )

ROOT = Path("/home/sagemaker-user/complaintiq")
RESULTS = ROOT / "eval" / "results"

if not RESULTS.exists():
    raise FileNotFoundError(f"Results directory not found: {RESULTS}")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
BACKUP_DIR = RESULTS / "correction_backups" / RUN_ID
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Back up existing G3 result artifacts.
backup_patterns = [
    "G3_citation_results.csv",
    "G3_critic_results.jsonl",
    "G3_critic_summary.csv",
    "G3_generated_answers.jsonl",
    "phase7_correction_summary.json"
]

for filename in backup_patterns:
    source = RESULTS / filename
    if source.exists():
        shutil.copy2(source, BACKUP_DIR / filename)

print(f"Backup directory: {BACKUP_DIR}")
print("Existing result artifacts backed up where present.")

# -----------------------------------------------------------------------------
# 1. REPAIR JSON EXTRACTION
# -----------------------------------------------------------------------------

_original_chat_json = llm.chat_json

def _extract_json_object(raw):
    """
    Extract and validate a JSON object without relying on the first
    opening brace and final closing brace.
    """
    if raw is None:
        raise ValueError("Model returned None")

    raw = str(raw).strip()

    if not raw:
        raise ValueError("Model returned an empty response")

    decoder = json.JSONDecoder()

    # Try every opening brace. This handles prose and fenced JSON.
    for match in re.finditer(r"\{", raw):
        start = match.start()

        try:
            obj, end = decoder.raw_decode(raw[start:])

            if isinstance(obj, dict):
                return json.dumps(obj)

        except json.JSONDecodeError:
            continue

    raise ValueError(
        "No valid JSON object found. "
        f"Response length={len(raw)}; "
        f"response preview={raw[:1200]!r}"
    )


def _repaired_chat_json(
    model_id,
    system,
    user,
    schema,
    **kwargs
):
    """
    JSON extraction with one controlled repair attempt.
    Raw responses are retained in diagnostic attributes.
    """
    json_system = (
        system
        + "\n\nReturn exactly one valid JSON object. "
          "Do not include markdown fences or explanatory prose."
    )

    raw_responses = []

    raw = llm.chat(
        model_id=model_id,
        system=json_system,
        user=user,
        **kwargs
    )
    raw_responses.append(raw)

    last_error = None

    for attempt in range(2):
        try:
            blob = _extract_json_object(raw)
            return schema.model_validate_json(blob)

        except (ValueError, ValidationError) as exc:
            last_error = exc

            if attempt == 1:
                break

            repair_user = (
                user
                + "\n\nYour previous response could not be parsed "
                  "or did not match the required schema."
                + f"\nValidation error: {str(exc)}"
                + "\nReturn only a valid JSON object matching this schema:"
                + json.dumps(schema.model_json_schema(), indent=2)
            )

            raw = llm.chat(
                model_id=model_id,
                system=json_system,
                user=repair_user,
                use_cache=False,
                **kwargs
            )
            raw_responses.append(raw)

    # Retain the failed responses for investigation.
    _repaired_chat_json.last_raw_responses = raw_responses

    raise ValueError(
        "JSON validation failed after the repair attempt. "
        f"Last error: {last_error}"
    )


_repaired_chat_json.last_raw_responses = []
llm.chat_json = _repaired_chat_json

print("\n[1/7] JSON extraction repaired.")

# -----------------------------------------------------------------------------
# 2. REPAIR CITATION PARSING
# -----------------------------------------------------------------------------

def extract_citation_ids(text):
    """
    Handles:
      [C:chunk_A]
      [C:chunk_A; chunk_B]
      [C:chunk_A; C:chunk_B]
      [C:chunk_A, chunk_B]

    Returns individual citation IDs.
    """
    if not text:
        return []

    found = []

    for marker in re.findall(r"\[C:([^\]]+)\]", str(text)):
        parts = re.split(r"[;,]", marker)

        for part in parts:
            cid = re.sub(r"^\s*C:\s*", "", part).strip()

            if cid:
                found.append(cid)

    return list(dict.fromkeys(found))


def validate_draft_citations(draft, evidence):
    known_ids = {
        str(e.get("chunk_id"))
        for e in evidence
        if isinstance(e, dict) and e.get("chunk_id")
    }

    cited_ids = extract_citation_ids(draft)

    invalid_ids = sorted(set(cited_ids) - known_ids)

    return {
        "citation_count": len(cited_ids),
        "valid_count": len(set(cited_ids) & known_ids),
        "invalid_ids": invalid_ids,
        "known_ids": sorted(known_ids)
    }


print("[2/7] Citation parser repaired.")

# -----------------------------------------------------------------------------
# 3. REPAIR CRITIC NODE
# -----------------------------------------------------------------------------

def corrected_critic_node(state):
    draft = str(state.get("draft") or "").strip()
    evidence = state.get("evidence") or []

    if not draft:
        raise ValueError(
            "Critic cannot evaluate an empty draft. "
            "Recover or regenerate the draft first."
        )

    citation_check = validate_draft_citations(draft, evidence)

    problems = [
        f"Unknown citation ID: {cid}"
        for cid in citation_check["invalid_ids"]
    ]

    # Existing PII patterns remain enforced.
    for pattern, tag in PII:
        if pattern.search(draft):
            problems.append(f"Possible unredacted PII: {tag}")

    evidence_text = "\n\n".join(
        f"[{e.get('chunk_id')}] "
        f"{str(e.get('text', ''))[:1200]}"
        for e in evidence
        if isinstance(e, dict)
    )

    judge = llm.chat_json(
        model_id=MODELS["judge"],
        system=CRITIC_SYS,
        user=(
            "EVIDENCE:\n"
            + evidence_text
            + "\n\nDATA:\n"
            + str((state.get("sql_rows") or [])[:30])
            + "\n\nDRAFT:\n"
            + draft
        ),
        schema=CriticOut,
        max_tokens=1200,
        temperature=0.0
    )

    judge_data = judge.model_dump()

    judge_verdict = judge_data.get("verdict", "REVISE")

    # Local validation always takes precedence.
    final_pass = (
        judge_verdict == "PASS"
        and len(problems) == 0
    )

    final_verdict = "PASS" if final_pass else "REVISE"

    # Keep the judge feedback and local findings separate.
    critique = {
        "pass": final_pass,
        "verdict": final_verdict,
        "judge_verdict": judge_verdict,
        "problems": problems,
        "unsupported_claims": judge_data.get(
            "unsupported_claims", []
        ),
        "tone_issues": judge_data.get("tone_issues", []),
        "feedback": judge_data.get("feedback", ""),
        "citation_validation": {
            "citation_count": citation_check["citation_count"],
            "valid_count": citation_check["valid_count"],
            "invalid_ids": citation_check["invalid_ids"]
        }
    }

    revisions = (
        int(state.get("revisions", 0))
        if final_pass
        else int(state.get("revisions", 0)) + 1
    )

    return {
        "critique": critique,
        "revisions": revisions,
        "trace": [
            {
                "node": "critic",
                "pass": final_pass,
                "problems": len(problems),
                "judge_verdict": judge_verdict
            }
        ]
    }


_original_critic_node = critic_node
critic_node = corrected_critic_node

print("[3/7] Critic citation and verdict logic repaired.")

# -----------------------------------------------------------------------------
# 4. RECOVER DRAFTS FROM EXISTING RECORDS
# -----------------------------------------------------------------------------

def find_draft_in_record(record):
    """
    Look for an existing draft in common output locations.
    Does not assume that record['draft'] is the only answer field.
    """
    candidate_keys = [
        "draft",
        "answer",
        "generated_answer",
        "response",
        "final_answer",
        "text",
        "output"
    ]

    for key in candidate_keys:
        value = record.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip(), key

        if isinstance(value, dict):
            for nested_key in candidate_keys:
                nested_value = value.get(nested_key)

                if isinstance(nested_value, str) and nested_value.strip():
                    return nested_value.strip(), f"{key}.{nested_key}"

    return "", None


def find_question_in_record(record):
    for key in [
        "question",
        "request",
        "clean_request",
        "prompt",
        "query"
    ]:
        value = record.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    return ""


def regenerate_missing_draft(record):
    """
    Regenerate only if both the question and evidence are present.
    No new retrieval is performed.
    """
    question = find_question_in_record(record)
    evidence = record.get("evidence") or []

    if not question:
        return None, "Question not available in saved record"

    if not evidence:
        return None, "Evidence not available in saved record"

    result = drafter_node({
        "request": question,
        "clean_request": question,
        "intent": "REG_QA",
        "evidence": evidence,
        "sql_rows": [],
        "similar_cases": [],
        "draft": "",
        "revisions": 0,
        "trace": []
    })

    draft, source = find_draft_in_record(result)

    if not draft:
        return None, (
            "Drafter returned no recoverable draft. "
            f"Returned keys: {list(result.keys())}"
        )

    return {
        "draft": draft,
        "drafter_output": result,
        "draft_source": f"regenerated:{source}"
    }, None


draft_recovery = []
prepared_answers = []

for record in g3_answer_results:
    record_copy = dict(record)

    draft, source = find_draft_in_record(record_copy)

    if draft:
        record_copy["draft"] = draft
        recovery_status = "RECOVERED_FROM_SAVED_RECORD"

    else:
        regenerated, error = regenerate_missing_draft(record_copy)

        if regenerated:
            record_copy.update(regenerated)
            recovery_status = "REGENERATED_FROM_SAVED_EVIDENCE"
        else:
            recovery_status = "UNRESOLVED"
            record_copy["_draft_recovery_error"] = error

    record_copy["_draft_recovery_status"] = recovery_status

    draft_recovery.append({
        "question_id": record_copy.get("question_id"),
        "status": recovery_status,
        "draft_length": len(record_copy.get("draft", "")),
        "error": record_copy.get("_draft_recovery_error")
    })

    prepared_answers.append(record_copy)


print("\n[4/7] Draft recovery results:")

for item in draft_recovery:
    print(
        item["question_id"],
        "|",
        item["status"],
        "| draft length:",
        item["draft_length"],
        "|",
        item["error"] or ""
    )

# -----------------------------------------------------------------------------
# 5. VALIDATE G3 ANSWERS AND CITATIONS
# -----------------------------------------------------------------------------

citation_results = []

for record in prepared_answers:
    draft = str(record.get("draft") or "").strip()
    evidence = record.get("evidence") or []

    check = validate_draft_citations(draft, evidence)

    citation_results.append({
        "question_id": record.get("question_id"),
        "draft_present": bool(draft),
        "draft_length": len(draft),
        "citation_count": check["citation_count"],
        "valid_count": check["valid_count"],
        "invalid_count": len(check["invalid_ids"]),
        "invalid_ids": check["invalid_ids"],
        "recovery_status": record.get("_draft_recovery_status")
    })


print("\n[5/7] Citation validation:")

for item in citation_results:
    print(
        item["question_id"],
        "| draft:",
        item["draft_present"],
        "| citations:",
        item["valid_count"],
        "/",
        item["citation_count"],
        "| invalid:",
        item["invalid_count"]
    )

# -----------------------------------------------------------------------------
# 6. RERUN CRITIC FOR RECOVERABLE DRAFTS
# -----------------------------------------------------------------------------

critic_results = []

for record in prepared_answers:
    qid = record.get("question_id")
    draft = str(record.get("draft") or "").strip()

    if not draft:
        critic_results.append({
            "question_id": qid,
            "critic_status": "NOT_EVALUATED",
            "passed": False,
            "verdict": "REVISE",
            "error": record.get("_draft_recovery_error")
                or "No draft available"
        })
        continue

    try:
        output = critic_node({
            "draft": draft,
            "evidence": record.get("evidence") or [],
            "sql_rows": record.get("sql_rows") or [],
            "revisions": 0
        })

        critique = output.get("critique", {})

        passed = bool(critique.get("pass", False))
        verdict = critique.get("verdict", "REVISE")

        # Final defensive assertion.
        if critique.get("problems"):
            passed = False
            verdict = "REVISE"

        if not passed:
            verdict = "REVISE"

        critic_results.append({
            "question_id": qid,
            "critic_status": "SUCCESS",
            "passed": passed,
            "verdict": verdict,
            "critique": critique,
            "critic_output": output,
            "error": None
        })

        print(
            qid,
            "->",
            verdict,
            "| passed:",
            passed
        )

    except Exception as exc:
        critic_results.append({
            "question_id": qid,
            "critic_status": "ERROR",
            "passed": False,
            "verdict": "REVISE",
            "error": repr(exc)
        })

        print(qid, "-> ERROR:", repr(exc))

# -----------------------------------------------------------------------------
# 7. SAVE RESULTS AND VALIDATION SUMMARY
# -----------------------------------------------------------------------------

import pandas as pd

citation_df = pd.DataFrame(citation_results)
critic_df = pd.DataFrame([
    {
        "question_id": r.get("question_id"),
        "critic_status": r.get("critic_status"),
        "passed": r.get("passed"),
        "verdict": r.get("verdict"),
        "error": r.get("error"),
        "recovery_status": next(
            (
                x["status"]
                for x in draft_recovery
                if x["question_id"] == r.get("question_id")
            ),
            None
        )
    }
    for r in critic_results
])

citation_df.to_csv(
    RESULTS / "G3_citation_results.csv",
    index=False
)

critic_df.to_csv(
    RESULTS / "G3_critic_summary.csv",
    index=False
)

with open(
    RESULTS / "G3_critic_results.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for result in critic_results:
        f.write(json.dumps(result, ensure_ascii=False, default=str) + "\n")

with open(
    RESULTS / "G3_generated_answers_validated.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for record in prepared_answers:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

summary = {
    "run_id": RUN_ID,
    "total_answers": len(prepared_answers),
    "drafts_recovered_or_generated": sum(
        1 for x in draft_recovery
        if x["status"] != "UNRESOLVED"
    ),
    "drafts_unresolved": sum(
        1 for x in draft_recovery
        if x["status"] == "UNRESOLVED"
    ),
    "answers_with_invalid_citations": sum(
        1 for x in citation_results
        if x["invalid_count"] > 0
    ),
    "total_citations": sum(
        x["citation_count"] for x in citation_results
    ),
    "valid_citations": sum(
        x["valid_count"] for x in citation_results
    ),
    "critic_executions": sum(
        1 for x in critic_results
        if x["critic_status"] == "SUCCESS"
    ),
    "critic_errors": sum(
        1 for x in critic_results
        if x["critic_status"] == "ERROR"
    ),
    "not_evaluated": sum(
        1 for x in critic_results
        if x["critic_status"] == "NOT_EVALUATED"
    ),
    "critic_passes": sum(
        1 for x in critic_results
        if x["passed"] is True
    ),
    "critic_revisions": sum(
        1 for x in critic_results
        if x["verdict"] == "REVISE"
    ),
    "backup_directory": str(BACKUP_DIR)
}

with open(
    RESULTS / "phase7_validation_resolution_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 85)
print("FINAL VALIDATION SUMMARY")
print("=" * 85)

print(json.dumps(summary, indent=2))

print("\nSaved artifacts:")
print(RESULTS / "G3_citation_results.csv")
print(RESULTS / "G3_critic_summary.csv")
print(RESULTS / "G3_critic_results.jsonl")
print(RESULTS / "G3_generated_answers_validated.jsonl")
print(RESULTS / "phase7_validation_resolution_summary.json")

print("\n" + "=" * 85)
print("VALIDATION AND RESOLUTION CELL COMPLETED")
print("=" * 85)


COMPLAINTIQ — G3 VALIDATION AND RESOLUTION
Backup directory: /home/sagemaker-user/complaintiq/eval/results/correction_backups/20261106T142806Z
Existing result artifacts backed up where present.

[1/7] JSON extraction repaired.
[2/7] Citation parser repaired.
[3/7] Critic citation and verdict logic repaired.

[4/7] Draft recovery results:
G3-01 | RECOVERED_FROM_SAVED_RECORD | draft length: 3322 | 
G3-02 | RECOVERED_FROM_SAVED_RECORD | draft length: 2501 | 
G3-03 | RECOVERED_FROM_SAVED_RECORD | draft length: 3111 | 
G3-04 | RECOVERED_FROM_SAVED_RECORD | draft length: 4098 | 
G3-05 | RECOVERED_FROM_SAVED_RECORD | draft length: 3500 | 
G3-06 | RECOVERED_FROM_SAVED_RECORD | draft length: 2429 | 
G3-07 | RECOVERED_FROM_SAVED_RECORD | draft length: 3110 | 
G3-08 | RECOVERED_FROM_SAVED_RECORD | draft length: 3555 | 
G3-09 | RECOVERED_FROM_SAVED_RECORD | draft length: 3962 | 
G3-10 | RECOVERED_FROM_SAVED_RECORD | draft length: 2605 | 
G3-11 | RECOVERED_FROM_SAVED_RECORD | draft length: 2100 | 


In [74]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/home/sagemaker-user/complaintiq")
RESULTS = ROOT / "eval" / "results"

critic_file = RESULTS / "G3_critic_results.jsonl"
summary_file = RESULTS / "G3_critic_summary.csv"

# Load critic results
critic_records = []

with open(critic_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            critic_records.append(json.loads(line))

print("=" * 100)
print("COMPLAINTIQ — G3 CRITIC FAILURE ANALYSIS")
print("=" * 100)

rows = []

for record in critic_records:

    question_id = (
        record.get("question_id")
        or record.get("id")
        or record.get("case_id")
        or "UNKNOWN"
    )

    passed = record.get("passed")
    critique = record.get("critique", {})

    if not isinstance(critique, dict):
        critique = {}

    verdict = (
        critique.get("verdict")
        or record.get("verdict")
        or "UNKNOWN"
    )

    unsupported = (
        critique.get("unsupported_claims")
        or record.get("unsupported_claims")
        or []
    )

    tone_issues = (
        critique.get("tone_issues")
        or record.get("tone_issues")
        or []
    )

    feedback = (
        critique.get("feedback")
        or record.get("feedback")
        or ""
    )

    if passed is False or verdict == "REVISE":

        rows.append({
            "question_id": question_id,
            "verdict": verdict,
            "passed": passed,
            "unsupported_claims": json.dumps(
                unsupported, ensure_ascii=False
            ),
            "tone_issues": json.dumps(
                tone_issues, ensure_ascii=False
            ),
            "feedback": feedback
        })

failure_df = pd.DataFrame(rows)

print(f"\nTotal critic records: {len(critic_records)}")
print(f"Failed/revision records: {len(failure_df)}")

if not failure_df.empty:

    print("\n" + "=" * 100)
    print("QUESTION-WISE FAILURE DETAILS")
    print("=" * 100)

    for _, row in failure_df.iterrows():

        print(f"\nQUESTION: {row['question_id']}")
        print(f"VERDICT: {row['verdict']}")
        print(f"PASSED: {row['passed']}")

        print("\nUNSUPPORTED CLAIMS:")
        print(row["unsupported_claims"])

        print("\nTONE ISSUES:")
        print(row["tone_issues"])

        print("\nCRITIC FEEDBACK:")
        print(row["feedback"])

        print("-" * 100)

    output_file = RESULTS / "G3_failure_analysis.csv"
    failure_df.to_csv(output_file, index=False)

    print(f"\nFailure analysis saved: {output_file}")

else:
    print("\nNo critic failures found.")

# Show final outcome distribution
print("\n" + "=" * 100)
print("FINAL OUTCOME DISTRIBUTION")
print("=" * 100)

print(
    pd.Series([
        "PASS" if r.get("passed") is True else
        "REVISE" if r.get("passed") is False else
        "UNKNOWN"
        for r in critic_records
    ]).value_counts()
)

COMPLAINTIQ — G3 CRITIC FAILURE ANALYSIS

Total critic records: 17
Failed/revision records: 15

QUESTION-WISE FAILURE DETAILS

QUESTION: G3-01
VERDICT: REVISE
PASSED: False

UNSUPPORTED CLAIMS:
["Section 1 claims the notice must be received 'no later than 60 days after the institution sends the periodic statement' cited to chunk_0001, but that chunk (ending at the start of paragraph (b)) does not contain the 60-day timing text; the citation does not support the claim.", "Section 3 states the institution 'may withhold a maximum of $50' from the provisionally credited amount, cited to chunk_0002. The evidence (chunk_0002) describes the withholding allowance tied to § 1005.6(a) but is truncated and does not specify a $50 maximum; the $50 figure is not supported by the cited evidence.", "Section 3 'Advises the consumer within 2 business days of the provisional crediting' and 'Gives the consumer full use of the funds' are cited to chunk_0002, but these specifics appear in exam_manual_v2_201

In [75]:
# =============================================================================
# COMPLAINTIQ — G3 TARGETED ANSWER REPAIR
#
# Purpose:
#   Repair the 15 G3 answers marked REVISE using:
#     1. Existing answer and evidence
#     2. Question-specific critic feedback
#     3. Fresh research evidence
#     4. A new drafter pass
#     5. A new critic evaluation
#
# Safety:
#   - Backs up existing result files
#   - Writes new repair artifacts separately
#   - Does NOT write to S3
#   - Does NOT publish reports
#   - Does NOT modify LangGraph checkpoints
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import inspect
import pandas as pd
import re
import traceback

ROOT = Path("/home/sagemaker-user/complaintiq")
RESULTS = ROOT / "eval" / "results"
GOLD = ROOT / "eval" / "gold"

RESULTS.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
BACKUP_DIR = RESULTS / "correction_backups" / f"{RUN_ID}_targeted_repair"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("COMPLAINTIQ — G3 TARGETED ANSWER REPAIR")
print("=" * 100)
print("Run ID:", RUN_ID)
print("Backup:", BACKUP_DIR)


# -----------------------------------------------------------------------------
# 1. BACKUP CURRENT VALIDATED ARTIFACTS
# -----------------------------------------------------------------------------

FILES_TO_BACKUP = [
    "G3_generated_answers_validated.jsonl",
    "G3_critic_results.jsonl",
    "G3_critic_summary.csv",
    "G3_citation_results.csv",
    "G3_failure_analysis.csv",
    "phase7_validation_resolution_summary.json",
]

for filename in FILES_TO_BACKUP:
    source = RESULTS / filename
    if source.exists():
        shutil.copy2(source, BACKUP_DIR / filename)

print("\n[1/8] Existing artifacts backed up.")


# -----------------------------------------------------------------------------
# 2. CHECK REQUIRED RUNTIME OBJECTS
# -----------------------------------------------------------------------------

required_objects = [
    "research_node",
    "drafter_node",
    "critic_node",
]

missing = [name for name in required_objects if name not in globals()]

if missing:
    raise RuntimeError(
        "Required runtime objects are missing: "
        + ", ".join(missing)
        + ". Run the Notebook 6 bootstrap cells first."
    )

print("\nRuntime objects available:")
for name in required_objects:
    obj = globals()[name]
    try:
        print(f"  {name}: {inspect.signature(obj)}")
    except Exception:
        print(f"  {name}: available")


# -----------------------------------------------------------------------------
# 3. LOAD THE LATEST ANSWERS AND CRITIC FEEDBACK
# -----------------------------------------------------------------------------

answer_file = RESULTS / "G3_generated_answers_validated.jsonl"
critic_file = RESULTS / "G3_critic_results.jsonl"
gold_file = GOLD / "G3_regulatory_gold.jsonl"

for path in [answer_file, critic_file, gold_file]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")


def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


answer_records = load_jsonl(answer_file)
critic_records = load_jsonl(critic_file)
gold_records = load_jsonl(gold_file)


def first_value(record, keys, default=None):
    if not isinstance(record, dict):
        return default

    for key in keys:
        value = record.get(key)
        if value is not None and value != "":
            return value

    return default


def get_question_id(record):
    return first_value(
        record,
        ["question_id", "id", "case_id", "qid"],
        "UNKNOWN"
    )


def get_draft(record):
    value = first_value(
        record,
        [
            "draft",
            "answer",
            "generated_answer",
            "response",
            "final_answer",
            "text",
            "output",
        ],
        ""
    )

    if isinstance(value, dict):
        value = first_value(
            value,
            ["draft", "answer", "text", "content", "response"],
            ""
        )

    return str(value or "").strip()


def get_evidence(record):
    for key in ["evidence", "regulatory_evidence", "results"]:
        value = record.get(key)
        if isinstance(value, list):
            return value
    return []


def get_critique(record):
    value = record.get("critique", {})
    return value if isinstance(value, dict) else {}


def get_feedback(record):
    critique = get_critique(record)

    feedback = (
        critique.get("feedback")
        or record.get("feedback")
        or ""
    )

    unsupported = (
        critique.get("unsupported_claims")
        or record.get("unsupported_claims")
        or []
    )

    tone_issues = (
        critique.get("tone_issues")
        or record.get("tone_issues")
        or []
    )

    return {
        "feedback": str(feedback),
        "unsupported_claims": unsupported,
        "tone_issues": tone_issues,
        "verdict": critique.get("verdict") or record.get("verdict"),
        "passed": record.get("passed"),
    }


answer_by_id = {get_question_id(r): r for r in answer_records}
critic_by_id = {get_question_id(r): r for r in critic_records}
gold_by_id = {get_question_id(r): r for r in gold_records}

# Some gold files use a different question ID field.
# Stop rather than silently pairing the wrong question with an answer.
if not answer_by_id or not critic_by_id:
    raise RuntimeError("Could not build question ID mappings.")

print("\n[2/8] Loaded:")
print("  Validated answers:", len(answer_records))
print("  Critic records:", len(critic_records))
print("  Gold questions:", len(gold_records))


# -----------------------------------------------------------------------------
# 4. IDENTIFY THE 15 ANSWERS THAT REQUIRE REVISION
# -----------------------------------------------------------------------------

failed_ids = []

for question_id, record in critic_by_id.items():
    critique = get_critique(record)
    verdict = critique.get("verdict") or record.get("verdict")

    if record.get("passed") is False or verdict == "REVISE":
        failed_ids.append(question_id)

failed_ids = sorted(failed_ids)

print("\n[3/8] Questions requiring revision:", len(failed_ids))
print(", ".join(failed_ids))

if not failed_ids:
    raise RuntimeError(
        "No REVISE records found. Check the critic results before proceeding."
    )


# -----------------------------------------------------------------------------
# 5. HELPERS FOR QUESTION AND EVIDENCE RECOVERY
# -----------------------------------------------------------------------------

def find_question(question_id):
    candidates = [
        answer_by_id.get(question_id, {}),
        gold_by_id.get(question_id, {}),
    ]

    question_keys = [
        "question",
        "request",
        "clean_request",
        "prompt",
        "query",
        "user_question",
    ]

    for record in candidates:
        value = first_value(record, question_keys)
        if isinstance(value, str) and value.strip():
            return value.strip()

    return None


def evidence_id(item):
    if not isinstance(item, dict):
        return None

    return (
        item.get("chunk_id")
        or item.get("id")
        or item.get("source_id")
    )


def merge_evidence(*evidence_lists):
    merged = []
    seen = set()

    for evidence_list in evidence_lists:
        for item in evidence_list or []:
            if not isinstance(item, dict):
                continue

            identifier = evidence_id(item)

            if identifier:
                if identifier in seen:
                    continue
                seen.add(identifier)

            merged.append(item)

    return merged


def extract_text_from_output(output, candidate_keys):
    if isinstance(output, str):
        return output.strip()

    if not isinstance(output, dict):
        return ""

    for key in candidate_keys:
        value = output.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

        if isinstance(value, dict):
            nested = extract_text_from_output(value, candidate_keys)
            if nested:
                return nested

    return ""


def extract_output_evidence(output):
    if not isinstance(output, dict):
        return []

    for key in ["evidence", "regulatory_evidence", "results"]:
        value = output.get(key)
        if isinstance(value, list):
            return value

    return []


# -----------------------------------------------------------------------------
# 6. RESEARCH, REVISE AND CRITIQUE EACH FAILED ANSWER
# -----------------------------------------------------------------------------

repaired_records = []
repair_errors = []

for index, question_id in enumerate(failed_ids, start=1):

    print("\n" + "=" * 100)
    print(f"[{index}/{len(failed_ids)}] REPAIRING {question_id}")
    print("=" * 100)

    try:
        old_answer = answer_by_id.get(question_id)
        old_critic = critic_by_id.get(question_id)

        if not old_answer or not old_critic:
            raise ValueError("Matching answer or critic record is missing.")

        question = find_question(question_id)

        if not question:
            raise ValueError(
                "Question text could not be recovered. "
                "Refusing to guess the question."
            )

        old_draft = get_draft(old_answer)
        old_evidence = get_evidence(old_answer)
        feedback = get_feedback(old_critic)

        if not old_draft:
            raise ValueError("Existing draft is empty.")

        if not feedback["feedback"] and not feedback["unsupported_claims"]:
            raise ValueError("Critic feedback is empty.")

        # Build explicit revision instructions.
        revision_instructions = """
You are revising a regulatory answer for ComplaintIQ.

MANDATORY REVISION RULES:

1. Use only statements directly supported by the supplied evidence.
2. Do not infer missing regulatory text from a truncated chunk.
3. Do not invent statutory sections, deadlines, monetary thresholds,
   exceptions, examples, or regulatory obligations.
4. Correct every citation mismatch identified by the critic.
5. If a claim cannot be supported by the available evidence,
   remove it or explicitly state that the supplied evidence does not
   establish that point.
6. Do not convert allegations or interpretations into established facts.
7. Do not cite a chunk merely because it discusses the same regulation.
   The cited chunk must support the specific statement.
8. Preserve useful, supported content from the original answer.
9. Keep the answer focused on the actual question.
10. Do not manufacture evidence or citation identifiers.

SPECIFIC CRITIC FEEDBACK:
""" + json.dumps(feedback, ensure_ascii=False, indent=2)

        # Fresh research using the actual question.
        research_output = research_node({
            "request": question,
            "clean_request": question,
            "intent": "REG_QA",
            "trace": [],
        })

        fresh_evidence = extract_output_evidence(research_output)

        # Retain previous evidence as well as fresh research evidence.
        combined_evidence = merge_evidence(
            old_evidence,
            fresh_evidence
        )

        if not combined_evidence:
            raise ValueError(
                "No evidence available after fresh research. "
                "Answer cannot be safely regenerated."
            )

        print("Original evidence chunks:", len(old_evidence))
        print("Fresh evidence chunks:", len(fresh_evidence))
        print("Combined evidence chunks:", len(combined_evidence))

        # Pass the critic feedback into the drafting state.
        # The original question remains separately available.
        draft_state = {
            "request": question,
            "clean_request": question,
            "intent": "REG_QA",
            "evidence": combined_evidence,
            "sql_rows": [],
            "similar_cases": [],
            "draft": old_draft,
            "revisions": 1,
            "trace": [],

            # Revision guidance for implementations that consume these fields.
            "feedback": revision_instructions,
            "critic_feedback": revision_instructions,
            "revision_feedback": revision_instructions,
            "unsupported_claims": feedback["unsupported_claims"],
        }

        draft_output = drafter_node(draft_state)

        revised_draft = extract_text_from_output(
            draft_output,
            [
                "draft",
                "answer",
                "generated_answer",
                "response",
                "final_answer",
                "text",
                "output",
            ]
        )

        if not revised_draft:
            raise ValueError(
                "Drafter did not return a recognizable answer field. "
                f"Output type: {type(draft_output).__name__}; "
                f"keys: {list(draft_output.keys()) if isinstance(draft_output, dict) else 'N/A'}"
            )

        # Include any evidence returned by the drafter.
        final_evidence = merge_evidence(
            combined_evidence,
            extract_output_evidence(draft_output)
        )

        # Critic receives the revised draft and complete evidence set.
        critic_state = {
            "request": question,
            "clean_request": question,
            "intent": "REG_QA",
            "draft": revised_draft,
            "evidence": final_evidence,
            "sql_rows": [],
            "similar_cases": [],
            "revisions": 1,
            "trace": [],
        }

        critic_output = critic_node(critic_state)

        if not isinstance(critic_output, dict):
            raise ValueError(
                "Critic returned an unexpected result type: "
                + type(critic_output).__name__
            )

        # Normalize common critic return shapes.
        critique = critic_output.get("critique")

        if not isinstance(critique, dict):
            critique = critic_output

        verdict = critique.get("verdict", "UNKNOWN")

        passed = (
            critic_output.get("passed")
            if "passed" in critic_output
            else verdict == "PASS"
        )

        # Never accept an inconsistent verdict.
        if verdict != "PASS":
            passed = False

        record = {
            "question_id": question_id,
            "question": question,
            "draft": revised_draft,
            "evidence": final_evidence,
            "critique": critique,
            "verdict": verdict,
            "passed": bool(passed),
            "repair_status": "REVISED_AND_REEVALUATED",
            "previous_feedback": feedback,
            "evidence_count": len(final_evidence),
            "run_id": RUN_ID,
        }

        repaired_records.append(record)

        print("Revised draft length:", len(revised_draft))
        print("Final evidence chunks:", len(final_evidence))
        print("New critic verdict:", verdict)
        print("New critic passed:", bool(passed))

    except Exception as exc:

        error_record = {
            "question_id": question_id,
            "error": repr(exc),
            "traceback": traceback.format_exc(),
        }

        repair_errors.append(error_record)

        print("REPAIR ERROR:", repr(exc))
        print("This question has not been marked as passed.")


# -----------------------------------------------------------------------------
# 7. SAVE REPAIRED ANSWERS AND EVALUATION RESULTS
# -----------------------------------------------------------------------------

repaired_file = RESULTS / "G3_repaired_answers.jsonl"
repair_summary_file = RESULTS / "G3_repair_summary.csv"
repair_errors_file = RESULTS / "G3_repair_errors.jsonl"

with open(repaired_file, "w", encoding="utf-8") as f:
    for record in repaired_records:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

summary_rows = []

for record in repaired_records:
    critique = record.get("critique", {})

    summary_rows.append({
        "question_id": record["question_id"],
        "repair_status": record["repair_status"],
        "verdict": record["verdict"],
        "passed": record["passed"],
        "draft_length": len(record["draft"]),
        "evidence_count": record["evidence_count"],
        "unsupported_claims": json.dumps(
            critique.get("unsupported_claims", []),
            ensure_ascii=False
        ),
        "feedback": critique.get("feedback", ""),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(repair_summary_file, index=False)

with open(repair_errors_file, "w", encoding="utf-8") as f:
    for record in repair_errors:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# -----------------------------------------------------------------------------
# 8. FINAL SUMMARY
# -----------------------------------------------------------------------------

passes = sum(1 for r in repaired_records if r["passed"] is True)
revisions = sum(1 for r in repaired_records if r["passed"] is False)

summary = {
    "run_id": RUN_ID,
    "questions_targeted": len(failed_ids),
    "questions_repaired_and_evaluated": len(repaired_records),
    "repair_errors": len(repair_errors),
    "critic_passes_after_repair": passes,
    "critic_revisions_after_repair": revisions,
    "unresolved": len(failed_ids) - len(repaired_records),
    "backup_directory": str(BACKUP_DIR),
    "repaired_answers": str(repaired_file),
    "repair_summary": str(repair_summary_file),
    "repair_errors_file": str(repair_errors_file),
}

summary_file = RESULTS / "G3_targeted_repair_run_summary.json"

with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 100)
print("G3 TARGETED REPAIR — FINAL SUMMARY")
print("=" * 100)
print(json.dumps(summary, indent=2))

print("\nQuestion-wise outcomes:")
if not summary_df.empty:
    print(
        summary_df[
            ["question_id", "verdict", "passed", "evidence_count"]
        ].to_string(index=False)
    )

print("\nSaved artifacts:")
print(repaired_file)
print(repair_summary_file)
print(repair_errors_file)
print(summary_file)

print("\nOriginal validated files remain unchanged.")
print("No S3 writes or checkpoint database modifications were performed.")
print("\nTARGETED REPAIR CELL COMPLETED.")

COMPLAINTIQ — G3 TARGETED ANSWER REPAIR
Run ID: 20261106T144421Z
Backup: /home/sagemaker-user/complaintiq/eval/results/correction_backups/20261106T144421Z_targeted_repair

[1/8] Existing artifacts backed up.

Runtime objects available:
  research_node: (state: __main__.CIQState) -> dict
  drafter_node: (state: __main__.CIQState) -> dict
  critic_node: (state)

[2/8] Loaded:
  Validated answers: 17
  Critic records: 17
  Gold questions: 20

[3/8] Questions requiring revision: 15
G3-01, G3-02, G3-03, G3-04, G3-05, G3-07, G3-08, G3-09, G3-10, G3-11, G3-12, G3-13, G3-14, G3-15, G3-16

[1/15] REPAIRING G3-01


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Original evidence chunks: 8
Fresh evidence chunks: 8
Combined evidence chunks: 8
Revised draft length: 3322
Final evidence chunks: 8
New critic verdict: REVISE
New critic passed: False

[2/15] REPAIRING G3-02


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Original evidence chunks: 8
Fresh evidence chunks: 8
Combined evidence chunks: 8
Revised draft length: 2501
Final evidence chunks: 8
New critic verdict: REVISE
New critic passed: False

[3/15] REPAIRING G3-03


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

... (21 outputs left out)

Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Original evidence chunks: 8
Fresh evidence chunks: 8
Combined evidence chunks: 8
Revised draft length: 2935
Final evidence chunks: 8
New critic verdict: REVISE
New critic passed: False

[15/15] REPAIRING G3-16


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

Original evidence chunks: 8
Fresh evidence chunks: 8
Combined evidence chunks: 8
Revised draft length: 2822
Final evidence chunks: 8
New critic verdict: REVISE
New critic passed: False

G3 TARGETED REPAIR — FINAL SUMMARY
{
  "run_id": "20261106T144421Z",
  "questions_targeted": 15,
  "questions_repaired_and_evaluated": 15,
  "repair_errors": 0,
  "critic_passes_after_repair": 0,
  "critic_revisions_after_repair": 15,
  "unresolved": 0,
  "backup_directory": "/home/sagemaker-user/complaintiq/eval/results/correction_backups/20261106T144421Z_targeted_repair",
  "repaired_answers": "/home/sagemaker-user/complaintiq/eval/results/G3_repaired_answers.jsonl",
  "repair_summary": "/home/sagemaker-user/complaintiq/eval/results/G3_repair_summary.csv",
  "repair_errors_file": "/home/sagemaker-user/complaintiq/eval/results/G3_repair_errors.jsonl"
}

Question-wise outcomes:
question_id verdict  passed  evidence_count
      G3-01  REVISE   False               8
      G3-02  REVISE   False            

In [76]:
# =============================================================================
# COMPLAINTIQ — G3 REPAIR PIPELINE DIAGNOSTICS
#
# READ-ONLY:
#   Does not call the LLM
#   Does not modify any result files
#   Does not modify checkpoints
#   Does not write to S3
# =============================================================================

from pathlib import Path
import json
import inspect
import hashlib
import difflib
import pandas as pd

ROOT = Path("/home/sagemaker-user/complaintiq")
RESULTS = ROOT / "eval" / "results"

original_file = RESULTS / "G3_generated_answers_validated.jsonl"
repaired_file = RESULTS / "G3_repaired_answers.jsonl"
critic_file = RESULTS / "G3_critic_results.jsonl"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def qid(r):
    return (
        r.get("question_id")
        or r.get("id")
        or r.get("case_id")
        or "UNKNOWN"
    )

def draft(r):
    for key in [
        "draft", "answer", "generated_answer",
        "response", "final_answer", "text", "output"
    ]:
        value = r.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
        if isinstance(value, dict):
            for nested_key in ["draft", "answer", "text", "content"]:
                nested = value.get(nested_key)
                if isinstance(nested, str) and nested.strip():
                    return nested.strip()
    return ""

def evidence(r):
    for key in ["evidence", "regulatory_evidence", "results"]:
        if isinstance(r.get(key), list):
            return r[key]
    return []

def chunk_ids(items):
    return [
        x.get("chunk_id") or x.get("id") or x.get("source_id")
        for x in items if isinstance(x, dict)
    ]

def sha(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]

original = load_jsonl(original_file)
repaired = load_jsonl(repaired_file)
critics = load_jsonl(critic_file)

original_map = {qid(r): r for r in original}
repaired_map = {qid(r): r for r in repaired}
critic_map = {qid(r): r for r in critics}

print("=" * 100)
print("1. ORIGINAL VS REPAIRED ANSWER COMPARISON")
print("=" * 100)

comparison = []

for question_id in sorted(repaired_map):

    old = original_map.get(question_id, {})
    new = repaired_map[question_id]

    old_draft = draft(old)
    new_draft = draft(new)

    old_ids = set(chunk_ids(evidence(old)))
    new_ids = set(chunk_ids(evidence(new)))

    same_text = old_draft == new_draft

    comparison.append({
        "question_id": question_id,
        "old_length": len(old_draft),
        "new_length": len(new_draft),
        "same_text": same_text,
        "old_hash": sha(old_draft),
        "new_hash": sha(new_draft),
        "old_evidence": len(old_ids),
        "new_evidence": len(new_ids),
        "new_chunks": len(new_ids - old_ids),
        "removed_chunks": len(old_ids - new_ids),
        "verdict": new.get("verdict"),
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df.to_string(index=False))

same_count = sum(x["same_text"] for x in comparison)

print("\nIdentical drafts:", same_count, "/", len(comparison))
print("Changed drafts:", len(comparison) - same_count)


print("\n" + "=" * 100)
print("2. SHOW FIRST ANSWER DIFFERENCE")
print("=" * 100)

for question_id in sorted(repaired_map):

    old = draft(original_map.get(question_id, {}))
    new = draft(repaired_map[question_id])

    if old != new:
        diff = list(difflib.unified_diff(
            old.splitlines(),
            new.splitlines(),
            fromfile="ORIGINAL",
            tofile="REPAIRED",
            lineterm="",
            n=2
        ))

        print("Question:", question_id)
        print("\n".join(diff[:100]))
        break
else:
    print("No answer text changed in any repaired record.")


print("\n" + "=" * 100)
print("3. NODE SIGNATURES AND IMPLEMENTATION")
print("=" * 100)

for name in ["drafter_node", "research_node", "critic_node"]:

    obj = globals().get(name)

    print(f"\n--- {name} ---")

    if obj is None:
        print("NOT AVAILABLE")
        continue

    try:
        print("Signature:", inspect.signature(obj))
    except Exception as e:
        print("Signature unavailable:", repr(e))

    try:
        print("Source:")
        print(inspect.getsource(obj)[:12000])
    except Exception as e:
        print("Source unavailable:", repr(e))

    code = getattr(obj, "__code__", None)

    if code:
        print("Referenced names:", code.co_names)
        print("Referenced variables:", code.co_varnames)


print("\n" + "=" * 100)
print("4. STATE SCHEMA")
print("=" * 100)

state_cls = globals().get("CIQState")

if state_cls is not None:

    print("State class:", state_cls)

    try:
        print("Annotations:")
        print(json.dumps(
            getattr(state_cls, "__annotations__", {}),
            indent=2,
            default=str
        ))
    except Exception as e:
        print("Could not read annotations:", repr(e))

    try:
        print("Pydantic fields:")
        print(list(state_cls.model_fields.keys()))
    except Exception:
        pass
else:
    print("CIQState is not available.")


print("\n" + "=" * 100)
print("5. CRITIC FEEDBACK AFTER REPAIR")
print("=" * 100)

for question_id in sorted(repaired_map):

    record = repaired_map[question_id]
    critique = record.get("critique", {})

    if not isinstance(critique, dict):
        critique = {}

    print(f"\n{question_id} | {record.get('verdict')}")

    print("Feedback:")
    print(critique.get("feedback", "No feedback field"))

    print("Unsupported claims:")
    print(json.dumps(
        critique.get("unsupported_claims", []),
        indent=2,
        ensure_ascii=False
    ))


print("\n" + "=" * 100)
print("6. EVIDENCE COMPARISON — G3-01")
print("=" * 100)

for question_id in ["G3-01", "G3-04", "G3-09", "G3-17"]:

    old = original_map.get(question_id, {})
    new = repaired_map.get(question_id, {})

    print(f"\n{question_id}")

    old_ids = set(chunk_ids(evidence(old)))
    new_ids = set(chunk_ids(evidence(new)))

    print("Original chunks:")
    for x in sorted(old_ids):
        print(" ", x)

    print("Repaired chunks:")
    for x in sorted(new_ids):
        print(" ", x)

    print("Newly retrieved chunks:")
    for x in sorted(new_ids - old_ids):
        print(" ", x)


print("\n" + "=" * 100)
print("DIAGNOSTICS COMPLETE — NO FILES MODIFIED")
print("=" * 100)

1. ORIGINAL VS REPAIRED ANSWER COMPARISON
question_id  old_length  new_length  same_text         old_hash         new_hash  old_evidence  new_evidence  new_chunks  removed_chunks verdict
      G3-01        3322        3322       True 93af4b37725e7a83 93af4b37725e7a83             8             8           0               0  REVISE
      G3-02        2501        2501       True 6415c39704c7aa53 6415c39704c7aa53             8             8           0               0  REVISE
      G3-03        3111        3111       True c943962df2596b0f c943962df2596b0f             8             8           0               0  REVISE
      G3-04        4098        4098       True 06d1ba9125c6cad0 06d1ba9125c6cad0             8             8           0               0  REVISE
      G3-05        3500        3500       True 8aeeb07eb42aadec 8aeeb07eb42aadec             8             8           0               0  REVISE
      G3-07        3110        3110       True 7f74334ade478ec3 7f74334ade478ec3        

In [77]:
import inspect
import dis
import json

print("=" * 100)
print("1. DRAFTER NODE — EXACT IMPLEMENTATION")
print("=" * 100)

print("Signature:", inspect.signature(drafter_node))
print("Code names:", drafter_node.__code__.co_names)
print("\nLocal variables:", drafter_node.__code__.co_varnames)
print("\nConstants:")
for i, value in enumerate(drafter_node.__code__.co_consts):
    if isinstance(value, str):
        print(f"\nCONST {i}:\n{value[:5000]}")
    elif value is not None:
        print(f"CONST {i}: {repr(value)[:1000]}")

print("\n--- BYTECODE ---")
dis.dis(drafter_node)


print("\n" + "=" * 100)
print("2. DRAFTER PROMPT TEMPLATES")
print("=" * 100)

for name in ["DRAFT_SYS", "COMMON_DRAFT_RULES"]:
    value = drafter_node.__globals__.get(name)
    print(f"\n--- {name} ---")
    print(repr(value)[:12000])


print("\n" + "=" * 100)
print("3. LLM CHAT METHOD")
print("=" * 100)

chat_method = getattr(llm, "chat", None)

if chat_method:
    print("Signature:", inspect.signature(chat_method))
    print("Code names:", getattr(chat_method, "__code__", None).co_names
          if hasattr(chat_method, "__code__") else "Unavailable")

    if hasattr(chat_method, "__code__"):
        print("\n--- CHAT BYTECODE ---")
        dis.dis(chat_method)
else:
    print("llm.chat was not found.")


print("\n" + "=" * 100)
print("4. RESEARCH NODE — EXACT IMPLEMENTATION")
print("=" * 100)

print("Signature:", inspect.signature(research_node))
print("Code names:", research_node.__code__.co_names)
print("Local variables:", research_node.__code__.co_varnames)

print("\nConstants:")
for i, value in enumerate(research_node.__code__.co_consts):
    if isinstance(value, str):
        print(f"\nCONST {i}:\n{value[:3000]}")
    elif value is not None:
        print(f"CONST {i}: {repr(value)[:500]}")

print("\n--- BYTECODE ---")
dis.dis(research_node)


print("\n" + "=" * 100)
print("5. RETRIEVAL INDEX INTERFACE")
print("=" * 100)

print("REG_INDEX type:", type(REG_INDEX))
print("REG_INDEX methods:", [
    name for name in dir(REG_INDEX)
    if not name.startswith("_")
])

search_method = getattr(REG_INDEX, "search", None)

if search_method:
    print("Search signature:", inspect.signature(search_method))

    if hasattr(search_method, "__code__"):
        print("Search code names:", search_method.__code__.co_names)
        print("Search constants:", [
            x for x in search_method.__code__.co_consts
            if isinstance(x, (str, int, float))
        ])


print("\n" + "=" * 100)
print("6. CRITIQUE FIELD — CURRENT STRUCTURE")
print("=" * 100)

print("CIQState annotations:")
print(json.dumps(
    {k: str(v) for k, v in CIQState.__annotations__.items()},
    indent=2
))

sample = next(
    (
        r for r in g3_repair_records
        if r.get("question_id") == "G3-01"
    ),
    None
)

if sample:
    print("\nG3-01 repair record keys:")
    print(list(sample.keys()))

    print("\nG3-01 current critique, if stored:")
    print(json.dumps(
        sample.get("critique", {}),
        indent=2,
        default=str
    )[:5000])

print("\n" + "=" * 100)
print("DIAGNOSTIC COMPLETE — NO FILES MODIFIED")
print("=" * 100)

1. DRAFTER NODE — EXACT IMPLEMENTATION
Signature: (state: __main__.CIQState) -> dict
Code names: ('DRAFT_SYS', 'ValueError', 'get', 'join', 'llm', 'chat', 'MODELS', 'COMMON_DRAFT_RULES')

Local variables: ('state', 'intent', 'evidence', 'evidence_text', 'user', 'critique', 'draft')

Constants:

CONST 1:
intent

CONST 2:
No drafting instructions for intent 

CONST 3:
evidence

CONST 4:



CONST 5: <code object <genexpr> at 0x7fcdb08090b0, file "notebook4_cell_62", line 34>

CONST 6:
<request>

CONST 7:
clean_request

CONST 8:
</request>
TRIAGE: 

CONST 9:
triage

CONST 10:

SIMILAR CASES: 

CONST 11:
similar_cases
CONST 12: 5

CONST 13:

DATA: 

CONST 14:
sql_rows
CONST 15: 50

CONST 16:

EVIDENCE:


CONST 17:
critique

CONST 18:
pass

CONST 19:

Reviewer feedback: 

CONST 20:

Revise the previous draft:


CONST 21:
draft

CONST 22:


CONST 23:
worker

CONST 24:


CONST 25: 1800
CONST 26: 0.0
CONST 27: ('max_tokens', 'temperature')

CONST 28:
drafter

CONST 29:
revisions
CONST 30: 0
CON

NameError: name 'g3_repair_records' is not defined